In [ ]:
# 원본 코드

# import system libs
import os
import time
import shutil
import pathlib
import itertools
from PIL import Image

# import data handling tools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
# Generate data paths with labels
data_dir = 'lung_colon_image_set'
filepaths = []
labels = []

folds = os.listdir(data_dir)
for fold in folds:
    foldpath = os.path.join(data_dir, fold)
    flist = os.listdir(foldpath)

    for f in flist:
        f_path = os.path.join(foldpath, f)
        filelist = os.listdir(f_path)

        for file in filelist:
            fpath = os.path.join(f_path, file)
            filepaths.append(fpath)

            if f == 'colon_aca':
                labels.append('Colon Adenocarcinoma')

            elif f == 'colon_n':
                labels.append('Colon Benign Tissue')

            elif f == 'lung_aca':
                labels.append('Lung Adenocarcinoma')

            elif f == 'lung_n':
                labels.append('Lung Benign Tissue')

            elif f == 'lung_scc':
                labels.append('Lung Squamous Cell Carcinoma')

# Concatenate data paths with labels into one dataframe
Fseries = pd.Series(filepaths, name= 'filepaths')
Lseries = pd.Series(labels, name='labels')
df = pd.concat([Fseries, Lseries], axis= 1)
# df = df.sample(frac=0.1) # Using only 10% of original data

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras import regularizers

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print ('modules loaded')
strat = df['labels']
train_df, dummy_df = train_test_split(df,  train_size= 0.8, shuffle= True, random_state= 123, stratify= strat)

# valid and test dataframe
strat = dummy_df['labels']
valid_df, test_df = train_test_split(dummy_df,  train_size= 0.5, shuffle= True, random_state= 123, stratify= strat)
# crobed image size
batch_size = 64
img_size = (224, 224)
channels = 3
img_shape = (img_size[0], img_size[1], channels)

tr_gen = ImageDataGenerator()
ts_gen = ImageDataGenerator()

train_gen = tr_gen.flow_from_dataframe( train_df, x_col= 'filepaths', y_col= 'labels', target_size= img_size, class_mode= 'categorical',
                                    color_mode= 'rgb', shuffle= True, batch_size= batch_size)

valid_gen = ts_gen.flow_from_dataframe( valid_df, x_col= 'filepaths', y_col= 'labels', target_size= img_size, class_mode= 'categorical',
                                        color_mode= 'rgb', shuffle= True, batch_size= batch_size)

test_gen = ts_gen.flow_from_dataframe( test_df, x_col= 'filepaths', y_col= 'labels', target_size= img_size, class_mode= 'categorical',
                                    color_mode= 'rgb', shuffle= False, batch_size= batch_size)

g_dict = train_gen.class_indices      # defines dictionary {'class': index}
classes = list(g_dict.keys())       # defines list of dictionary's kays (classes), classes names : string
images, labels = next(train_gen)      # get a batch size samples from the generator

plt.figure(figsize= (20, 20))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    image = images[i] / 255       # scales data to range (0 - 255)
    plt.imshow(image)
    index = np.argmax(labels[i])  # get image index
    class_name = classes[index]   # get class of image
    plt.title(class_name, color= 'blue', fontsize= 12)
    plt.axis('off')
plt.show()
# Create Model Structure
img_size = (224, 224)
channels = 3
img_shape = (img_size[0], img_size[1], channels)
class_count = len(list(train_gen.class_indices.keys()))

model = Sequential([
    Conv2D(filters=64, kernel_size=(3,3), padding="same", activation="relu", input_shape= img_shape),
    Conv2D(filters=64, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=128, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=256, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    Conv2D(filters=512, kernel_size=(3,3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(256,activation = "relu"),
    Dense(64,activation = "relu"),
    Dense(class_count, activation = "softmax")
])

model.compile(Adamax(learning_rate= 0.001), loss= 'categorical_crossentropy', metrics= ['accuracy'])
epochs = 10   # number of all epochs in training

history = model.fit(x= train_gen, epochs= epochs, verbose= 1, validation_data= test_gen,
                    validation_steps= None, shuffle= False)
# Define needed variables
tr_acc = history.history['accuracy']
tr_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
index_loss = np.argmin(val_loss)
val_lowest = val_loss[index_loss]
index_acc = np.argmax(val_acc)
acc_highest = val_acc[index_acc]
Epochs = [i+1 for i in range(len(tr_acc))]
loss_label = f'best epoch= {str(index_loss + 1)}'
acc_label = f'best epoch= {str(index_acc + 1)}'

# Plot training history
plt.figure(figsize= (20, 8))
plt.style.use('fivethirtyeight')

plt.subplot(1, 2, 1)
plt.plot(Epochs, tr_loss, 'r', label= 'Training loss')
plt.plot(Epochs, val_loss, 'g', label= 'Validation loss')
plt.scatter(index_loss + 1, val_lowest, s= 150, c= 'blue', label= loss_label)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(Epochs, tr_acc, 'r', label= 'Training Accuracy')
plt.plot(Epochs, val_acc, 'g', label= 'Validation Accuracy')
plt.scatter(index_acc + 1 , acc_highest, s= 150, c= 'blue', label= acc_label)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout
plt.show()
ts_length = len(test_df)
test_batch_size = max(sorted([ts_length // n for n in range(1, ts_length + 1) if ts_length%n == 0 and ts_length/n <= 80]))
test_steps = ts_length // test_batch_size

train_score = model.evaluate(train_gen, steps= test_steps, verbose= 1)
valid_score = model.evaluate(valid_gen, steps= test_steps, verbose= 1)
test_score = model.evaluate(test_gen, steps= test_steps, verbose= 1)

print("Train Loss: ", train_score[0])
print("Train Accuracy: ", train_score[1])
print('-' * 20)
print("Valid Loss: ", valid_score[0])
print("Valid Accuracy: ", valid_score[1])
print('-' * 20)
print("Test Loss: ", test_score[0])
print("Test Accuracy: ", test_score[1])
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)
g_dict = test_gen.class_indices
classes = list(g_dict.keys())

# Confusion matrix
cm = confusion_matrix(test_gen.classes, y_pred)

plt.figure(figsize= (10, 10))
plt.imshow(cm, interpolation= 'nearest', cmap= plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation= 45)
plt.yticks(tick_marks, classes)


thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], horizontalalignment= 'center', color= 'white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.show()
# Classification report
print(classification_report(test_gen.classes, y_pred, target_names= classes))
#Save the model
model.save('Model.h5')
loaded_model = tf.keras.models.load_model('Model.h5', compile=False)
loaded_model.compile(Adamax(learning_rate= 0.001), loss= 'categorical_crossentropy', metrics= ['accuracy'])
image_path = 'lung_colon_image_set/lung_image_sets/lung_aca/lungaca1001.jpeg'
image = Image.open(image_path)

# Preprocess the image
img = image.resize((224, 224))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

# Make predictions
predictions = loaded_model.predict(img_array)
class_labels = classes
score = tf.nn.softmax(predictions[0])
print(f"{class_labels[tf.argmax(score)]}")
```

In [ ]:
# 블록 나누기
import os
import math
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
                                      Dropout, BatchNormalization, Concatenate)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras import mixed_precision
import tensorflow.keras.backend as K
from PIL import Image
import random
import json as json_module
import gc

# ======================================================
# 한글 폰트 설정
# ======================================================
import matplotlib
import matplotlib.font_manager as fm
import glob

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# matplotlib 캐시 삭제 (구버전 호환)
cache_dir = matplotlib.get_cachedir()
for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
    try: os.remove(f)
    except: pass

# 폰트 등록 및 적용
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [font_name]
matplotlib.rcParams["axes.unicode_minus"] = False

print(f"✅ 한글 폰트 적용: {font_name}")

# ======================================================
# RTX 4090 최적화
# ======================================================
mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(False)
print("GPU:", tf.config.list_physical_devices("GPU"))

# ======================================================
# 설정
# ======================================================
BLOCK_SIZES  = [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]
BASE_TRAIN   = "/workspace/Train"
WORKSPACE    = "/workspace"
IMG_SIZE     = 512
BATCH_SIZE   = 4
EPOCHS_S1    = 30
EPOCHS_S2    = 60
NUM_CLASSES  = 3
SEED         = 42
N_FOLDS      = 7
TTA_STEPS    = 20
SWA_START    = 20
TARGET_COUNT = 800
TEST_RATIO   = 0.10
AUTOTUNE     = tf.data.AUTOTUNE

# ======================================================
# 1. 블록 암호화
# ======================================================
def block_scramble(img_array, seed, block_size):
    h, w, c = img_array.shape
    if block_size >= min(h, w):
        return img_array.copy()
    pad_h = (block_size - h % block_size) % block_size
    pad_w = (block_size - w % block_size) % block_size
    if pad_h > 0 or pad_w > 0:
        img_array = np.pad(img_array, ((0,pad_h),(0,pad_w),(0,0)), mode="reflect")
    ph, pw   = img_array.shape[:2]
    n_h, n_w = ph//block_size, pw//block_size
    blocks   = [img_array[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
                for i in range(n_h) for j in range(n_w)]
    rng      = np.random.default_rng(seed)
    order    = rng.permutation(n_h * n_w)
    result   = np.zeros_like(img_array)
    for new_idx, old_idx in enumerate(order):
        i, j = new_idx//n_w, new_idx%n_w
        result[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = blocks[old_idx]
    return result[:h, :w]

def generate_block_dataset(block_size, seed=42):
    if block_size >= 512:
        print(f"  Block512: 원본 사용")
        return BASE_TRAIN
    out_dir = os.path.join(WORKSPACE, f"Train_Block{block_size}")
    classes = sorted([d for d in os.listdir(BASE_TRAIN)
                      if os.path.isdir(os.path.join(BASE_TRAIN, d))])
    total_src = sum(len([f for f in os.listdir(os.path.join(BASE_TRAIN,c))
                         if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                    for c in classes)
    total_ex = sum(len([f for f in os.listdir(os.path.join(out_dir,c))
                        if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                   for c in classes if os.path.exists(os.path.join(out_dir,c))
                   ) if os.path.exists(out_dir) else 0
    if total_ex >= total_src:
        print(f"  Block{block_size}: 이미 존재 ({total_ex}장) → 스킵")
        return out_dir
    print(f"  Block{block_size}: 암호화 중 ({total_src}장)...")
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        for f in os.listdir(os.path.join(BASE_TRAIN, cls)):
            if not f.lower().endswith((".jpg",".jpeg",".png",".bmp")): continue
            dst = os.path.join(out_dir, cls, f)
            if os.path.exists(dst): continue
            try:
                img = np.array(Image.open(os.path.join(BASE_TRAIN,cls,f)).convert("RGB"), dtype=np.uint8)
                Image.fromarray(block_scramble(img, seed, block_size)).save(dst, "JPEG", quality=95)
            except: pass
    print(f"  Block{block_size}: 완료!")
    return out_dir

# ======================================================
# 2. 오버샘플링 (원본 폴더 보호 - 별도 폴더에 복사 후 증강)
# ======================================================
def oversample_dir(base_dir, target_count, seed=42):
    """
    ⚠️ 원본 보호: base_dir 안에 직접 저장하지 않음
    base_dir_oversampled/ 폴더에 복사 후 증강
    이미 존재하면 재사용
    """
    out_dir = base_dir.rstrip("/") + "_oversampled"

    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir,d))])

    # 이미 완성된 폴더인지 확인
    if os.path.exists(out_dir):
        # 각 클래스별로 target_count 이상인지 확인
        all_done = all(
            len([f for f in os.listdir(os.path.join(out_dir,c))
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]) >= target_count
            for c in classes if os.path.exists(os.path.join(out_dir,c))
        ) and all(os.path.exists(os.path.join(out_dir,c)) for c in classes)
        if all_done:
            total = sum(len([f for f in os.listdir(os.path.join(out_dir,c))
                             if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                        for c in classes)
            print(f"    오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    # 원본 → 출력 폴더로 복사
    random.seed(seed)
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in os.listdir(os.path.join(base_dir,cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]  # ← aug_ 파일 제외
        # 원본 파일만 복사 (아직 없는 것만)
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir,cls,f), dst)

    # 증강으로 target_count 채우기
    for cls in classes:
        cls_out = os.path.join(out_dir, cls)
        files = [f for f in os.listdir(cls_out)
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
        src_files = [f for f in os.listdir(os.path.join(base_dir,cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]  # ← 항상 순수 원본만 선택
        needed = target_count - len(files)
        if needed <= 0:
            print(f"    {cls}: {len(files)}장 (충분)"); continue
        print(f"    {cls}: {len(files)}장 → {target_count}장 ({needed}장 증강)")
        generated = 0
        while generated < needed:
            src_f = random.choice(src_files)  # ← 항상 원본에서 선택
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(random.uniform(-30,30), expand=False)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
                from PIL import ImageEnhance
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7,1.3))
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"),
                         "JPEG", quality=95)
                generated += 1
            except: continue

    total = sum(len([f for f in os.listdir(os.path.join(out_dir,c))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                for c in classes)
    print(f"    ✅ 오버샘플링 완료: {total}장 → {out_dir}")
    return out_dir

# ======================================================
# 3. Train/Test 분리
# ======================================================
def prepare_split(base_dir, split_dir, test_ratio=0.10, seed=42):
    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")
    if os.path.exists(test_dir) and os.path.exists(kfold_dir):
        print(f"    Split 폴더 이미 존재 → 재사용"); return
    np.random.seed(seed)
    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir,d))])
    for cls in classes:
        os.makedirs(os.path.join(test_dir,  cls), exist_ok=True)
        os.makedirs(os.path.join(kfold_dir, cls), exist_ok=True)
    for cls in classes:
        files = [f for f in os.listdir(os.path.join(base_dir,cls))
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
        np.random.shuffle(files)
        n_test = int(len(files)*test_ratio)
        for f in files[:n_test]:
            shutil.copy2(os.path.join(base_dir,cls,f), os.path.join(test_dir,cls,f))
        for f in files[n_test:]:
            shutil.copy2(os.path.join(base_dir,cls,f), os.path.join(kfold_dir,cls,f))
        print(f"    {cls}: kfold={len(files)-n_test}, test={n_test}")

# ======================================================
# 4. tf.data 파이프라인 (ImageDataGenerator 완전 교체)
# ======================================================
def get_paths_labels(directory, class_names):
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        if not os.path.exists(cls_dir): continue
        for f in os.listdir(cls_dir):
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
                paths.append(os.path.join(cls_dir, f))
                one_hot = [0.0]*len(class_names)
                one_hot[idx] = 1.0
                labels.append(one_hot)
    return paths, labels

@tf.function
def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label

@tf.function
def augment_train(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.25*255)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.25)
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=5.0, dtype=tf.float32)
    img = tf.clip_by_value(img + noise, 0.0, 255.0)
    return img, label

@tf.function
def cutmix_batch(x, y, alpha=0.3):
    """CutMix: 이미지 일부를 다른 이미지로 대체"""
    batch_size = tf.shape(x)[0]
    idx   = tf.random.shuffle(tf.range(batch_size))
    lam   = tf.cast(tf.random.uniform([], alpha, 1.0-alpha), tf.float32)
    h     = tf.cast(tf.shape(x)[1], tf.float32)
    w     = tf.cast(tf.shape(x)[2], tf.float32)
    cut_h = tf.cast(h * tf.math.sqrt(1.0 - lam), tf.int32)
    cut_w = tf.cast(w * tf.math.sqrt(1.0 - lam), tf.int32)
    cx    = tf.random.uniform([], 0, tf.cast(h, tf.int32), dtype=tf.int32)
    cy    = tf.random.uniform([], 0, tf.cast(w, tf.int32), dtype=tf.int32)
    x1    = tf.clip_by_value(cx - cut_h // 2, 0, tf.cast(h, tf.int32))
    y1    = tf.clip_by_value(cy - cut_w // 2, 0, tf.cast(w, tf.int32))
    x2    = tf.clip_by_value(cx + cut_h // 2, 0, tf.cast(h, tf.int32))
    y2    = tf.clip_by_value(cy + cut_w // 2, 0, tf.cast(w, tf.int32))
    # 마스크 생성
    mask  = tf.ones([x2-x1, y2-y1, tf.shape(x)[3]])
    paddings = [[x1, tf.shape(x)[1]-x2], [y1, tf.shape(x)[2]-y2], [0, 0]]
    mask  = tf.pad(mask, paddings)
    mask  = tf.expand_dims(mask, 0)
    # 이미지 합성
    x     = tf.cast(x, tf.float32)
    mixed = x * (1.0 - mask) + tf.gather(x, idx) * mask
    # 라벨 비율
    lam_actual = 1.0 - tf.cast((x2-x1)*(y2-y1), tf.float32) / (h*w)
    y     = tf.cast(y, tf.float32)
    mixed_y = lam_actual * y + (1.0-lam_actual) * tf.gather(y, idx)
    return mixed, mixed_y

def apply_cutmix(ds, alpha=0.3):
    return ds.map(lambda x,y: cutmix_batch(x,y,alpha),
                  num_parallel_calls=AUTOTUNE).prefetch(1)

def apply_mixup_cutmix(ds, alpha=0.3, cutmix_prob=0.5):
    """Mixup과 CutMix를 랜덤하게 섞어서 적용"""
    def _apply(x, y):
        use_cutmix = tf.random.uniform([]) < cutmix_prob
        mx, my = mixup_batch(x, y, alpha)
        cx, cy = cutmix_batch(x, y, alpha)
        return tf.cond(use_cutmix, lambda: (cx, cy), lambda: (mx, my))
    return ds.map(_apply, num_parallel_calls=AUTOTUNE).prefetch(1)

@tf.function
def augment_tta(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15*255)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    return img, label

def make_train_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(min(n,1000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.map(lambda x,y: augment_train(x,y), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(1)
    return ds, n

def make_eval_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(1)
    return ds, n, paths

def make_tta_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.map(lambda x,y: augment_tta(x,y), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(1)
    return ds, n

@tf.function
def mixup_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx = tf.random.shuffle(tf.range(batch_size))
    lam = tf.cast(tf.random.uniform([], minval=alpha, maxval=1.0-alpha+0.001), tf.float32)
    x = tf.cast(x, tf.float32); y = tf.cast(y, tf.float32)
    return lam*x+(1-lam)*tf.gather(x,idx), lam*y+(1-lam)*tf.gather(y,idx)

def apply_mixup(ds, alpha=0.3):
    return ds.map(lambda x,y: mixup_batch(x,y,alpha),
                  num_parallel_calls=AUTOTUNE).prefetch(1)

# ======================================================
# 5. 학습 기법
# ======================================================
def focal_loss(gamma=2.0, label_smoothing=0.05):
    def loss_fn(y_true, y_pred):
        num_cls   = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_sm = y_true*(1.0-label_smoothing)+(label_smoothing/num_cls)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce        = -y_true_sm * tf.math.log(y_pred)
        weight    = y_true_sm * tf.math.pow(1.0-y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight*ce, axis=-1))
    return loss_fn

class CosineAnnealingWarmRestart(tf.keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, T0=10, T_mult=2):
        super().__init__()
        self.lr_max=lr_max; self.lr_min=lr_min
        self.T0=T0; self.T_mult=T_mult; self.T_cur=0; self.T_i=T0
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.lr_min+0.5*(self.lr_max-self.lr_min)*(1+math.cos(math.pi*self.T_cur/self.T_i))
        K.set_value(self.model.optimizer.learning_rate, lr)
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur=0; self.T_i=int(self.T_i*self.T_mult)

class SWACallback(tf.keras.callbacks.Callback):
    def __init__(self, start_epoch, swa_lr=1e-5):
        super().__init__()
        self.start_epoch=start_epoch; self.swa_lr=swa_lr
        self.swa_weights=None; self.n_averaged=0
    def on_epoch_end(self, epoch, logs=None):
        if epoch >= self.start_epoch:
            if self.swa_weights is None:
                self.swa_weights = [w.copy() for w in self.model.get_weights()]
            else:
                for i,w in enumerate(self.model.get_weights()):
                    self.swa_weights[i] = (self.swa_weights[i]*self.n_averaged+w)/(self.n_averaged+1)
            self.n_averaged += 1
            K.set_value(self.model.optimizer.learning_rate, self.swa_lr)
    def on_train_end(self, logs=None):
        if self.swa_weights is not None:
            self.model.set_weights(self.swa_weights)
            print(f"\n✅ SWA 완료: {self.n_averaged}개 평균")

# ======================================================
# 6. 모델 (EfficientNetV2S만 사용 - 메모리 절약)
# ======================================================
def build_model():
    base = EfficientNetV2S(weights="imagenet", include_top=False,
                           input_shape=(IMG_SIZE,IMG_SIZE,3), include_preprocessing=True)
    base.trainable = False
    gap = GlobalAveragePooling2D()(base.output)
    gmp = GlobalMaxPooling2D()(base.output)
    x   = Concatenate()([gap, gmp])
    x = Dense(512, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation("swish")(x)
    x = Dropout(0.35)(x)
    x = Dense(256, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation("swish")(x)
    x = Dropout(0.25)(x)
    out = Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return Model(inputs=base.input, outputs=out)

# ======================================================
# 7. 단일 블록 크기 실험
# ======================================================
def run_experiment(block_size, block_dir):
    n_blocks = 1 if block_size>=512 else (512*512 if block_size==1 else (512//block_size)**2)
    label = "원본" if block_size>=512 else f"Block{block_size} ({n_blocks}블록)"
    print(f"\n{'='*60}\n🧩 {label}\n{'='*60}")

    split_dir = os.path.join(WORKSPACE, f"AllExp_Split_{block_size}")
    gc.collect()

    print("  오버샘플링...")
    oversampled_dir = oversample_dir(block_dir, TARGET_COUNT)
    print("  Train/Test 분리...")
    prepare_split(oversampled_dir, split_dir, TEST_RATIO)

    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")

    class_names  = sorted([d for d in os.listdir(kfold_dir)
                            if os.path.isdir(os.path.join(kfold_dir,d))])
    class_to_idx = {c:i for i,c in enumerate(class_names)}
    benign_idx   = class_to_idx.get("Bengin cases", 0)
    normal_idx   = class_to_idx.get("Normal cases", 2)

    all_paths, all_labels = [], []
    for cls in class_names:
        for f in os.listdir(os.path.join(kfold_dir,cls)):
            if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
                all_paths.append(os.path.join(kfold_dir,cls,f))
                all_labels.append(class_to_idx[cls])
    all_paths  = np.array(all_paths)
    all_labels = np.array(all_labels)
    print(f"  학습: {len(all_paths)}장 | 테스트: {sum(len(os.listdir(os.path.join(test_dir,c))) for c in class_names)}장")

    skf           = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
        print(f"\n  --- Fold {fold+1}/{N_FOLDS} ---")
        fold_train_dir = os.path.join(split_dir, f"fold_{fold}_train")
        fold_val_dir   = os.path.join(split_dir, f"fold_{fold}_val")

        if not os.path.exists(fold_train_dir):
            for cls in class_names:
                os.makedirs(os.path.join(fold_train_dir,cls), exist_ok=True)
                os.makedirs(os.path.join(fold_val_dir,  cls), exist_ok=True)
            for idx in train_idx:
                src=all_paths[idx]; cls=class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_train_dir,cls,os.path.basename(src)))
            for idx in val_idx:
                src=all_paths[idx]; cls=class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_val_dir,cls,os.path.basename(src)))

        fold_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(all_labels[train_idx]), y=all_labels[train_idx])))
        fold_cw[benign_idx] = fold_cw[benign_idx] * 2.5
        fold_cw[normal_idx] = fold_cw[normal_idx] * 1.3

        train_ds, train_n = make_train_ds(fold_train_dir, class_names)
        val_ds,   val_n,_ = make_eval_ds(fold_val_dir,   class_names)
        train_steps = math.ceil(train_n / BATCH_SIZE)
        val_steps   = math.ceil(val_n   / BATCH_SIZE)
        for model_list, prefix in [
            (fold_models, f"allexp_{block_size}_eff_f{fold}"),
        ]:
            TEMP = f"{prefix}_temp.keras"
            BEST = f"{prefix}_best.keras"

            # 1단계
            model = build_model()
            model.compile(optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
                          loss=focal_loss(2.0, 0.02), metrics=["accuracy"])
            model.fit(
                apply_mixup_cutmix(train_ds, 0.3, cutmix_prob=0.5),
                steps_per_epoch=train_steps,
                validation_data=val_ds, validation_steps=val_steps,
                epochs=EPOCHS_S1,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=10,
                                  restore_best_weights=True, verbose=0),
                    ModelCheckpoint(TEMP, monitor="val_loss",
                                    save_best_only=True, mode="min", verbose=0),
                    CosineAnnealingWarmRestart(lr_max=1e-3, lr_min=1e-5, T0=5, T_mult=2)
                ], class_weight=fold_cw, verbose=1)

            # 2단계
            model = load_model(TEMP, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
            bl = None
            for layer in model.layers:
                if hasattr(layer,"layers") and len(getattr(layer,"layers",[])) > 10:
                    bl = layer; break
            if bl:
                bl.trainable = True
                for layer in bl.layers[:-30]: layer.trainable = False

            swa_cb = SWACallback(start_epoch=SWA_START, swa_lr=1e-5)
            model.compile(optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
                          loss=focal_loss(2.0, 0.02), metrics=["accuracy"])
            model.fit(
                apply_mixup_cutmix(train_ds, 0.1, cutmix_prob=0.3),
                steps_per_epoch=train_steps,
                validation_data=val_ds, validation_steps=val_steps,
                epochs=EPOCHS_S2,
                callbacks=[
                    EarlyStopping(monitor="val_loss", patience=20,
                                  restore_best_weights=False, verbose=0),
                    ModelCheckpoint(BEST, monitor="val_loss",
                                    save_best_only=True, mode="min", verbose=1),
                    CosineAnnealingWarmRestart(lr_max=1e-4, lr_min=1e-7, T0=10, T_mult=2),
                    swa_cb
                ], class_weight=fold_cw, verbose=1)

            model.save(BEST)
            model_list.append(BEST)
            if os.path.exists(TEMP): os.remove(TEMP)
            del model
            tf.keras.backend.clear_session()
            gc.collect()

    # 앙상블 + TTA
    print(f"\n  🔮 앙상블 + TTA ({N_FOLDS}개 × {TTA_STEPS}회) [EfficientNetV2S]...")
    test_ds, test_n, test_files = make_eval_ds(test_dir, class_names)
    test_steps = math.ceil(test_n / BATCH_SIZE)

    y_true_list   = [yb.numpy() for _, yb in test_ds]
    y_true_onehot = np.concatenate(y_true_list, axis=0)
    y_true        = np.argmax(y_true_onehot, axis=1)

    all_fold_preds = []
    for fold, mp in enumerate(fold_models):
        fm = load_model(mp, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        tta_preds = []
        for t in range(TTA_STEPS):
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(fm.predict(ds_t, steps=test_steps, verbose=0))
            print(f"    Fold{fold+1} TTA {t+1}/{TTA_STEPS}", end="\r")
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del fm; tf.keras.backend.clear_session(); gc.collect()
    print()

    pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
    y_pred     = np.argmax(pred_probs, axis=1)
    test_acc   = float(np.mean(y_true == y_pred))
    print(f"  앙상블: {test_acc*100:.2f}%")

    # Pseudo Labeling
    print("  🔄 Pseudo Labeling...")
    pseudo_dir = os.path.join(split_dir, "pseudo")
    if os.path.exists(pseudo_dir): shutil.rmtree(pseudo_dir)
    pseudo_count = 0
    for fpath, prob in zip(test_files, pred_probs):
        if np.max(prob) >= 0.88:
            cls = class_names[np.argmax(prob)]
            os.makedirs(os.path.join(pseudo_dir,cls), exist_ok=True)
            shutil.copy2(fpath, os.path.join(pseudo_dir,cls,os.path.basename(fpath)))
            pseudo_count += 1
    print(f"  확신도 92%↑: {pseudo_count}장")

    if pseudo_count >= 10:
        pseudo_kfold = os.path.join(split_dir, "pseudo_kfold")
        if os.path.exists(pseudo_kfold): shutil.rmtree(pseudo_kfold)
        for cls in class_names:
            os.makedirs(os.path.join(pseudo_kfold,cls), exist_ok=True)
            for f in os.listdir(os.path.join(kfold_dir,cls)):
                if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
                    shutil.copy2(os.path.join(kfold_dir,cls,f),
                                 os.path.join(pseudo_kfold,cls,f))
            p = os.path.join(pseudo_dir,cls)
            if os.path.exists(p):
                for f in os.listdir(p):
                    shutil.copy2(os.path.join(p,f),
                                 os.path.join(pseudo_kfold,cls,f"pseudo_{f}"))

        pm = load_model(fold_models[0], custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        p_labels = []
        for cls in class_names:
            cnt = len([f for f in os.listdir(os.path.join(pseudo_kfold,cls))
                       if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
            p_labels.extend([class_to_idx[cls]]*cnt)
        p_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(p_labels), y=p_labels)))
        p_cw[benign_idx] = p_cw[benign_idx]*1.5
        p_cw[normal_idx] = p_cw[normal_idx]*1.3

        p_train_ds, p_n = make_train_ds(pseudo_kfold, class_names)
        p_val_ds, p_vn,_ = make_eval_ds(os.path.join(split_dir,"fold_0_val"), class_names)
        pm.compile(optimizer=AdamW(learning_rate=3e-5, weight_decay=1e-4),
                   loss=focal_loss(2.0,0.05), metrics=["accuracy"])
        pm.fit(apply_mixup(p_train_ds, 0.1),
               steps_per_epoch=math.ceil(p_n/BATCH_SIZE),
               validation_data=p_val_ds, validation_steps=math.ceil(p_vn/BATCH_SIZE),
               epochs=15,
               callbacks=[EarlyStopping(monitor="val_loss", patience=7,
                                        restore_best_weights=True, verbose=0)],
               class_weight=p_cw, verbose=0)

        tta_preds = []
        for t in range(TTA_STEPS):
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(pm.predict(ds_t, steps=test_steps, verbose=0))
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del pm; tf.keras.backend.clear_session(); gc.collect()

        pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
        y_pred     = np.argmax(pred_probs, axis=1)
        test_acc   = float(np.mean(y_true == y_pred))
        print(f"  Pseudo 후: {test_acc*100:.2f}%")

    cm      = confusion_matrix(y_true, y_pred)
    roc_auc = float(roc_auc_score(y_true_onehot[:len(pred_probs)], pred_probs,
                                   multi_class="ovr", average="macro"))
    per_class = {cls: float(cm[i,i]/cm[i].sum()) for i,cls in enumerate(class_names)}

    for mp in fold_models:
        if os.path.exists(mp): os.remove(mp)

    print(f"\n  ✅ Block{block_size}: {test_acc*100:.2f}% | ROC-AUC={roc_auc:.4f}")
    return {"block_size": block_size, "n_blocks": n_blocks,
            "acc": test_acc, "roc_auc": roc_auc,
            "per_class_acc": per_class, "cm": cm.tolist(), "class_names": class_names}

# ======================================================
# 8. 전체 실험 루프
# ======================================================
print("\n" + "="*60)
print("🔬 블록 암호화 크기별 실험 시작")
print(f"   블록: {BLOCK_SIZES}")
print(f"   K={N_FOLDS} | TTA={TTA_STEPS} | BATCH={BATCH_SIZE} | IMG={IMG_SIZE}")
print("="*60)

all_results = {}
result_path = os.path.join(WORKSPACE, "all_block_results.json")

if os.path.exists(result_path):
    with open(result_path) as f:
        saved = json_module.load(f)
    for k,v in saved.items():
        all_results[int(k)] = v
    print(f"\n✅ 이전 결과: {sorted(all_results.keys())}")

for block_size in BLOCK_SIZES:
    if block_size in all_results:
        print(f"\n⏭️  Block{block_size}: 완료({all_results[block_size]['acc']*100:.2f}%) → 스킵")
        continue
    print(f"\n📦 Block{block_size} 데이터셋 준비...")
    block_dir = generate_block_dataset(block_size)
    result    = run_experiment(block_size, block_dir)
    all_results[block_size] = result
    with open(result_path, "w") as f:
        json_module.dump({str(k):v for k,v in all_results.items()}, f, indent=2, ensure_ascii=False)
    print(f"  💾 중간 저장 완료")

# ======================================================
# 9. 결과 표 + 쉬운 해석
# ======================================================
print("\n\n" + "="*85)
print("📊 블록 암호화 크기별 실험 결과")
print("="*85)
print(f"  {'블록크기':>8}  {'정확도(전체)':>12}  {'ROC-AUC':>8}  {'양성(Benign)':>13}  {'악성(Malig)':>12}  {'정상(Normal)':>13}  {'평가':>6}")
print("-"*85)
for bs in BLOCK_SIZES:
    if bs not in all_results: continue
    r   = all_results[bs]
    b   = r["per_class_acc"].get("Bengin cases",    0)*100
    m   = r["per_class_acc"].get("Malignant cases", 0)*100
    n   = r["per_class_acc"].get("Normal cases",    0)*100
    acc = r["acc"]*100
    lbl = "원본" if bs>=512 else f"{bs}x{bs}"
    grade = "🟢 우수" if acc>=85 else ("🟡 양호" if acc>=70 else ("🟠 미흡" if acc>=55 else "🔴 불량"))
    print(f"  {lbl:>8}  {acc:>11.2f}%  {r['roc_auc']:>8.4f}"
          f"  {b:>12.1f}%  {m:>11.1f}%  {n:>12.1f}%  {grade}")
print("="*85)
print()
print("📋 클래스별 상세 분석")
print("-"*85)
for bs in BLOCK_SIZES:
    if bs not in all_results: continue
    r   = all_results[bs]
    b   = r["per_class_acc"].get("Bengin cases",    0)*100
    m   = r["per_class_acc"].get("Malignant cases", 0)*100
    n   = r["per_class_acc"].get("Normal cases",    0)*100
    acc = r["acc"]*100
    lbl = "원본" if bs>=512 else f"{bs}x{bs}"
    weak = []
    if b < 80: weak.append(f"양성↓({b:.0f}%)")
    if m < 80: weak.append(f"악성↓({m:.0f}%)")
    if n < 80: weak.append(f"정상↓({n:.0f}%)")
    note = "⚠️ " + ", ".join(weak) if weak else "✅ 모든 클래스 80%↑"
    print(f"  {lbl:>8}: 전체 {acc:.1f}% | Benign {b:.1f}% | Malignant {m:.1f}% | Normal {n:.1f}% | {note}")
print("="*85)

# 쉬운 해석
print("\n" + "="*75)
print("💡 쉬운 해석")
print("="*75)
done = [bs for bs in BLOCK_SIZES if bs in all_results]
best_bs  = max(done, key=lambda b: all_results[b]["acc"])
worst_bs = min(done, key=lambda b: all_results[b]["acc"])

print(f"""
🔐 블록 암호화란?
   CT 이미지를 블록 단위로 잘라서 섞는 방법입니다.
   블록이 작을수록 더 잘게 섞여서 사람이 알아보기 어렵습니다.
   (개인정보 보호 목적)

📊 실험 결과 요약:
   • 원본 (암호화 없음): {all_results.get(512, {}).get("acc", 0)*100:.1f}% - AI가 가장 잘 맞힘
   • 256x256 블록:       {all_results.get(256, {}).get("acc", 0)*100:.1f}% - 거의 원본과 비슷
   • 32x32 블록:         {all_results.get(32,  {}).get("acc", 0)*100:.1f}% - 성능 하락 시작
   • 4x4 블록:           {all_results.get(4,   {}).get("acc", 0)*100:.1f}% - 많이 어려워짐
   • 1x1 (픽셀 섞기):    {all_results.get(1,   {}).get("acc", 0)*100:.1f}% - 거의 맞추지 못함

🏆 최고 성능: {("원본" if best_bs>=512 else f"{best_bs}x{best_bs}")} → {all_results[best_bs]["acc"]*100:.2f}%
📉 최저 성능: {("원본" if worst_bs>=512 else f"{worst_bs}x{worst_bs}")} → {all_results[worst_bs]["acc"]*100:.2f}%

🎯 결론:
   → 블록이 클수록 (원본에 가까울수록) AI 성능 높음
   → 블록이 작을수록 (더 많이 섞을수록) 보안 강함
   → 보안과 AI 성능은 서로 반비례 관계
   → 적절한 균형점: 64x64 ~ 128x128 블록 추천
     (보안 유지하면서 80%+ 정확도 가능)

📋 클래스별 특징:
   • Malignant(악성): 블록 암호화에도 비교적 강함 (특징이 굵음)
   • Normal(정상):    중간 블록에서 취약 (세밀한 특징 의존)
   • Benign(양성):    블록 크기에 따라 가장 변동 큼
""")
print("="*75)

# ======================================================
# 10. 다양한 그래프 (10개)
# ======================================================
bs_done   = [bs for bs in BLOCK_SIZES if bs in all_results]
labels    = ["원본\n(512)" if bs>=512 else ("픽셀\n(1x1)" if bs==1 else f"{bs}x{bs}") for bs in bs_done]
labels_kr = ["원본" if bs>=512 else ("픽셀\n1x1" if bs==1 else f"{bs}x{bs}") for bs in bs_done]
accs     = [all_results[bs]["acc"]*100       for bs in bs_done]
roc_aucs = [all_results[bs]["roc_auc"]       for bs in bs_done]
b_accs   = [all_results[bs]["per_class_acc"].get("Bengin cases",   0)*100 for bs in bs_done]
m_accs   = [all_results[bs]["per_class_acc"].get("Malignant cases", 0)*100 for bs in bs_done]
n_accs   = [all_results[bs]["per_class_acc"].get("Normal cases",   0)*100 for bs in bs_done]
n_blocks = [all_results[bs]["n_blocks"]      for bs in bs_done]
x        = np.arange(len(bs_done))
cmap     = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(bs_done)))

def style_ax(ax):
    ax.set_facecolor("#161b22")
    for sp in ax.spines.values(): sp.set_color("#444444")
    ax.tick_params(colors="#aaaaaa", labelsize=9)
    ax.yaxis.grid(True, color="#333333", linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)

fig1, ax = plt.subplots(figsize=(16, 7))
fig1.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")
cmap_bars = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(bs_done)))
bars = ax.bar(x, accs, color=cmap_bars, edgecolor="#333", linewidth=0.8, zorder=3, width=0.6)
ax.axhline(max(accs), color="#ff6b6b", ls="--", lw=1.5, alpha=0.8, label=f"최고 {max(accs):.1f}%")
ax.axhline(min(accs), color="#4ecdc4", ls="--", lw=1.5, alpha=0.8, label=f"최저 {min(accs):.1f}%")
ax.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
for bar, acc, bs in zip(bars, accs, bs_done):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels_kr, fontsize=10)
ax.set_ylim(0, max(accs)+18)
ax.set_title("블록 암호화 크기별 AI 정확도 (Block Encryption Size vs Accuracy)\n왼쪽=보안 강함 (High Security) / 오른쪽=원본 (Original)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("정확도 (%)", color="#aaa", fontsize=12)
ax.set_xlabel("블록 크기", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph1_accuracy_bar.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프1 저장")

# ============ 그래프 2: ROC-AUC 꺾은선 ============
fig2, ax = plt.subplots(figsize=(14, 6))
fig2.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
ax.plot(labels_kr, roc_aucs, "o-", color="#7c83fd", lw=2.5, ms=12, mfc="white", mec="#7c83fd", zorder=3)
ax.fill_between(range(len(labels_kr)), roc_aucs, 0.4, alpha=0.1, color="#7c83fd")
for i,(lbl,val) in enumerate(zip(labels_kr,roc_aucs)):
    ax.annotate(f"{val:.4f}", (i,val), textcoords="offset points",
                xytext=(0,12), ha="center", color="#7c83fd", fontsize=9, fontweight="bold")
ax.axhline(0.9, color="#6bcb77", ls="--", lw=1, alpha=0.6, label="ROC 0.9 기준")
ax.axhline(0.8, color="#ffd93d", ls="--", lw=1, alpha=0.6, label="ROC 0.8 기준")
ax.set_ylim(0.4, 1.08)
ax.set_title("블록 크기별 ROC-AUC (Block Size vs ROC-AUC)\n1.0에 가까울수록 좋음 (Higher is Better)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("ROC-AUC", color="#aaa", fontsize=12); ax.set_xlabel("블록 크기", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph2_roc_auc.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프2 저장")

# ============ 그래프 3: 클래스별 정확도 그룹 막대 ============
fig3, ax = plt.subplots(figsize=(16, 7))
fig3.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
w = 0.25
ax.bar(x-w, b_accs, width=w, label="양성 Benign", color="#ff6b6b", edgecolor="#222", zorder=3)
ax.bar(x,   m_accs, width=w, label="악성 Malignant", color="#ffd93d", edgecolor="#222", zorder=3)
ax.bar(x+w, n_accs, width=w, label="정상 Normal", color="#6bcb77", edgecolor="#222", zorder=3)
ax.axhline(80, color="white", ls="--", lw=0.8, alpha=0.4, label="80% 기준선")
ax.set_xticks(x); ax.set_xticklabels(labels_kr); ax.set_ylim(0, 118)
ax.set_title("클래스별 정확도 (Per-Class Accuracy)\n악성 종양은 블록 암호화에도 강함 (Malignant is robust)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("정확도 (%)", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph3_class_accuracy.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프3 저장")

# ============ 그래프 4: 보안-성능 트레이드오프 산점도 ============
fig4, ax = plt.subplots(figsize=(10, 8))
fig4.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
security = [100-acc for acc in accs]
sc = ax.scatter(security, accs, c=accs, cmap="RdYlGn", s=300,
                edgecolors="white", lw=1.5, zorder=3, vmin=30, vmax=100)
for i,lbl in enumerate(labels_kr):
    ax.annotate(lbl, (security[i], accs[i]), textcoords="offset points",
                xytext=(8,5), color="white", fontsize=9, fontweight="bold")
ax.axvspan(0, 20, alpha=0.05, color="#6bcb77", label="고성능 구역")
ax.axvspan(60, 100, alpha=0.05, color="#ff6b6b", label="고보안 구역")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("AI 정확도 / Accuracy (%)", color="white", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax.set_xlabel("암호화 강도 / Encryption Strength (높을수록 보안 강함 / Higher = More Secure) →", color="#aaa", fontsize=12)
ax.set_ylabel("AI 정확도 (%)", color="#aaa", fontsize=12)
ax.set_title("보안성 vs AI 성능 트레이드오프 (Security vs Performance)\n이상적: 왼쪽 위 (Ideal: Top-Left) / 최악: 오른쪽 아래 (Worst: Bottom-Right)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph4_tradeoff.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프4 저장")

# ============ 그래프 5: 정확도 추세선 (전체+클래스별) ============
fig5, ax = plt.subplots(figsize=(16, 7))
fig5.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
ax.plot(labels_kr, accs,   "o-",  color="#f9ca24", lw=3, ms=10, label="전체 정확도 (Overall)", zorder=4, mfc="white", mec="#f9ca24")
ax.plot(labels_kr, b_accs, "s--", color="#ff6b6b", lw=1.8, ms=7, label="양성 Benign", zorder=3)
ax.plot(labels_kr, m_accs, "^--", color="#ffd93d", lw=1.8, ms=7, label="악성 Malignant", zorder=3)
ax.plot(labels_kr, n_accs, "D--", color="#6bcb77", lw=1.8, ms=7, label="정상 Normal", zorder=3)
ax.fill_between(range(len(labels_kr)), accs, alpha=0.08, color="#f9ca24")
ax.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
ax.set_ylim(0, 115)
ax.set_title("블록 암호화 강도에 따른 정확도 변화 추세 (Accuracy Trend by Encryption Strength)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("정확도 (%)", color="#aaa", fontsize=12); ax.set_xlabel("블록 크기 (오른쪽→왼쪽으로 갈수록 보안 강함)", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10, loc="upper right")
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph5_trend.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프5 저장")

# ============ 그래프 6: 블록 수 vs 정확도 이중축 ============
fig6, ax6 = plt.subplots(figsize=(14, 6))
fig6.patch.set_facecolor("#0d1117"); ax6.set_facecolor("#161b22")
ax6b = ax6.twinx()
ax6.plot(labels_kr, accs, "o-", color="#f9ca24", lw=2.5, ms=10, zorder=4, mfc="white", mec="#f9ca24", label="정확도")
ax6b.bar(x, n_blocks, alpha=0.3, color="#74b9ff", zorder=2, label="블록 수")
ax6b.set_yscale("log")
for i,(lbl,acc) in enumerate(zip(labels_kr,accs)):
    ax6.text(i, acc+1.5, f"{acc:.1f}%", ha="center", color="#f9ca24", fontsize=9, fontweight="bold")
ax6.set_ylim(0, max(accs)+20); ax6.set_xticks(x); ax6.set_xticklabels(labels_kr)
ax6.set_title("블록 수 vs AI 정확도 (Number of Blocks vs Accuracy)\n블록 많을수록 정확도 하락 (More Blocks = Lower Accuracy)", color="white", fontsize=14, fontweight="bold", pad=15)
ax6.set_ylabel("정확도 (%)", color="#f9ca24", fontsize=12)
ax6b.set_ylabel("블록 수 / Number of Blocks (log scale)", color="#74b9ff", fontsize=12)
ax6.tick_params(axis="y", colors="#f9ca24"); ax6b.tick_params(axis="y", colors="#74b9ff")
ax6.tick_params(axis="x", colors="#aaa")
for sp in ax6.spines.values(): sp.set_color("#444")
ax6.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax6.set_axisbelow(True)
lines1, labels1 = ax6.get_legend_handles_labels()
lines2, labels2 = ax6b.get_legend_handles_labels()
ax6.legend(lines1+lines2, labels1+labels2, facecolor="#222", labelcolor="white", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph6_blocks_vs_acc.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프6 저장")

# ============ 그래프 7: 히트맵 (클래스별 정확도) ============
fig7, ax = plt.subplots(figsize=(14, 5))
fig7.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
heatmap_data = np.array([b_accs, m_accs, n_accs])
im = ax.imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(labels_kr))); ax.set_xticklabels(labels_kr, color="white", fontsize=10)
ax.set_yticks([0,1,2]); ax.set_yticklabels(["양성\n(Benign)", "악성\n(Malignant)", "정상\n(Normal)"], color="white", fontsize=11)
for i in range(3):
    for j in range(len(bs_done)):
        val = heatmap_data[i,j]
        ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                color="black" if val>60 else "white", fontsize=10, fontweight="bold")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("정확도 / Accuracy (%)", color="white"); cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax.set_title("클래스별 정확도 히트맵 (Per-Class Accuracy Heatmap)\n초록=높음 (Green=High) / 빨강=낮음 (Red=Low)", color="white", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph7_heatmap.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프7 저장")

# ============ 그래프 8: 레이더 차트 (주요 4개 블록 비교) ============
from matplotlib.patches import FancyArrowPatch
key_bs = [bs for bs in [512, 64, 16, 1] if bs in all_results]
categories = ["전체 정확도", "Benign", "Malignant", "Normal", "ROC-AUC×100"]
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
fig8, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig8.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
colors_radar = ["#f9ca24", "#6bcb77", "#ff6b6b", "#7c83fd"]
for idx, bs in enumerate(key_bs):
    r = all_results[bs]
    b = r["per_class_acc"].get("Bengin cases",   0)*100
    m = r["per_class_acc"].get("Malignant cases", 0)*100
    n = r["per_class_acc"].get("Normal cases",   0)*100
    vals = [r["acc"]*100, b, m, n, r["roc_auc"]*100]
    vals += vals[:1]
    lbl = "원본" if bs>=512 else ("픽셀" if bs==1 else f"Block{bs}")
    ax.plot(angles, vals, "o-", lw=2, color=colors_radar[idx], label=lbl)
    ax.fill(angles, vals, alpha=0.05, color=colors_radar[idx])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, color="white", fontsize=11)
ax.set_ylim(0, 100); ax.set_yticks([20,40,60,80,100])
ax.set_yticklabels(["20","40","60","80","100"], color="#aaa", fontsize=8)
ax.tick_params(colors="#aaa")
ax.grid(color="#444", linestyle="--", alpha=0.6)
ax.spines["polar"].set_color("#444")
ax.set_title("주요 블록 크기별 성능 레이더 차트 (Radar Chart by Block Size)", color="white", fontsize=14, fontweight="bold", pad=20)
ax.legend(facecolor="#222", labelcolor="white", fontsize=11, loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph8_radar.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프8 저장")

# ============ 그래프 9: Confusion Matrix (주요 2개) ============
key_blocks_cm = [bs for bs in [512, 1] if bs in all_results]
if key_blocks_cm:
    fig9, axes9 = plt.subplots(1, len(key_blocks_cm), figsize=(14, 6))
    fig9.patch.set_facecolor("#0d1117")
    if len(key_blocks_cm) == 1: axes9 = [axes9]
    for ax_cm, bs in zip(axes9, key_blocks_cm):
        ax_cm.set_facecolor("#161b22")
        r      = all_results[bs]
        cm_arr = np.array(r["cm"])
        cn     = ["Benign", "Malignant", "Normal"]
        lbl    = "원본 (암호화 없음)" if bs>=512 else f"픽셀 섞기 (완전 암호화)"
        im = ax_cm.imshow(cm_arr, interpolation="nearest", cmap="Blues")
        plt.colorbar(im, ax=ax_cm)
        ax_cm.set_xticks(range(len(cn))); ax_cm.set_xticklabels(cn, color="white", fontsize=10)
        ax_cm.set_yticks(range(len(cn))); ax_cm.set_yticklabels(cn, color="white", fontsize=10)
        for i in range(len(cn)):
            for j in range(len(cn)):
                ax_cm.text(j, i, str(cm_arr[i,j]), ha="center", va="center",
                           color="white" if cm_arr[i,j]<cm_arr.max()/2 else "black", fontsize=14, fontweight="bold")
        ax_cm.set_title(f"{lbl}\n정확도: {r['acc']*100:.1f}%", color="white", fontsize=12, fontweight="bold", pad=10)
        ax_cm.set_xlabel("AI 예측 (Predicted)", color="#aaa", fontsize=11)
        ax_cm.set_ylabel("실제 정답 (Actual)", color="#aaa", fontsize=11)
    fig9.suptitle("Confusion Matrix: 원본 vs 완전 암호화 비교\n대각선=정답 (Diagonal=Correct) / 나머지=오답 (Others=Wrong)", color="white", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(WORKSPACE, "graph9_confusion.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show(); print("✅ 그래프9 저장")

# ============ 그래프 10: 종합 요약 대시보드 ============
fig10 = plt.figure(figsize=(20, 12))
fig10.patch.set_facecolor("#0d1117")
gs10 = GridSpec(2, 3, figure=fig10, hspace=0.45, wspace=0.35)

ax10_1 = fig10.add_subplot(gs10[0, :2])
ax10_1.set_facecolor("#161b22")
bars10 = ax10_1.bar(x, accs, color=plt.cm.RdYlGn(np.linspace(0.15,0.95,len(bs_done))), edgecolor="#333", zorder=3)
for bar, acc in zip(bars10, accs):
    ax10_1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
ax10_1.set_xticks(x); ax10_1.set_xticklabels(labels_kr, fontsize=9)
ax10_1.set_ylim(0, max(accs)+15)
ax10_1.set_title("블록 크기별 정확도", color="white", fontsize=12, fontweight="bold")
ax10_1.set_ylabel("정확도 (%)", color="#aaa")
for sp in ax10_1.spines.values(): sp.set_color("#444")
ax10_1.tick_params(colors="#aaa"); ax10_1.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_1.set_axisbelow(True)

ax10_2 = fig10.add_subplot(gs10[0, 2])
ax10_2.set_facecolor("#161b22"); ax10_2.axis("off")
best_bs2  = max(bs_done, key=lambda b: all_results[b]["acc"])
worst_bs2 = min(bs_done, key=lambda b: all_results[b]["acc"])
best_lbl  = "원본" if best_bs2>=512  else f"{best_bs2}x{best_bs2}"
worst_lbl = "원본" if worst_bs2>=512 else f"{worst_bs2}x{worst_bs2}"
summary_txt = (
    f"📊 실험 요약\n\n"
    f"총 실험: {len(bs_done)}가지 블록 크기\n\n"
    f"🏆 최고 성능\n"
    f"  {best_lbl}\n"
    f"  정확도: {all_results[best_bs2]['acc']*100:.1f}%\n"
    f"  ROC-AUC: {all_results[best_bs2]['roc_auc']:.4f}\n\n"
    f"📉 최저 성능\n"
    f"  {worst_lbl}\n"
    f"  정확도: {all_results[worst_bs2]['acc']*100:.1f}%\n\n"
    f"🎯 추천 균형점\n"
    f"  64x64 ~ 128x128\n"
    f"  보안 + 성능 균형\n\n"
    f"💡 핵심 발견\n"
    f"  악성 종양 탐지는\n"
    f"  암호화에도 강함!"
)
ax10_2.text(0.05, 0.95, summary_txt, transform=ax10_2.transAxes,
            color="white", fontsize=10, verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#21262d", edgecolor="#555"))
ax10_2.set_title("실험 결론", color="white", fontsize=12, fontweight="bold")

ax10_3 = fig10.add_subplot(gs10[1, 0])
ax10_3.set_facecolor("#161b22")
ax10_3.plot(range(len(labels_kr)), roc_aucs, "o-", color="#7c83fd", lw=2, ms=8, mfc="white")
ax10_3.set_xticks(range(len(labels_kr))); ax10_3.set_xticklabels(labels_kr, fontsize=8)
ax10_3.set_ylim(0.5, 1.05); ax10_3.set_title("ROC-AUC", color="white", fontsize=12, fontweight="bold")
ax10_3.set_ylabel("ROC-AUC", color="#aaa")
for sp in ax10_3.spines.values(): sp.set_color("#444")
ax10_3.tick_params(colors="#aaa"); ax10_3.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_3.set_axisbelow(True)

ax10_4 = fig10.add_subplot(gs10[1, 1])
ax10_4.set_facecolor("#161b22")
ax10_4.plot(range(len(labels_kr)), b_accs, "s-", color="#ff6b6b", lw=1.5, ms=6, label="양성 (Benign)")
ax10_4.plot(range(len(labels_kr)), m_accs, "^-", color="#ffd93d", lw=1.5, ms=6, label="악성 (Malignant)")
ax10_4.plot(range(len(labels_kr)), n_accs, "D-", color="#6bcb77", lw=1.5, ms=6, label="정상 (Normal)")
ax10_4.set_xticks(range(len(labels_kr))); ax10_4.set_xticklabels(labels_kr, fontsize=8)
ax10_4.set_ylim(0, 115); ax10_4.set_title("클래스별 정확도", color="white", fontsize=12, fontweight="bold")
ax10_4.legend(facecolor="#222", labelcolor="white", fontsize=8)
for sp in ax10_4.spines.values(): sp.set_color("#444")
ax10_4.tick_params(colors="#aaa"); ax10_4.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_4.set_axisbelow(True)

ax10_5 = fig10.add_subplot(gs10[1, 2])
ax10_5.set_facecolor("#161b22")
sc10 = ax10_5.scatter(security, accs, c=accs, cmap="RdYlGn", s=150,
                      edgecolors="white", lw=1, zorder=3, vmin=30, vmax=100)
for i,lbl in enumerate(labels_kr):
    ax10_5.annotate(lbl, (security[i],accs[i]), textcoords="offset points",
                    xytext=(5,4), color="white", fontsize=7)
ax10_5.set_xlabel("암호화 강도 (Encryption Strength) →", color="#aaa")
ax10_5.set_ylabel("AI 정확도 (%)", color="#aaa")
ax10_5.set_title("보안 vs 성능", color="white", fontsize=12, fontweight="bold")
for sp in ax10_5.spines.values(): sp.set_color("#444")
ax10_5.tick_params(colors="#aaa"); ax10_5.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_5.set_axisbelow(True)

fig10.suptitle("블록 암호화 크기별 AI 성능 종합 대시보드 (Block Encryption AI Performance Dashboard)\nPrivacy-Preserving Lung Cancer Classification",
               color="white", fontsize=15, fontweight="bold", y=1.01)
plt.savefig(os.path.join(WORKSPACE, "graph10_dashboard.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프10 (대시보드) 저장")

print(f"""
✅ 모든 그래프 저장 완료!
   graph1_accuracy_bar.png   - 전체 정확도 막대
   graph2_roc_auc.png        - ROC-AUC 꺾은선
   graph3_class_accuracy.png - 클래스별 그룹 막대
   graph4_tradeoff.png       - 보안-성능 트레이드오프
   graph5_trend.png          - 추세선
   graph6_blocks_vs_acc.png  - 블록 수 vs 정확도
   graph7_heatmap.png        - 히트맵
   graph8_radar.png          - 레이더 차트
   graph9_confusion.png      - Confusion Matrix
   graph10_dashboard.png     - 종합 대시보드
✅ JSON: {result_path}
🎉 모든 실험 완료!
""")


In [ ]:
# 블러링 처리 하기
import os
import math
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
                                      Dropout, BatchNormalization, Concatenate)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.regularizers import l2
from tensorflow.keras import mixed_precision
import tensorflow.keras.backend as K
from PIL import Image, ImageFilter
import random
import json as json_module
import gc

# ======================================================
# 한글 폰트 설정
# ======================================================
import matplotlib
import matplotlib.font_manager as fm
import glob

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

cache_dir = matplotlib.get_cachedir()
for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
    try: os.remove(f)
    except: pass

fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [font_name]
matplotlib.rcParams["axes.unicode_minus"] = False

print(f"✅ 한글 폰트 적용: {font_name}")

# ======================================================
# RTX 4090 최적화
# ======================================================
mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(False)
print("GPU:", tf.config.list_physical_devices("GPU"))

# ======================================================
# 설정
# ======================================================
# ★ 블러 sigma 값 목록 (0 = 원본, 숫자가 클수록 더 강한 블러)
BLUR_SIGMAS  = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]

BASE_TRAIN   = "/workspace/Train"
WORKSPACE    = "/workspace"
IMG_SIZE     = 512
BATCH_SIZE   = 4
EPOCHS_S1    = 30
EPOCHS_S2    = 60
NUM_CLASSES  = 3
SEED         = 42
N_FOLDS      = 7
TTA_STEPS    = 20
SWA_START    = 20
TARGET_COUNT = 800
TEST_RATIO   = 0.10
AUTOTUNE     = tf.data.AUTOTUNE

# ======================================================
# 1. 블러 처리  ← 블록 암호화 대체
# ======================================================
def apply_gaussian_blur(img_array, sigma):
    """
    PIL GaussianBlur로 sigma 강도 블러 적용
    sigma=0 이면 원본 그대로 반환
    """
    if sigma == 0:
        return img_array.copy()
    pil_img = Image.fromarray(img_array.astype(np.uint8))
    blurred = pil_img.filter(ImageFilter.GaussianBlur(radius=sigma))
    return np.array(blurred, dtype=np.uint8)


def generate_blur_dataset(sigma, seed=42):
    """
    sigma=0  → 원본 디렉터리 그대로 사용
    sigma>0  → Train_BlurSigma{sigma}/ 에 블러 이미지 저장
    """
    if sigma == 0:
        print(f"  Sigma0: 원본 사용")
        return BASE_TRAIN

    out_dir = os.path.join(WORKSPACE, f"Train_BlurSigma{sigma}")
    classes = sorted([d for d in os.listdir(BASE_TRAIN)
                      if os.path.isdir(os.path.join(BASE_TRAIN, d))])

    total_src = sum(
        len([f for f in os.listdir(os.path.join(BASE_TRAIN, c))
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
        for c in classes
    )
    total_ex = (
        sum(
            len([f for f in os.listdir(os.path.join(out_dir, c))
                 if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
            for c in classes
            if os.path.exists(os.path.join(out_dir, c))
        )
        if os.path.exists(out_dir) else 0
    )

    if total_ex >= total_src:
        print(f"  BlurSigma{sigma}: 이미 존재 ({total_ex}장) → 스킵")
        return out_dir

    print(f"  BlurSigma{sigma}: 블러 처리 중 ({total_src}장)...")
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        for f in os.listdir(os.path.join(BASE_TRAIN, cls)):
            if not f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                continue
            dst = os.path.join(out_dir, cls, f)
            if os.path.exists(dst):
                continue
            try:
                img = np.array(
                    Image.open(os.path.join(BASE_TRAIN, cls, f)).convert("RGB"),
                    dtype=np.uint8
                )
                Image.fromarray(apply_gaussian_blur(img, sigma)).save(
                    dst, "JPEG", quality=95
                )
            except:
                pass
    print(f"  BlurSigma{sigma}: 완료!")
    return out_dir

# ======================================================
# 2. 오버샘플링 (원본 폴더 보호)
# ======================================================
def oversample_dir(base_dir, target_count, seed=42):
    out_dir = base_dir.rstrip("/") + "_oversampled"
    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, d))])

    if os.path.exists(out_dir):
        all_done = all(
            len([f for f in os.listdir(os.path.join(out_dir, c))
                 if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]) >= target_count
            for c in classes if os.path.exists(os.path.join(out_dir, c))
        ) and all(os.path.exists(os.path.join(out_dir, c)) for c in classes)
        if all_done:
            total = sum(
                len([f for f in os.listdir(os.path.join(out_dir, c))
                     if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
                for c in classes
            )
            print(f"    오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    random.seed(seed)
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
                     and not f.startswith("aug_")]
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir, cls, f), dst)

    for cls in classes:
        cls_out = os.path.join(out_dir, cls)
        files = [f for f in os.listdir(cls_out)
                 if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
                     and not f.startswith("aug_")]
        needed = target_count - len(files)
        if needed <= 0:
            print(f"    {cls}: {len(files)}장 (충분)"); continue
        print(f"    {cls}: {len(files)}장 → {target_count}장 ({needed}장 증강)")
        generated = 0
        while generated < needed:
            src_f = random.choice(src_files)
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(random.uniform(-30, 30), expand=False)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
                from PIL import ImageEnhance
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"),
                         "JPEG", quality=95)
                generated += 1
            except:
                continue

    total = sum(
        len([f for f in os.listdir(os.path.join(out_dir, c))
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
        for c in classes
    )
    print(f"    ✅ 오버샘플링 완료: {total}장 → {out_dir}")
    return out_dir

# ======================================================
# 3. Train/Test 분리
# ======================================================
def prepare_split(base_dir, split_dir, test_ratio=0.10, seed=42):
    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")
    if os.path.exists(test_dir) and os.path.exists(kfold_dir):
        print(f"    Split 폴더 이미 존재 → 재사용"); return
    np.random.seed(seed)
    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, d))])
    for cls in classes:
        os.makedirs(os.path.join(test_dir,  cls), exist_ok=True)
        os.makedirs(os.path.join(kfold_dir, cls), exist_ok=True)
    for cls in classes:
        files = [f for f in os.listdir(os.path.join(base_dir, cls))
                 if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))]
        np.random.shuffle(files)
        n_test = int(len(files) * test_ratio)
        for f in files[:n_test]:
            shutil.copy2(os.path.join(base_dir, cls, f), os.path.join(test_dir, cls, f))
        for f in files[n_test:]:
            shutil.copy2(os.path.join(base_dir, cls, f), os.path.join(kfold_dir, cls, f))
        print(f"    {cls}: kfold={len(files)-n_test}, test={n_test}")

# ======================================================
# 4. tf.data 파이프라인
# ======================================================
def get_paths_labels(directory, class_names):
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        if not os.path.exists(cls_dir): continue
        for f in os.listdir(cls_dir):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                paths.append(os.path.join(cls_dir, f))
                one_hot = [0.0] * len(class_names)
                one_hot[idx] = 1.0
                labels.append(one_hot)
    return paths, labels

@tf.function
def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label

@tf.function
def augment_train(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.25 * 255)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.25)
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=5.0, dtype=tf.float32)
    img = tf.clip_by_value(img + noise, 0.0, 255.0)
    return img, label

@tf.function
def cutmix_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx   = tf.random.shuffle(tf.range(batch_size))
    lam   = tf.cast(tf.random.uniform([], alpha, 1.0 - alpha), tf.float32)
    h     = tf.cast(tf.shape(x)[1], tf.float32)
    w     = tf.cast(tf.shape(x)[2], tf.float32)
    cut_h = tf.cast(h * tf.math.sqrt(1.0 - lam), tf.int32)
    cut_w = tf.cast(w * tf.math.sqrt(1.0 - lam), tf.int32)
    cx    = tf.random.uniform([], 0, tf.cast(h, tf.int32), dtype=tf.int32)
    cy    = tf.random.uniform([], 0, tf.cast(w, tf.int32), dtype=tf.int32)
    x1    = tf.clip_by_value(cx - cut_h // 2, 0, tf.cast(h, tf.int32))
    y1    = tf.clip_by_value(cy - cut_w // 2, 0, tf.cast(w, tf.int32))
    x2    = tf.clip_by_value(cx + cut_h // 2, 0, tf.cast(h, tf.int32))
    y2    = tf.clip_by_value(cy + cut_w // 2, 0, tf.cast(w, tf.int32))
    mask  = tf.ones([x2 - x1, y2 - y1, tf.shape(x)[3]])
    paddings = [[x1, tf.shape(x)[1] - x2], [y1, tf.shape(x)[2] - y2], [0, 0]]
    mask  = tf.pad(mask, paddings)
    mask  = tf.expand_dims(mask, 0)
    x     = tf.cast(x, tf.float32)
    mixed = x * (1.0 - mask) + tf.gather(x, idx) * mask
    lam_actual = 1.0 - tf.cast((x2 - x1) * (y2 - y1), tf.float32) / (h * w)
    y     = tf.cast(y, tf.float32)
    mixed_y = lam_actual * y + (1.0 - lam_actual) * tf.gather(y, idx)
    return mixed, mixed_y

def apply_cutmix(ds, alpha=0.3):
    return ds.map(lambda x, y: cutmix_batch(x, y, alpha),
                  num_parallel_calls=AUTOTUNE).prefetch(1)

def apply_mixup_cutmix(ds, alpha=0.3, cutmix_prob=0.5):
    def _apply(x, y):
        use_cutmix = tf.random.uniform([]) < cutmix_prob
        mx, my = mixup_batch(x, y, alpha)
        cx, cy = cutmix_batch(x, y, alpha)
        return tf.cond(use_cutmix, lambda: (cx, cy), lambda: (mx, my))
    return ds.map(_apply, num_parallel_calls=AUTOTUNE).prefetch(1)

@tf.function
def augment_tta(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15 * 255)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    return img, label

def make_train_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(min(n, 1000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.map(lambda x, y: augment_train(x, y), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(1)
    return ds, n

def make_eval_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(1)
    return ds, n, paths

def make_tta_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.map(lambda x, y: augment_tta(x, y), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(1)
    return ds, n

@tf.function
def mixup_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx = tf.random.shuffle(tf.range(batch_size))
    lam = tf.cast(tf.random.uniform([], minval=alpha, maxval=1.0 - alpha + 0.001), tf.float32)
    x = tf.cast(x, tf.float32); y = tf.cast(y, tf.float32)
    return lam * x + (1 - lam) * tf.gather(x, idx), lam * y + (1 - lam) * tf.gather(y, idx)

def apply_mixup(ds, alpha=0.3):
    return ds.map(lambda x, y: mixup_batch(x, y, alpha),
                  num_parallel_calls=AUTOTUNE).prefetch(1)

# ======================================================
# 5. 학습 기법
# ======================================================
def focal_loss(gamma=2.0, label_smoothing=0.05):
    def loss_fn(y_true, y_pred):
        num_cls   = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_sm = y_true * (1.0 - label_smoothing) + (label_smoothing / num_cls)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce        = -y_true_sm * tf.math.log(y_pred)
        weight    = y_true_sm * tf.math.pow(1.0 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))
    return loss_fn

class CosineAnnealingWarmRestart(tf.keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, T0=10, T_mult=2):
        super().__init__()
        self.lr_max = lr_max; self.lr_min = lr_min
        self.T0 = T0; self.T_mult = T_mult; self.T_cur = 0; self.T_i = T0
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1 + math.cos(math.pi * self.T_cur / self.T_i))
        K.set_value(self.model.optimizer.learning_rate, lr)
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur = 0; self.T_i = int(self.T_i * self.T_mult)

class SWACallback(tf.keras.callbacks.Callback):
    def __init__(self, start_epoch, swa_lr=1e-5):
        super().__init__()
        self.start_epoch = start_epoch; self.swa_lr = swa_lr
        self.swa_weights = None; self.n_averaged = 0
    def on_epoch_end(self, epoch, logs=None):
        if epoch >= self.start_epoch:
            if self.swa_weights is None:
                self.swa_weights = [w.copy() for w in self.model.get_weights()]
            else:
                for i, w in enumerate(self.model.get_weights()):
                    self.swa_weights[i] = (self.swa_weights[i] * self.n_averaged + w) / (self.n_averaged + 1)
            self.n_averaged += 1
            K.set_value(self.model.optimizer.learning_rate, self.swa_lr)
    def on_train_end(self, logs=None):
        if self.swa_weights is not None:
            self.model.set_weights(self.swa_weights)
            print(f"\n✅ SWA 완료: {self.n_averaged}개 평균")

# ======================================================
# 6. 모델
# ======================================================
def build_model():
    base = EfficientNetV2S(weights="imagenet", include_top=False,
                           input_shape=(IMG_SIZE, IMG_SIZE, 3), include_preprocessing=True)
    base.trainable = False
    gap = GlobalAveragePooling2D()(base.output)
    gmp = GlobalMaxPooling2D()(base.output)
    x   = Concatenate()([gap, gmp])
    x = Dense(512, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation("swish")(x)
    x = Dropout(0.35)(x)
    x = Dense(256, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation("swish")(x)
    x = Dropout(0.25)(x)
    out = Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return Model(inputs=base.input, outputs=out)

# ======================================================
# 7. 단일 블러 sigma 실험  ← run_experiment 대체
# ======================================================
def run_experiment(sigma, blur_dir):
    label = "원본" if sigma == 0 else f"BlurSigma{sigma}"
    print(f"\n{'='*60}\n🌫️  {label}\n{'='*60}")

    split_dir = os.path.join(WORKSPACE, f"AllExp_Split_Blur{sigma}")
    gc.collect()

    print("  오버샘플링...")
    oversampled_dir = oversample_dir(blur_dir, TARGET_COUNT)
    print("  Train/Test 분리...")
    prepare_split(oversampled_dir, split_dir, TEST_RATIO)

    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")

    class_names  = sorted([d for d in os.listdir(kfold_dir)
                            if os.path.isdir(os.path.join(kfold_dir, d))])
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    benign_idx   = class_to_idx.get("Bengin cases", 0)
    normal_idx   = class_to_idx.get("Normal cases", 2)

    all_paths, all_labels = [], []
    for cls in class_names:
        for f in os.listdir(os.path.join(kfold_dir, cls)):
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                all_paths.append(os.path.join(kfold_dir, cls, f))
                all_labels.append(class_to_idx[cls])
    all_paths  = np.array(all_paths)
    all_labels = np.array(all_labels)
    print(f"  학습: {len(all_paths)}장 | 테스트: "
          f"{sum(len(os.listdir(os.path.join(test_dir, c))) for c in class_names)}장")

    skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
        print(f"\n  --- Fold {fold+1}/{N_FOLDS} ---")
        fold_train_dir = os.path.join(split_dir, f"fold_{fold}_train")
        fold_val_dir   = os.path.join(split_dir, f"fold_{fold}_val")

        if not os.path.exists(fold_train_dir):
            for cls in class_names:
                os.makedirs(os.path.join(fold_train_dir, cls), exist_ok=True)
                os.makedirs(os.path.join(fold_val_dir,   cls), exist_ok=True)
            for idx in train_idx:
                src = all_paths[idx]; cls = class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_train_dir, cls, os.path.basename(src)))
            for idx in val_idx:
                src = all_paths[idx]; cls = class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_val_dir, cls, os.path.basename(src)))

        fold_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(all_labels[train_idx]), y=all_labels[train_idx])))
        fold_cw[benign_idx] = fold_cw[benign_idx] * 2.5
        fold_cw[normal_idx] = fold_cw[normal_idx] * 1.3

        train_ds, train_n = make_train_ds(fold_train_dir, class_names)
        val_ds, val_n, _  = make_eval_ds(fold_val_dir,   class_names)
        train_steps = math.ceil(train_n / BATCH_SIZE)
        val_steps   = math.ceil(val_n   / BATCH_SIZE)

        prefix = f"allexp_blur{sigma}_eff_f{fold}"
        TEMP   = f"{prefix}_temp.keras"
        BEST   = f"{prefix}_best.keras"

        # 1단계
        model = build_model()
        model.compile(optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
                      loss=focal_loss(2.0, 0.02), metrics=["accuracy"])
        model.fit(
            apply_mixup_cutmix(train_ds, 0.3, cutmix_prob=0.5),
            steps_per_epoch=train_steps,
            validation_data=val_ds, validation_steps=val_steps,
            epochs=EPOCHS_S1,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=10,
                              restore_best_weights=True, verbose=0),
                ModelCheckpoint(TEMP, monitor="val_loss",
                                save_best_only=True, mode="min", verbose=0),
                CosineAnnealingWarmRestart(lr_max=1e-3, lr_min=1e-5, T0=5, T_mult=2)
            ], class_weight=fold_cw, verbose=1)

        # 2단계
        model = load_model(TEMP, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        bl = None
        for layer in model.layers:
            if hasattr(layer, "layers") and len(getattr(layer, "layers", [])) > 10:
                bl = layer; break
        if bl:
            bl.trainable = True
            for layer in bl.layers[:-30]: layer.trainable = False

        swa_cb = SWACallback(start_epoch=SWA_START, swa_lr=1e-5)
        model.compile(optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
                      loss=focal_loss(2.0, 0.02), metrics=["accuracy"])
        model.fit(
            apply_mixup_cutmix(train_ds, 0.1, cutmix_prob=0.3),
            steps_per_epoch=train_steps,
            validation_data=val_ds, validation_steps=val_steps,
            epochs=EPOCHS_S2,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=20,
                              restore_best_weights=False, verbose=0),
                ModelCheckpoint(BEST, monitor="val_loss",
                                save_best_only=True, mode="min", verbose=1),
                CosineAnnealingWarmRestart(lr_max=1e-4, lr_min=1e-7, T0=10, T_mult=2),
                swa_cb
            ], class_weight=fold_cw, verbose=1)

        model.save(BEST)
        fold_models.append(BEST)
        if os.path.exists(TEMP): os.remove(TEMP)
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    # 앙상블 + TTA
    print(f"\n  🔮 앙상블 + TTA ({N_FOLDS}개 × {TTA_STEPS}회) [EfficientNetV2S]...")
    test_ds, test_n, test_files = make_eval_ds(test_dir, class_names)
    test_steps = math.ceil(test_n / BATCH_SIZE)

    y_true_list   = [yb.numpy() for _, yb in test_ds]
    y_true_onehot = np.concatenate(y_true_list, axis=0)
    y_true        = np.argmax(y_true_onehot, axis=1)

    all_fold_preds = []
    for fold, mp in enumerate(fold_models):
        fm = load_model(mp, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        tta_preds = []
        for t in range(TTA_STEPS):
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(fm.predict(ds_t, steps=test_steps, verbose=0))
            print(f"    Fold{fold+1} TTA {t+1}/{TTA_STEPS}", end="\r")
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del fm; tf.keras.backend.clear_session(); gc.collect()
    print()

    pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
    y_pred     = np.argmax(pred_probs, axis=1)
    test_acc   = float(np.mean(y_true == y_pred))
    print(f"  앙상블: {test_acc*100:.2f}%")

    # Pseudo Labeling
    print("  🔄 Pseudo Labeling...")
    pseudo_dir = os.path.join(split_dir, "pseudo")
    if os.path.exists(pseudo_dir): shutil.rmtree(pseudo_dir)
    pseudo_count = 0
    for fpath, prob in zip(test_files, pred_probs):
        if np.max(prob) >= 0.88:
            cls = class_names[np.argmax(prob)]
            os.makedirs(os.path.join(pseudo_dir, cls), exist_ok=True)
            shutil.copy2(fpath, os.path.join(pseudo_dir, cls, os.path.basename(fpath)))
            pseudo_count += 1
    print(f"  확신도 88%↑: {pseudo_count}장")

    if pseudo_count >= 10:
        pseudo_kfold = os.path.join(split_dir, "pseudo_kfold")
        if os.path.exists(pseudo_kfold): shutil.rmtree(pseudo_kfold)
        for cls in class_names:
            os.makedirs(os.path.join(pseudo_kfold, cls), exist_ok=True)
            for f in os.listdir(os.path.join(kfold_dir, cls)):
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    shutil.copy2(os.path.join(kfold_dir, cls, f),
                                 os.path.join(pseudo_kfold, cls, f))
            p = os.path.join(pseudo_dir, cls)
            if os.path.exists(p):
                for f in os.listdir(p):
                    shutil.copy2(os.path.join(p, f),
                                 os.path.join(pseudo_kfold, cls, f"pseudo_{f}"))

        pm = load_model(fold_models[0], custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        p_labels = []
        for cls in class_names:
            cnt = len([f for f in os.listdir(os.path.join(pseudo_kfold, cls))
                       if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))])
            p_labels.extend([class_to_idx[cls]] * cnt)
        p_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(p_labels), y=p_labels)))
        p_cw[benign_idx] = p_cw[benign_idx] * 1.5
        p_cw[normal_idx] = p_cw[normal_idx] * 1.3

        p_train_ds, p_n     = make_train_ds(pseudo_kfold, class_names)
        p_val_ds, p_vn, _   = make_eval_ds(os.path.join(split_dir, "fold_0_val"), class_names)
        pm.compile(optimizer=AdamW(learning_rate=3e-5, weight_decay=1e-4),
                   loss=focal_loss(2.0, 0.05), metrics=["accuracy"])
        pm.fit(apply_mixup(p_train_ds, 0.1),
               steps_per_epoch=math.ceil(p_n / BATCH_SIZE),
               validation_data=p_val_ds, validation_steps=math.ceil(p_vn / BATCH_SIZE),
               epochs=15,
               callbacks=[EarlyStopping(monitor="val_loss", patience=7,
                                        restore_best_weights=True, verbose=0)],
               class_weight=p_cw, verbose=0)

        tta_preds = []
        for t in range(TTA_STEPS):
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(pm.predict(ds_t, steps=test_steps, verbose=0))
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del pm; tf.keras.backend.clear_session(); gc.collect()

        pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
        y_pred     = np.argmax(pred_probs, axis=1)
        test_acc   = float(np.mean(y_true == y_pred))
        print(f"  Pseudo 후: {test_acc*100:.2f}%")

    cm      = confusion_matrix(y_true, y_pred)
    roc_auc = float(roc_auc_score(y_true_onehot[:len(pred_probs)], pred_probs,
                                   multi_class="ovr", average="macro"))
    per_class = {cls: float(cm[i, i] / cm[i].sum()) for i, cls in enumerate(class_names)}

    for mp in fold_models:
        if os.path.exists(mp): os.remove(mp)

    print(f"\n  ✅ BlurSigma{sigma}: {test_acc*100:.2f}% | ROC-AUC={roc_auc:.4f}")
    return {"sigma": sigma, "acc": test_acc, "roc_auc": roc_auc,
            "per_class_acc": per_class, "cm": cm.tolist(), "class_names": class_names}

# ======================================================
# 8. 전체 실험 루프
# ======================================================
print("\n" + "=" * 60)
print("🔬 블러 처리 강도별 실험 시작")
print(f"   Sigma: {BLUR_SIGMAS}")
print(f"   K={N_FOLDS} | TTA={TTA_STEPS} | BATCH={BATCH_SIZE} | IMG={IMG_SIZE}")
print("=" * 60)

all_results = {}
result_path = os.path.join(WORKSPACE, "all_blur_results.json")

if os.path.exists(result_path):
    with open(result_path) as f:
        saved = json_module.load(f)
    for k, v in saved.items():
        all_results[int(k)] = v
    print(f"\n✅ 이전 결과: {sorted(all_results.keys())}")

for sigma in BLUR_SIGMAS:
    if sigma in all_results:
        print(f"\n⏭️  BlurSigma{sigma}: 완료({all_results[sigma]['acc']*100:.2f}%) → 스킵")
        continue
    print(f"\n📦 BlurSigma{sigma} 데이터셋 준비...")
    blur_dir = generate_blur_dataset(sigma)
    result   = run_experiment(sigma, blur_dir)
    all_results[sigma] = result
    with open(result_path, "w") as f:
        json_module.dump({str(k): v for k, v in all_results.items()}, f, indent=2, ensure_ascii=False)
    print(f"  💾 중간 저장 완료")

# ======================================================
# 9. 결과 표 + 쉬운 해석
# ======================================================
print("\n\n" + "=" * 85)
print("📊 블러 강도별 실험 결과")
print("=" * 85)
print(f"  {'Sigma':>8}  {'정확도(전체)':>12}  {'ROC-AUC':>8}  {'양성(Benign)':>13}  {'악성(Malig)':>12}  {'정상(Normal)':>13}  {'평가':>6}")
print("-" * 85)
for sigma in BLUR_SIGMAS:
    if sigma not in all_results: continue
    r   = all_results[sigma]
    b   = r["per_class_acc"].get("Bengin cases",    0) * 100
    m   = r["per_class_acc"].get("Malignant cases", 0) * 100
    n   = r["per_class_acc"].get("Normal cases",    0) * 100
    acc = r["acc"] * 100
    lbl = "원본" if sigma == 0 else f"σ={sigma}"
    grade = "🟢 우수" if acc >= 85 else ("🟡 양호" if acc >= 70 else ("🟠 미흡" if acc >= 55 else "🔴 불량"))
    print(f"  {lbl:>8}  {acc:>11.2f}%  {r['roc_auc']:>8.4f}"
          f"  {b:>12.1f}%  {m:>11.1f}%  {n:>12.1f}%  {grade}")
print("=" * 85)
print()
print("📋 클래스별 상세 분석")
print("-" * 85)
for sigma in BLUR_SIGMAS:
    if sigma not in all_results: continue
    r   = all_results[sigma]
    b   = r["per_class_acc"].get("Bengin cases",    0) * 100
    m   = r["per_class_acc"].get("Malignant cases", 0) * 100
    n   = r["per_class_acc"].get("Normal cases",    0) * 100
    acc = r["acc"] * 100
    lbl = "원본" if sigma == 0 else f"σ={sigma}"
    weak = []
    if b < 80: weak.append(f"양성↓({b:.0f}%)")
    if m < 80: weak.append(f"악성↓({m:.0f}%)")
    if n < 80: weak.append(f"정상↓({n:.0f}%)")
    note = "⚠️ " + ", ".join(weak) if weak else "✅ 모든 클래스 80%↑"
    print(f"  {lbl:>8}: 전체 {acc:.1f}% | Benign {b:.1f}% | Malignant {m:.1f}% | Normal {n:.1f}% | {note}")
print("=" * 85)

# 쉬운 해석
print("\n" + "=" * 75)
print("💡 쉬운 해석")
print("=" * 75)
done     = [sigma for sigma in BLUR_SIGMAS if sigma in all_results]
best_s   = max(done, key=lambda s: all_results[s]["acc"])
worst_s  = min(done, key=lambda s: all_results[s]["acc"])
best_lbl  = "원본" if best_s  == 0 else f"σ={best_s}"
worst_lbl = "원본" if worst_s == 0 else f"σ={worst_s}"

print(f"""
🌫️  가우시안 블러란?
   CT 이미지에 블러(뿌옇게 처리)를 적용하는 방법입니다.
   Sigma 값이 클수록 더 강하게 뿌옇게 처리됩니다.
   (개인정보 보호 / 비식별화 목적)

📊 실험 결과 요약:
   • 원본 (σ=0):    {all_results.get(0,   {}).get("acc", 0)*100:.1f}% - AI가 가장 잘 맞힘
   • 약한 블러(σ=2): {all_results.get(2,   {}).get("acc", 0)*100:.1f}% - 거의 원본과 비슷
   • 중간 블러(σ=16):{all_results.get(16,  {}).get("acc", 0)*100:.1f}% - 성능 하락 시작
   • 강한 블러(σ=64):{all_results.get(64,  {}).get("acc", 0)*100:.1f}% - 많이 어려워짐
   • 최강 블러(σ=256):{all_results.get(256,{}).get("acc", 0)*100:.1f}% - 거의 맞추지 못함

🏆 최고 성능: {best_lbl}  → {all_results[best_s]["acc"]*100:.2f}%
📉 최저 성능: {worst_lbl} → {all_results[worst_s]["acc"]*100:.2f}%

🎯 결론:
   → Sigma가 작을수록 (원본에 가까울수록) AI 성능 높음
   → Sigma가 클수록 (더 뿌옇게 처리할수록) 보안 강함
   → 보안과 AI 성능은 서로 반비례 관계
   → 적절한 균형점: σ=4 ~ σ=8 추천
     (보안 유지하면서 80%+ 정확도 가능)

📋 클래스별 특징:
   • Malignant(악성): 블러에도 비교적 강함 (특징이 굵음)
   • Normal(정상):    강한 블러에서 취약 (세밀한 특징 의존)
   • Benign(양성):    블러 강도에 따라 가장 변동 큼
""")
print("=" * 75)

# ======================================================
# 10. 다양한 그래프 (10개)
# ======================================================
done      = [sigma for sigma in BLUR_SIGMAS if sigma in all_results]
labels_kr = ["원본\n(σ=0)" if s == 0 else f"σ={s}" for s in done]
accs      = [all_results[s]["acc"] * 100       for s in done]
roc_aucs  = [all_results[s]["roc_auc"]         for s in done]
b_accs    = [all_results[s]["per_class_acc"].get("Bengin cases",    0) * 100 for s in done]
m_accs    = [all_results[s]["per_class_acc"].get("Malignant cases", 0) * 100 for s in done]
n_accs    = [all_results[s]["per_class_acc"].get("Normal cases",    0) * 100 for s in done]
x         = np.arange(len(done))
security  = [100 - acc for acc in accs]

def style_ax(ax):
    ax.set_facecolor("#161b22")
    for sp in ax.spines.values(): sp.set_color("#444444")
    ax.tick_params(colors="#aaaaaa", labelsize=9)
    ax.yaxis.grid(True, color="#333333", linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)

# ─── 그래프 1: 정확도 막대 ───
fig1, ax = plt.subplots(figsize=(16, 7))
fig1.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
bars = ax.bar(x, accs, color=plt.cm.RdYlGn(np.linspace(0.95, 0.15, len(done))),
              edgecolor="#333", linewidth=0.8, zorder=3, width=0.6)
ax.axhline(max(accs), color="#ff6b6b", ls="--", lw=1.5, alpha=0.8, label=f"최고 {max(accs):.1f}%")
ax.axhline(min(accs), color="#4ecdc4", ls="--", lw=1.5, alpha=0.8, label=f"최저 {min(accs):.1f}%")
ax.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
for bar, acc, s in zip(bars, accs, done):
    grade = "🟢" if acc >= 85 else ("🟡" if acc >= 70 else ("🟠" if acc >= 55 else "🔴"))
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels_kr, fontsize=10)
ax.set_ylim(0, max(accs) + 18)
ax.set_title("블러 강도별 AI 정확도 (Blur Sigma vs Accuracy)\n왼쪽=원본 (Original) / 오른쪽=강한 블러 (Heavy Blur)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("정확도 (%)", color="#aaa", fontsize=12)
ax.set_xlabel("블러 Sigma (σ)", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph1_accuracy_bar.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프1 저장")

# ─── 그래프 2: ROC-AUC 꺾은선 ───
fig2, ax = plt.subplots(figsize=(14, 6))
fig2.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
ax.plot(labels_kr, roc_aucs, "o-", color="#7c83fd", lw=2.5, ms=12, mfc="white", mec="#7c83fd", zorder=3)
ax.fill_between(range(len(labels_kr)), roc_aucs, 0.4, alpha=0.1, color="#7c83fd")
for i, (lbl, val) in enumerate(zip(labels_kr, roc_aucs)):
    ax.annotate(f"{val:.4f}", (i, val), textcoords="offset points",
                xytext=(0, 12), ha="center", color="#7c83fd", fontsize=9, fontweight="bold")
ax.axhline(0.9, color="#6bcb77", ls="--", lw=1, alpha=0.6, label="ROC 0.9 기준")
ax.axhline(0.8, color="#ffd93d", ls="--", lw=1, alpha=0.6, label="ROC 0.8 기준")
ax.set_ylim(0.4, 1.08)
ax.set_title("블러 강도별 ROC-AUC (Blur Sigma vs ROC-AUC)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("ROC-AUC", color="#aaa", fontsize=12); ax.set_xlabel("블러 Sigma (σ)", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph2_roc_auc.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프2 저장")

# ─── 그래프 3: 클래스별 그룹 막대 ───
fig3, ax = plt.subplots(figsize=(16, 7))
fig3.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
w = 0.25
ax.bar(x - w, b_accs, width=w, label="양성 Benign",     color="#ff6b6b", edgecolor="#222", zorder=3)
ax.bar(x,     m_accs, width=w, label="악성 Malignant",  color="#ffd93d", edgecolor="#222", zorder=3)
ax.bar(x + w, n_accs, width=w, label="정상 Normal",     color="#6bcb77", edgecolor="#222", zorder=3)
ax.axhline(80, color="white", ls="--", lw=0.8, alpha=0.4, label="80% 기준선")
ax.set_xticks(x); ax.set_xticklabels(labels_kr); ax.set_ylim(0, 118)
ax.set_title("클래스별 정확도 (Per-Class Accuracy by Blur Sigma)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("정확도 (%)", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph3_class_accuracy.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프3 저장")

# ─── 그래프 4: 보안-성능 트레이드오프 ───
fig4, ax = plt.subplots(figsize=(10, 8))
fig4.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
sc = ax.scatter(security, accs, c=accs, cmap="RdYlGn", s=300,
                edgecolors="white", lw=1.5, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(labels_kr):
    ax.annotate(lbl.replace("\n", ""), (security[i], accs[i]),
                textcoords="offset points", xytext=(8, 5), color="white", fontsize=9, fontweight="bold")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("AI 정확도 / Accuracy (%)", color="white", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax.set_xlabel("블러 강도 / Blur Strength (높을수록 보안 강함) →", color="#aaa", fontsize=12)
ax.set_ylabel("AI 정확도 (%)", color="#aaa", fontsize=12)
ax.set_title("보안성 vs AI 성능 트레이드오프 (Security vs Performance)", color="white", fontsize=14, fontweight="bold", pad=15)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph4_tradeoff.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프4 저장")

# ─── 그래프 5: 추세선 ───
fig5, ax = plt.subplots(figsize=(16, 7))
fig5.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
ax.plot(labels_kr, accs,   "o-",  color="#f9ca24", lw=3,   ms=10, label="전체 정확도", zorder=4, mfc="white", mec="#f9ca24")
ax.plot(labels_kr, b_accs, "s--", color="#ff6b6b", lw=1.8, ms=7,  label="양성 Benign")
ax.plot(labels_kr, m_accs, "^--", color="#ffd93d", lw=1.8, ms=7,  label="악성 Malignant")
ax.plot(labels_kr, n_accs, "D--", color="#6bcb77", lw=1.8, ms=7,  label="정상 Normal")
ax.fill_between(range(len(labels_kr)), accs, alpha=0.08, color="#f9ca24")
ax.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
ax.set_ylim(0, 115)
ax.set_title("블러 강도에 따른 정확도 변화 추세 (Accuracy Trend by Blur Sigma)", color="white", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("정확도 (%)", color="#aaa", fontsize=12)
ax.set_xlabel("블러 Sigma (왼쪽=원본 / 오른쪽=강한 블러)", color="#aaa", fontsize=12)
ax.legend(facecolor="#222", labelcolor="white", fontsize=10)
for sp in ax.spines.values(): sp.set_color("#444")
ax.tick_params(colors="#aaa"); ax.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph5_trend.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프5 저장")

# ─── 그래프 6: Sigma vs 정확도 (log scale x축) ───
fig6, ax6 = plt.subplots(figsize=(14, 6))
fig6.patch.set_facecolor("#0d1117"); ax6.set_facecolor("#161b22")
sigma_vals = [max(s, 0.5) for s in done]   # log축 위해 0 → 0.5
ax6.plot(sigma_vals, accs, "o-", color="#f9ca24", lw=2.5, ms=10, zorder=4, mfc="white", mec="#f9ca24")
ax6.set_xscale("log")
ax6.set_xticks(sigma_vals)
ax6.set_xticklabels(["원본\n(σ=0)" if s == 0 else f"σ={s}" for s in done], fontsize=9)
for i, (sv, acc) in enumerate(zip(sigma_vals, accs)):
    ax6.text(sv, acc + 1.5, f"{acc:.1f}%", ha="center", color="#f9ca24", fontsize=9, fontweight="bold")
ax6.set_ylim(0, max(accs) + 20)
ax6.set_title("블러 Sigma vs AI 정확도 (log scale)\nSigma 클수록 정확도 하락 (Higher Sigma = Lower Accuracy)", color="white", fontsize=14, fontweight="bold", pad=15)
ax6.set_ylabel("정확도 (%)", color="#f9ca24", fontsize=12)
ax6.set_xlabel("블러 Sigma (log scale) →", color="#aaa", fontsize=12)
ax6.tick_params(axis="y", colors="#f9ca24"); ax6.tick_params(axis="x", colors="#aaa")
for sp in ax6.spines.values(): sp.set_color("#444")
ax6.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax6.set_axisbelow(True)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph6_sigma_vs_acc.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프6 저장")

# ─── 그래프 7: 히트맵 ───
fig7, ax = plt.subplots(figsize=(14, 5))
fig7.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
heatmap_data = np.array([b_accs, m_accs, n_accs])
im = ax.imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
ax.set_xticks(range(len(labels_kr))); ax.set_xticklabels(labels_kr, color="white", fontsize=10)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["양성\n(Benign)", "악성\n(Malignant)", "정상\n(Normal)"], color="white", fontsize=11)
for i in range(3):
    for j in range(len(done)):
        val = heatmap_data[i, j]
        ax.text(j, i, f"{val:.0f}%", ha="center", va="center",
                color="black" if val > 60 else "white", fontsize=10, fontweight="bold")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("정확도 / Accuracy (%)", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax.set_title("클래스별 정확도 히트맵 (Per-Class Accuracy Heatmap)", color="white", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph7_heatmap.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프7 저장")

# ─── 그래프 8: 레이더 차트 ───
key_sigmas = [s for s in [0, 4, 32, 256] if s in all_results]
categories = ["전체 정확도", "Benign", "Malignant", "Normal", "ROC-AUC×100"]
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist(); angles += angles[:1]
fig8, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig8.patch.set_facecolor("#0d1117"); ax.set_facecolor("#161b22")
colors_radar = ["#f9ca24", "#6bcb77", "#ff6b6b", "#7c83fd"]
for idx, s in enumerate(key_sigmas):
    r = all_results[s]
    b = r["per_class_acc"].get("Bengin cases",    0) * 100
    m = r["per_class_acc"].get("Malignant cases", 0) * 100
    n = r["per_class_acc"].get("Normal cases",    0) * 100
    vals = [r["acc"] * 100, b, m, n, r["roc_auc"] * 100]; vals += vals[:1]
    lbl = "원본" if s == 0 else f"σ={s}"
    ax.plot(angles, vals, "o-", lw=2, color=colors_radar[idx], label=lbl)
    ax.fill(angles, vals, alpha=0.05, color=colors_radar[idx])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, color="white", fontsize=11)
ax.set_ylim(0, 100); ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(["20", "40", "60", "80", "100"], color="#aaa", fontsize=8)
ax.grid(color="#444", linestyle="--", alpha=0.6)
ax.spines["polar"].set_color("#444")
ax.set_title("주요 블러 강도별 성능 레이더 차트 (Radar Chart by Blur Sigma)",
             color="white", fontsize=14, fontweight="bold", pad=20)
ax.legend(facecolor="#222", labelcolor="white", fontsize=11, loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, "graph8_radar.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프8 저장")

# ─── 그래프 9: Confusion Matrix ───
key_sigma_cm = [s for s in [0, 256] if s in all_results]
if key_sigma_cm:
    fig9, axes9 = plt.subplots(1, len(key_sigma_cm), figsize=(14, 6))
    fig9.patch.set_facecolor("#0d1117")
    if len(key_sigma_cm) == 1: axes9 = [axes9]
    for ax_cm, s in zip(axes9, key_sigma_cm):
        ax_cm.set_facecolor("#161b22")
        r      = all_results[s]
        cm_arr = np.array(r["cm"])
        cn     = ["Benign", "Malignant", "Normal"]
        lbl    = "원본 (블러 없음)" if s == 0 else f"최강 블러 (σ={s})"
        im = ax_cm.imshow(cm_arr, interpolation="nearest", cmap="Blues")
        plt.colorbar(im, ax=ax_cm)
        ax_cm.set_xticks(range(len(cn))); ax_cm.set_xticklabels(cn, color="white", fontsize=10)
        ax_cm.set_yticks(range(len(cn))); ax_cm.set_yticklabels(cn, color="white", fontsize=10)
        for i in range(len(cn)):
            for j in range(len(cn)):
                ax_cm.text(j, i, str(cm_arr[i, j]), ha="center", va="center",
                           color="white" if cm_arr[i, j] < cm_arr.max() / 2 else "black",
                           fontsize=14, fontweight="bold")
        ax_cm.set_title(f"{lbl}\n정확도: {r['acc']*100:.1f}%", color="white", fontsize=12, fontweight="bold", pad=10)
        ax_cm.set_xlabel("AI 예측 (Predicted)", color="#aaa", fontsize=11)
        ax_cm.set_ylabel("실제 정답 (Actual)",  color="#aaa", fontsize=11)
    fig9.suptitle("Confusion Matrix: 원본 vs 최강 블러 비교", color="white", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(WORKSPACE, "graph9_confusion.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
    plt.show(); print("✅ 그래프9 저장")

# ─── 그래프 10: 종합 대시보드 ───
fig10 = plt.figure(figsize=(20, 12))
fig10.patch.set_facecolor("#0d1117")
gs10 = GridSpec(2, 3, figure=fig10, hspace=0.45, wspace=0.35)

ax10_1 = fig10.add_subplot(gs10[0, :2]); ax10_1.set_facecolor("#161b22")
bars10 = ax10_1.bar(x, accs, color=plt.cm.RdYlGn(np.linspace(0.95, 0.15, len(done))),
                    edgecolor="#333", zorder=3)
for bar, acc in zip(bars10, accs):
    ax10_1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
ax10_1.set_xticks(x); ax10_1.set_xticklabels(labels_kr, fontsize=9)
ax10_1.set_ylim(0, max(accs) + 15)
ax10_1.set_title("블러 강도별 정확도", color="white", fontsize=12, fontweight="bold")
ax10_1.set_ylabel("정확도 (%)", color="#aaa")
for sp in ax10_1.spines.values(): sp.set_color("#444")
ax10_1.tick_params(colors="#aaa"); ax10_1.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_1.set_axisbelow(True)

ax10_2 = fig10.add_subplot(gs10[0, 2]); ax10_2.set_facecolor("#161b22"); ax10_2.axis("off")
summary_txt = (
    f"📊 실험 요약\n\n"
    f"총 실험: {len(done)}가지 블러 강도\n\n"
    f"🏆 최고 성능\n"
    f"  {'원본' if best_s==0 else f'σ={best_s}'}\n"
    f"  정확도: {all_results[best_s]['acc']*100:.1f}%\n"
    f"  ROC-AUC: {all_results[best_s]['roc_auc']:.4f}\n\n"
    f"📉 최저 성능\n"
    f"  {'원본' if worst_s==0 else f'σ={worst_s}'}\n"
    f"  정확도: {all_results[worst_s]['acc']*100:.1f}%\n\n"
    f"🎯 추천 균형점\n"
    f"  σ=4 ~ σ=8\n"
    f"  보안 + 성능 균형\n\n"
    f"💡 핵심 발견\n"
    f"  악성 종양 탐지는\n"
    f"  블러에도 강함!"
)
ax10_2.text(0.05, 0.95, summary_txt, transform=ax10_2.transAxes,
            color="white", fontsize=10, verticalalignment="top", fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#21262d", edgecolor="#555"))
ax10_2.set_title("실험 결론", color="white", fontsize=12, fontweight="bold")

ax10_3 = fig10.add_subplot(gs10[1, 0]); ax10_3.set_facecolor("#161b22")
ax10_3.plot(range(len(labels_kr)), roc_aucs, "o-", color="#7c83fd", lw=2, ms=8, mfc="white")
ax10_3.set_xticks(range(len(labels_kr))); ax10_3.set_xticklabels(labels_kr, fontsize=8)
ax10_3.set_ylim(0.5, 1.05); ax10_3.set_title("ROC-AUC", color="white", fontsize=12, fontweight="bold")
ax10_3.set_ylabel("ROC-AUC", color="#aaa")
for sp in ax10_3.spines.values(): sp.set_color("#444")
ax10_3.tick_params(colors="#aaa"); ax10_3.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_3.set_axisbelow(True)

ax10_4 = fig10.add_subplot(gs10[1, 1]); ax10_4.set_facecolor("#161b22")
ax10_4.plot(range(len(labels_kr)), b_accs, "s-", color="#ff6b6b", lw=1.5, ms=6, label="양성 (Benign)")
ax10_4.plot(range(len(labels_kr)), m_accs, "^-", color="#ffd93d", lw=1.5, ms=6, label="악성 (Malignant)")
ax10_4.plot(range(len(labels_kr)), n_accs, "D-", color="#6bcb77", lw=1.5, ms=6, label="정상 (Normal)")
ax10_4.set_xticks(range(len(labels_kr))); ax10_4.set_xticklabels(labels_kr, fontsize=8)
ax10_4.set_ylim(0, 115); ax10_4.set_title("클래스별 정확도", color="white", fontsize=12, fontweight="bold")
ax10_4.legend(facecolor="#222", labelcolor="white", fontsize=8)
for sp in ax10_4.spines.values(): sp.set_color("#444")
ax10_4.tick_params(colors="#aaa"); ax10_4.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_4.set_axisbelow(True)

ax10_5 = fig10.add_subplot(gs10[1, 2]); ax10_5.set_facecolor("#161b22")
sc10 = ax10_5.scatter(security, accs, c=accs, cmap="RdYlGn", s=150,
                      edgecolors="white", lw=1, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(labels_kr):
    ax10_5.annotate(lbl.replace("\n", ""), (security[i], accs[i]),
                    textcoords="offset points", xytext=(5, 4), color="white", fontsize=7)
ax10_5.set_xlabel("블러 강도 →", color="#aaa")
ax10_5.set_ylabel("AI 정확도 (%)", color="#aaa")
ax10_5.set_title("보안 vs 성능", color="white", fontsize=12, fontweight="bold")
for sp in ax10_5.spines.values(): sp.set_color("#444")
ax10_5.tick_params(colors="#aaa"); ax10_5.yaxis.grid(True, color="#333", linestyle="--", alpha=0.5); ax10_5.set_axisbelow(True)

fig10.suptitle("블러 강도별 AI 성능 종합 대시보드 (Gaussian Blur AI Performance Dashboard)\nPrivacy-Preserving Lung Cancer Classification",
               color="white", fontsize=15, fontweight="bold", y=1.01)
plt.savefig(os.path.join(WORKSPACE, "graph10_dashboard.png"), dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show(); print("✅ 그래프10 (대시보드) 저장")

print(f"""
✅ 모든 그래프 저장 완료!
   graph1_accuracy_bar.png   - 전체 정확도 막대
   graph2_roc_auc.png        - ROC-AUC 꺾은선
   graph3_class_accuracy.png - 클래스별 그룹 막대
   graph4_tradeoff.png       - 보안-성능 트레이드오프
   graph5_trend.png          - 추세선
   graph6_sigma_vs_acc.png   - Sigma vs 정확도 (log scale)
   graph7_heatmap.png        - 히트맵
   graph8_radar.png          - 레이더 차트
   graph9_confusion.png      - Confusion Matrix
   graph10_dashboard.png     - 종합 대시보드
✅ JSON: {result_path}  →  all_blur_results.json
🎉 모든 실험 완료!
""")


In [ ]:
# 풀링 실험 - 원본 코드 기반
import os
import shutil
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.applications import MobileNetV2
import random
import warnings
warnings.filterwarnings("ignore")

print("modules loaded")

# =====================================================
# 설정
# =====================================================
BASE_DIR     = "/workspace/Lung Cancer/Train"
TARGET_COUNT = 1000
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 64
EPOCHS       = 50
EPOCHS_FT    = 20
EPOCHS_S2    = 50
POOL_SIZES   = [1, 2, 3, 4, 5, 6, 7]
SAVE_DIR     = "/workspace/viz"
os.makedirs(SAVE_DIR, exist_ok=True)

# =====================================================
# 오버샘플링 (원본 보호)
# =====================================================
def oversample_dir(base_dir, target_count, seed=42):
    out_dir = base_dir.rstrip("/") + "_oversampled"
    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, d))])
    if os.path.exists(out_dir):
        all_done = all(
            os.path.exists(os.path.join(out_dir, c)) and
            len([f for f in os.listdir(os.path.join(out_dir, c))
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]) >= target_count
            for c in classes
        )
        if all_done:
            total = sum(len([f for f in os.listdir(os.path.join(out_dir, c))
                             if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                        for c in classes)
            print(f"오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    random.seed(seed)
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir, cls, f), dst)

    for cls in classes:
        cls_out   = os.path.join(out_dir, cls)
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]
        cur_files = [f for f in os.listdir(cls_out)
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
        needed = target_count - len(cur_files)
        if needed <= 0:
            print(f"  {cls}: {len(cur_files)}장 (충분)"); continue
        print(f"  {cls}: {len(cur_files)}장 → {target_count}장")
        generated = 0
        while generated < needed:
            src_f = random.choice(src_files)
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(random.uniform(-30, 30), expand=False)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
                from PIL import ImageEnhance
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"), "JPEG", quality=95)
                generated += 1
            except: continue

    total = sum(len([f for f in os.listdir(os.path.join(out_dir, c))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                for c in classes)
    print(f"오버샘플링 완료: {total}장")
    return out_dir

# =====================================================
# 오버샘플링 실행
# =====================================================
print("\n오버샘플링 시작...")
train_dir = oversample_dir(BASE_DIR, TARGET_COUNT)

# =====================================================
# 원본 코드 방식: filepaths + labels → DataFrame
# =====================================================
filepaths = []
labels    = []

classes_list = sorted([d for d in os.listdir(train_dir)
                        if os.path.isdir(os.path.join(train_dir, d))])

for cls in classes_list:
    cls_dir = os.path.join(train_dir, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
            filepaths.append(os.path.join(cls_dir, f))
            labels.append(cls)

df = pd.concat([pd.Series(filepaths, name="filepaths"),
                pd.Series(labels,    name="labels")], axis=1)

print(f"\n전체 데이터: {len(df)}장")
print(df["labels"].value_counts())

# =====================================================
# 원본 코드 방식: train / valid / test 분리
# =====================================================
train_df, dummy_df = train_test_split(df, train_size=0.8, shuffle=True,
                                       random_state=123, stratify=df["labels"])
valid_df, test_df  = train_test_split(dummy_df, train_size=0.5, shuffle=True,
                                       random_state=123, stratify=dummy_df["labels"])

print(f"\n학습: {len(train_df)}장 | 검증: {len(valid_df)}장 | 테스트: {len(test_df)}장")

# =====================================================
# 원본 코드 방식: ImageDataGenerator
# =====================================================
tr_gen = ImageDataGenerator(rescale=1./255, rotation_range=15,
                              zoom_range=0.1, horizontal_flip=True)
ts_gen = ImageDataGenerator(rescale=1./255)

train_gen = tr_gen.flow_from_dataframe(train_df, x_col="filepaths", y_col="labels",
                                        target_size=IMG_SIZE, class_mode="categorical",
                                        color_mode="rgb", shuffle=True,  batch_size=BATCH_SIZE)
valid_gen = ts_gen.flow_from_dataframe(valid_df, x_col="filepaths", y_col="labels",
                                        target_size=IMG_SIZE, class_mode="categorical",
                                        color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE)
test_gen  = ts_gen.flow_from_dataframe(test_df,  x_col="filepaths", y_col="labels",
                                        target_size=IMG_SIZE, class_mode="categorical",
                                        color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE)

g_dict     = train_gen.class_indices
class_names = list(g_dict.keys())
print(f"\n클래스: {g_dict}")

# =====================================================
# STEP 1: MobileNetV2 사전학습
# 원본 코드의 VGG 구조 대신 MobileNetV2 사용
# (풀링 실험을 위해 Conv 레이어 분리가 필요하기 때문)
# =====================================================
print("\n" + "="*55)
print("STEP 1: MobileNetV2 사전학습")
print("="*55)

base_model = MobileNetV2(weights="imagenet", include_top=False,
                          input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
base_model.trainable = False

x   = GlobalAveragePooling2D()(base_model.output)
x   = Dropout(0.3)(x)
out = Dense(len(class_names), activation="softmax")(x)
pretrain_model = Model(inputs=base_model.input, outputs=out)

# 원본 코드와 동일: Adamax, categorical_crossentropy
pretrain_model.compile(Adamax(learning_rate=0.001),
                        loss="categorical_crossentropy", metrics=["accuracy"])

print(f"[1단계] 헤드 학습 ({EPOCHS}ep)...")
pretrain_model.fit(train_gen, epochs=EPOCHS, verbose=1, validation_data=valid_gen)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
pretrain_model.compile(Adamax(learning_rate=0.0001),
                        loss="categorical_crossentropy", metrics=["accuracy"])

print(f"\n[2단계] Fine-tuning ({EPOCHS_FT}ep)...")
history_pretrain = pretrain_model.fit(train_gen, epochs=EPOCHS_FT, verbose=1,
                                       validation_data=valid_gen)

print(f"\n✅ 사전학습 완료!")
print(f"   최종 학습 정확도: {history_pretrain.history['accuracy'][-1]*100:.2f}%")
print(f"   최종 검증 정확도: {history_pretrain.history['val_accuracy'][-1]*100:.2f}%")

# =====================================================
# conv_extractor: pretrain_model에서 직접 추출
# (가중치 공유 보장 - 핵심!)
# =====================================================
conv_output = None
for layer in pretrain_model.layers:
    if hasattr(layer, "output_shape"):
        shape = layer.output_shape
        if isinstance(shape, tuple) and len(shape) == 4:
            conv_output = layer.output

conv_extractor = Model(inputs=pretrain_model.input, outputs=conv_output,
                        name="conv_extractor")
conv_extractor.trainable = False

feat_h, feat_w, feat_c = conv_extractor.output_shape[1:]
print(f"특징맵 크기: {feat_h}×{feat_w}×{feat_c}")

# =====================================================
# STEP 2: 풀링 크기별 실험
# =====================================================
print("\n" + "="*55)
print("STEP 2: 풀링 크기별 실험")
print("="*55)

results = {}

for pool_size in POOL_SIZES:
    print(f"\n--- Pool {pool_size}×{pool_size} ---")

    if pool_size >= feat_h:
        x          = GlobalAveragePooling2D()(conv_extractor.output)
        compressed = feat_c
        out_h, out_w = 1, 1
        label = "GlobalAvgPool"
    else:
        x     = MaxPooling2D(pool_size=(pool_size, pool_size))(conv_extractor.output)
        out_h = feat_h // pool_size
        out_w = feat_w // pool_size
        x     = Flatten()(x)
        compressed = out_h * out_w * feat_c
        label = f"MaxPool {pool_size}×{pool_size}"

    x   = Dropout(0.3)(x)
    out = Dense(len(class_names), activation="softmax")(x)

    pipeline = Model(inputs=conv_extractor.input, outputs=out,
                      name=f"pipeline_pool{pool_size}")

    # 원본 코드와 동일: Adamax, categorical_crossentropy
    pipeline.compile(Adamax(learning_rate=0.001),
                      loss="categorical_crossentropy", metrics=["accuracy"])

    from tensorflow.keras.callbacks import EarlyStopping
    es = EarlyStopping(monitor="val_accuracy", patience=10,
                        restore_best_weights=True, verbose=1)

    hist = pipeline.fit(train_gen, epochs=EPOCHS_S2, verbose=1,
                         validation_data=valid_gen, callbacks=[es])

    # 원본 코드 방식으로 평가
    test_gen.reset()
    preds  = pipeline.predict(test_gen,
                               steps=len(test_df)//BATCH_SIZE + 1, verbose=0)
    y_pred = np.argmax(preds[:len(test_df)], axis=1)
    y_true = test_gen.classes[:len(y_pred)]
    acc    = float(np.mean(y_true == y_pred))
    cm     = confusion_matrix(y_true, y_pred)

    per_class = {}
    for i, cn in enumerate(class_names):
        per_class[cn] = float(cm[i,i] / cm[i].sum()) if cm[i].sum() > 0 else 0.0

    results[pool_size] = {
        "acc": acc, "val_acc": max(hist.history["val_accuracy"]),
        "compressed": compressed, "out_size": f"{out_h}×{out_w}",
        "label": label, "history": hist.history,
        "cm": cm.tolist(), "per_class": per_class,
        "epochs_run": len(hist.history["accuracy"])
    }
    print(f"  ✅ Pool {pool_size}×{pool_size}: {acc*100:.2f}%")
    del pipeline

# =====================================================
# 결과 표 (원본 코드 스타일로)
# =====================================================
best_ps  = max(results, key=lambda k: results[k]["acc"])
worst_ps = min(results, key=lambda k: results[k]["acc"])

print("\n\n" + "="*95)
print("📊 풀링 크기별 실험 결과 (상세)")
print("="*95)
print(f"  {'풀링':>5}  {'출력크기':>8}  {'전송크기':>10}  {'압축률':>7}  {'전체정확도':>10}  {'양성Benign':>10}  {'악성Malig':>10}  {'정상Normal':>10}  {'평가':>6}")
print("-"*95)
for ps in POOL_SIZES:
    r     = results[ps]
    lbl   = f"{ps}×{ps}" if ps < feat_h else "GAP"
    ratio = r["compressed"] / (IMG_SIZE[0]*IMG_SIZE[1]*3) * 100
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    grade = "🟢 우수" if r["acc"]>=0.85 else ("🟡 양호" if r["acc"]>=0.70 else ("🟠 미흡" if r["acc"]>=0.55 else "🔴 불량"))
    mark  = " 🏆" if ps==best_ps else (" 📉" if ps==worst_ps else "")
    print(f"  {lbl:>5}  {r['out_size']:>8}  {r['compressed']:>9,}개  {ratio:>6.1f}%  {r['acc']*100:>9.2f}%  {b:>9.1f}%  {m:>9.1f}%  {n:>9.1f}%  {grade}{mark}")
print("="*95)

print("\n📋 클래스별 상세 분석")
print("-"*95)
for ps in POOL_SIZES:
    r   = results[ps]
    lbl = f"{ps}×{ps}" if ps < feat_h else "GAP"
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    weak = []
    if b < 70: weak.append(f"양성↓({b:.0f}%)")
    if m < 70: weak.append(f"악성↓({m:.0f}%)")
    if n < 70: weak.append(f"정상↓({n:.0f}%)")
    note = "⚠️ " + ", ".join(weak) if weak else "✅ 모든 클래스 70%↑"
    print(f"  Pool {lbl:>4}: 전체 {r['acc']*100:.1f}% | "
          f"양성 {b:.1f}% | 악성 {m:.1f}% | 정상 {n:.1f}% | {note}")
print("="*95)

# =====================================================
# 원본 코드 스타일 그래프
# =====================================================
# 사전학습 곡선 (원본과 동일)
tr_acc  = history_pretrain.history["accuracy"]
tr_loss = history_pretrain.history["loss"]
val_acc = history_pretrain.history["val_accuracy"]
val_loss= history_pretrain.history["val_loss"]
Epochs  = [i+1 for i in range(len(tr_acc))]

plt.figure(figsize=(20, 8))
plt.style.use("fivethirtyeight")

plt.subplot(1, 2, 1)
plt.plot(Epochs, tr_loss,  "r", label="Training loss")
plt.plot(Epochs, val_loss, "g", label="Validation loss")
plt.scatter(np.argmin(val_loss)+1, min(val_loss), s=150, c="blue",
            label=f"best epoch= {np.argmin(val_loss)+1}")
plt.title("Training and Validation Loss")
plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(Epochs, tr_acc,  "r", label="Training Accuracy")
plt.plot(Epochs, val_acc, "g", label="Validation Accuracy")
plt.scatter(np.argmax(val_acc)+1, max(val_acc), s=150, c="blue",
            label=f"best epoch= {np.argmax(val_acc)+1}")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epochs"); plt.ylabel("Accuracy"); plt.legend()

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/pretrain_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 사전학습 그래프 저장")

# Confusion Matrix (원본 스타일)
test_gen.reset()
preds_final = pretrain_model.predict(test_gen,
                                      steps=len(test_df)//BATCH_SIZE + 1, verbose=0)
y_pred_final = np.argmax(preds_final[:len(test_df)], axis=1)
cm_final = confusion_matrix(test_gen.classes[:len(y_pred_final)], y_pred_final)

plt.figure(figsize=(10, 10))
plt.imshow(cm_final, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix - 사전학습 모델")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)
thresh = cm_final.max() / 2.
for i, j in itertools.product(range(cm_final.shape[0]), range(cm_final.shape[1])):
    plt.text(j, i, cm_final[i,j], horizontalalignment="center",
             color="white" if cm_final[i,j] > thresh else "black")
plt.tight_layout()
plt.ylabel("True Label"); plt.xlabel("Predicted Label")
plt.savefig(f"{SAVE_DIR}/pretrain_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(test_gen.classes[:len(y_pred_final)],
                              y_pred_final, target_names=class_names))

# =====================================================
# 한글 폰트 설정
# =====================================================
import subprocess
import matplotlib
import matplotlib.font_manager as fm
import glob

subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], capture_output=True)
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

cache_dir = matplotlib.get_cachedir()
for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
    try: os.remove(f)
    except: pass

fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [font_name]
matplotlib.rcParams["axes.unicode_minus"] = False
print(f"✅ 한글 폰트 적용: {font_name}")

# =====================================================
# 그래프 공통 설정
# =====================================================
BG   = "#0d1117"
CELL = "#161b22"
GRID = "#333333"
TICK = "#aaaaaa"

def style_ax(ax):
    ax.set_facecolor(CELL)
    for sp in ax.spines.values(): sp.set_color("#444444")
    ax.tick_params(colors=TICK, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)

pool_labels = [f"{ps}×{ps}" if ps < feat_h else "GAP" for ps in POOL_SIZES]
accs        = [results[ps]["acc"]*100     for ps in POOL_SIZES]
val_accs    = [results[ps]["val_acc"]*100 for ps in POOL_SIZES]
compressed  = [results[ps]["compressed"]  for ps in POOL_SIZES]
epochs_run  = [results[ps]["epochs_run"]  for ps in POOL_SIZES]
b_accs      = [results[ps]["per_class"].get("Bengin cases",    0)*100 for ps in POOL_SIZES]
m_accs      = [results[ps]["per_class"].get("Malignant cases", 0)*100 for ps in POOL_SIZES]
n_accs      = [results[ps]["per_class"].get("Normal cases",    0)*100 for ps in POOL_SIZES]
security    = [100-(c/(IMG_SIZE[0]*IMG_SIZE[1]*3)*100) for c in compressed]
x_pos       = np.arange(len(POOL_SIZES))
cmap_rg     = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(POOL_SIZES)))

# ─────────────────────────────────────────────────────
# 그래프 1: 전체 정확도 막대 (테스트 vs 검증)
# ─────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(14, 7))
fig1.patch.set_facecolor(BG)
w = 0.35
bars1 = ax1.bar(x_pos - w/2, accs,     width=w, color=cmap_rg, edgecolor="#222", zorder=3, label="테스트 정확도")
bars2 = ax1.bar(x_pos + w/2, val_accs, width=w, color=plt.cm.Blues(np.linspace(0.4, 0.9, len(POOL_SIZES))),
                edgecolor="#222", zorder=3, label="최고 검증 정확도")
ax1.axhline(max(accs), color="#ff6b6b", ls="--", lw=1.5, alpha=0.8, label=f"최고 테스트 {max(accs):.1f}%")
ax1.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
for bar, acc in zip(bars1, accs):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
for bar, acc in zip(bars2, val_accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{acc:.1f}%", ha="center", va="bottom", color="#74b9ff", fontsize=8)
ax1.set_xticks(x_pos); ax1.set_xticklabels(pool_labels, fontsize=10)
ax1.set_ylim(0, max(max(accs), max(val_accs))+20)
ax1.set_title("풀링 크기별 테스트 & 검증 정확도\n(작은 풀링 = 보안 강함 / 큰 풀링 = 정보 많음)",
              color="white", fontsize=13, fontweight="bold")
ax1.set_ylabel("정확도 (%)", color=TICK); ax1.set_xlabel("풀링 크기", color=TICK)
ax1.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax1)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph1_accuracy_bar.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프1 저장: 전체 정확도 막대")

# ─────────────────────────────────────────────────────
# 그래프 2: 클래스별 정확도 그룹 막대
# ─────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(16, 7))
fig2.patch.set_facecolor(BG)
w = 0.2
ax2.bar(x_pos-w*1.5, b_accs, width=w, label="양성 (Benign)",    color="#ff6b6b", edgecolor="#222", zorder=3)
ax2.bar(x_pos-w*0.5, m_accs, width=w, label="악성 (Malignant)", color="#ffd93d", edgecolor="#222", zorder=3)
ax2.bar(x_pos+w*0.5, n_accs, width=w, label="정상 (Normal)",    color="#6bcb77", edgecolor="#222", zorder=3)
ax2.bar(x_pos+w*1.5, accs,   width=w, label="전체 (Overall)",   color="#74b9ff", edgecolor="#222", zorder=3, alpha=0.85)
ax2.axhline(80, color="white", ls="--", lw=1, alpha=0.5, label="80% 기준선")
for i, ps in enumerate(POOL_SIZES):
    b = results[ps]["per_class"].get("Bengin cases",    0)*100
    m = results[ps]["per_class"].get("Malignant cases", 0)*100
    n = results[ps]["per_class"].get("Normal cases",    0)*100
    a = results[ps]["acc"]*100
    for xoff, val, col in [(-w*1.5,b,"#ff6b6b"),(-w*0.5,m,"#ffd93d"),(w*0.5,n,"#6bcb77"),(w*1.5,a,"#74b9ff")]:
        ax2.text(i+xoff, val+0.5, f"{val:.0f}%", ha="center", va="bottom", color=col, fontsize=7, fontweight="bold")
ax2.set_xticks(x_pos); ax2.set_xticklabels(pool_labels, fontsize=10); ax2.set_ylim(0, 120)
ax2.set_title("풀링 크기별 클래스별 + 전체 정확도\n(클래스별로 풀링에 따른 성능 변화 비교)",
              color="white", fontsize=13, fontweight="bold")
ax2.set_ylabel("정확도 (%)", color=TICK)
ax2.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax2)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph2_class_accuracy.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프2 저장: 클래스별 정확도")

# ─────────────────────────────────────────────────────
# 그래프 3: 정확도 추세선 (전체 + 클래스별)
# ─────────────────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(14, 7))
fig3.patch.set_facecolor(BG)
ax3.plot(pool_labels, accs,   "o-",  color="#74b9ff", lw=3,   ms=10, mfc="white", mec="#74b9ff", label="전체 정확도", zorder=4)
ax3.plot(pool_labels, b_accs, "s--", color="#ff6b6b", lw=1.8, ms=7,  mfc="white", label="양성 (Benign)")
ax3.plot(pool_labels, m_accs, "^--", color="#ffd93d", lw=1.8, ms=7,  mfc="white", label="악성 (Malignant)")
ax3.plot(pool_labels, n_accs, "D--", color="#6bcb77", lw=1.8, ms=7,  mfc="white", label="정상 (Normal)")
ax3.fill_between(range(len(pool_labels)), accs, alpha=0.07, color="#74b9ff")
ax3.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
for i, (a,b,m,n) in enumerate(zip(accs, b_accs, m_accs, n_accs)):
    ax3.annotate(f"{a:.1f}%", (i, a), textcoords="offset points",
                 xytext=(0,10), ha="center", color="#74b9ff", fontsize=9, fontweight="bold")
ax3.set_ylim(0, max(accs)+20)
ax3.set_title("풀링 크기별 정확도 추세\n(전체 + 클래스별 변화 비교)",
              color="white", fontsize=13, fontweight="bold")
ax3.set_ylabel("정확도 (%)", color=TICK); ax3.set_xlabel("풀링 크기 (왼쪽=보안강함)", color=TICK)
ax3.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph3_trend.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프3 저장: 정확도 추세선")

# ─────────────────────────────────────────────────────
# 그래프 4: 보안-성능 트레이드오프 산점도
# ─────────────────────────────────────────────────────
fig4, ax4 = plt.subplots(figsize=(10, 8))
fig4.patch.set_facecolor(BG)
sc = ax4.scatter(security, accs, c=accs, cmap="RdYlGn", s=300,
                 edgecolors="white", lw=1.5, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(pool_labels):
    ax4.annotate(lbl, (security[i], accs[i]), textcoords="offset points",
                 xytext=(10, 5), color="white", fontsize=10, fontweight="bold")
ax4.axvspan(0,  20, alpha=0.05, color="#6bcb77", label="고성능 구역")
ax4.axvspan(80, 100, alpha=0.05, color="#ff6b6b", label="고보안 구역")
cbar = plt.colorbar(sc, ax=ax4)
cbar.set_label("AI 정확도 (%)", color="white", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax4.set_xlabel("보안 강도 (%) →\n(높을수록 더 많이 압축, 원본 정보 적음)", color=TICK, fontsize=11)
ax4.set_ylabel("AI 정확도 (%)", color=TICK, fontsize=11)
ax4.set_title("보안성 vs AI 성능 트레이드오프\n이상적 위치: 왼쪽 위 (보안 낮음 + 정확도 높음)", color="white", fontsize=13, fontweight="bold")
ax4.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax4)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph4_tradeoff.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프4 저장: 보안-성능 트레이드오프")

# ─────────────────────────────────────────────────────
# 그래프 5: 전송 데이터 크기 vs 정확도 이중축
# ─────────────────────────────────────────────────────
fig5, ax5a = plt.subplots(figsize=(14, 6))
fig5.patch.set_facecolor(BG)
ax5b = ax5a.twinx()
ax5a.bar(x_pos, accs, color=cmap_rg, edgecolor="#222", zorder=3, width=0.6, alpha=0.85)
ax5b.plot(x_pos, compressed, "D-", color="#74b9ff", lw=2.5, ms=10, mfc="white", zorder=4, label="전송 데이터 크기")
ax5b.set_yscale("log")
for i, (a, c) in enumerate(zip(accs, compressed)):
    ax5a.text(i, a+0.5, f"{a:.1f}%", ha="center", color="white", fontsize=9, fontweight="bold")
    ax5b.text(i, c*1.2, f"{c:,}", ha="center", color="#74b9ff", fontsize=7)
ax5a.set_xticks(x_pos); ax5a.set_xticklabels(pool_labels); ax5a.set_ylim(0, max(accs)+18)
ax5a.set_title("전송 데이터 크기 vs AI 정확도\n(풀링이 클수록 전송 데이터 많음 → 정확도↑, 보안↓)",
               color="white", fontsize=13, fontweight="bold")
ax5a.set_ylabel("정확도 (%)", color="#6bcb77", fontsize=12)
ax5b.set_ylabel("전송 데이터 크기 (log scale, 개)", color="#74b9ff", fontsize=12)
ax5a.tick_params(axis="y", colors="#6bcb77"); ax5b.tick_params(axis="y", colors="#74b9ff")
ax5a.tick_params(axis="x", colors=TICK)
for sp in ax5a.spines.values(): sp.set_color("#444")
ax5a.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); ax5a.set_axisbelow(True)
ax5b.legend(facecolor="#222", labelcolor="white")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph5_datasize_vs_acc.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프5 저장: 전송 크기 vs 정확도")

# ─────────────────────────────────────────────────────
# 그래프 6: 클래스별 정확도 히트맵
# ─────────────────────────────────────────────────────
fig6, ax6 = plt.subplots(figsize=(14, 5))
fig6.patch.set_facecolor(BG)
hm = np.array([b_accs, m_accs, n_accs, accs])
im = ax6.imshow(hm, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
ax6.set_xticks(range(len(pool_labels))); ax6.set_xticklabels(pool_labels, color="white", fontsize=11)
ax6.set_yticks(range(4))
ax6.set_yticklabels(["양성\n(Benign)", "악성\n(Malignant)", "정상\n(Normal)", "전체\n(Overall)"],
                     color="white", fontsize=11)
for i in range(4):
    for j in range(len(POOL_SIZES)):
        v = hm[i, j]
        ax6.text(j, i, f"{v:.1f}%", ha="center", va="center",
                 color="black" if v > 60 else "white", fontsize=10, fontweight="bold")
cbar6 = plt.colorbar(im, ax=ax6)
cbar6.set_label("정확도 (%)", color="white")
cbar6.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar6.ax.yaxis.get_ticklabels(), color="white")
ax6.set_title("클래스별 + 전체 정확도 히트맵\n초록=높음 / 빨강=낮음",
              color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph6_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프6 저장: 히트맵")

# ─────────────────────────────────────────────────────
# 그래프 7: Confusion Matrix (최고 + 최저 풀링)
# ─────────────────────────────────────────────────────
fig7, axes7 = plt.subplots(1, 2, figsize=(16, 7))
fig7.patch.set_facecolor(BG)
for ax_cm, ps, ttl in zip(axes7, [best_ps, worst_ps], ["🏆 최고 성능", "📉 최저 성능"]):
    ax_cm.set_facecolor(CELL)
    cm_arr = np.array(results[ps]["cm"])
    lbl    = f"{ps}×{ps}" if ps < feat_h else "GAP"
    im7    = ax_cm.imshow(cm_arr, interpolation="nearest", cmap="Blues")
    plt.colorbar(im7, ax=ax_cm)
    tick_marks = np.arange(len(class_names))
    short_names = [c.replace(" cases","").replace("Bengin","양성").replace("Malignant","악성").replace("Normal","정상")
                   for c in class_names]
    ax_cm.set_xticks(tick_marks); ax_cm.set_xticklabels(short_names, color="white", fontsize=10, rotation=15)
    ax_cm.set_yticks(tick_marks); ax_cm.set_yticklabels(short_names, color="white", fontsize=10)
    thresh = cm_arr.max() / 2.
    for i, j in itertools.product(range(cm_arr.shape[0]), range(cm_arr.shape[1])):
        ax_cm.text(j, i, str(cm_arr[i,j]), ha="center", va="center",
                   color="white" if cm_arr[i,j] < thresh else "black", fontsize=14, fontweight="bold")
    ax_cm.set_title(f"{ttl}\nPool {lbl} | 정확도: {results[ps]['acc']*100:.1f}%",
                    color="white", fontsize=12, fontweight="bold")
    ax_cm.set_xlabel("AI 예측 (Predicted)", color=TICK, fontsize=11)
    ax_cm.set_ylabel("실제 정답 (Actual)",  color=TICK, fontsize=11)
fig7.suptitle("Confusion Matrix: 최고 풀링 vs 최저 풀링 비교\n대각선=정답 / 나머지=오답",
              color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph7_confusion.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프7 저장: Confusion Matrix")

# ─────────────────────────────────────────────────────
# 그래프 8: 레이더 차트 (주요 풀링 비교)
# ─────────────────────────────────────────────────────
key_ps = [POOL_SIZES[0], POOL_SIZES[len(POOL_SIZES)//2], POOL_SIZES[-2], POOL_SIZES[-1]]
key_ps = sorted(set(key_ps))
categories = ["전체 정확도", "양성 정확도", "악성 정확도", "정상 정확도", "보안 강도"]
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
fig8, ax8 = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig8.patch.set_facecolor(BG); ax8.set_facecolor(CELL)
colors_radar = ["#f9ca24", "#6bcb77", "#ff6b6b", "#7c83fd"]
for idx, ps in enumerate(key_ps):
    r = results[ps]
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    sec = 100 - (r["compressed"] / (IMG_SIZE[0]*IMG_SIZE[1]*3) * 100)
    vals = [r["acc"]*100, b, m, n, sec]
    vals += vals[:1]
    lbl = f"Pool {ps}×{ps}" if ps < feat_h else "Pool GAP"
    ax8.plot(angles, vals, "o-", lw=2, color=colors_radar[idx % 4], label=lbl)
    ax8.fill(angles, vals, alpha=0.05, color=colors_radar[idx % 4])
ax8.set_xticks(angles[:-1])
ax8.set_xticklabels(categories, color="white", fontsize=11)
ax8.set_ylim(0, 100); ax8.set_yticks([20, 40, 60, 80, 100])
ax8.set_yticklabels(["20%","40%","60%","80%","100%"], color=TICK, fontsize=8)
ax8.grid(color="#444", linestyle="--", alpha=0.6)
ax8.spines["polar"].set_color("#444")
ax8.set_title("주요 풀링 크기별 성능 레이더 차트\n(각 축: 정확도/보안 강도)",
              color="white", fontsize=13, fontweight="bold", pad=25)
ax8.legend(facecolor="#222", labelcolor="white", fontsize=11,
           loc="upper right", bbox_to_anchor=(1.4, 1.1))
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph8_radar.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프8 저장: 레이더 차트")

# ─────────────────────────────────────────────────────
# 그래프 9: 풀링별 학습 곡선 전체
# ─────────────────────────────────────────────────────
fig9, axes9 = plt.subplots(2, 4, figsize=(24, 11))
fig9.patch.set_facecolor(BG)
axes9 = axes9.flatten()
for i, ps in enumerate(POOL_SIZES):
    ax = axes9[i]; ax.set_facecolor(CELL)
    h  = results[ps]["history"]
    ep = range(1, len(h["accuracy"])+1)
    ax.plot(ep, [v*100 for v in h["accuracy"]],     color="#6bcb77", lw=2, label="학습 정확도")
    ax.plot(ep, [v*100 for v in h["val_accuracy"]], color="#ffd93d", lw=2, label="검증 정확도")
    ax.plot(ep, [v*100 for v in h["loss"]],          color="#ff6b6b", lw=1.5, ls="--", label="학습 손실", alpha=0.7)
    ax.plot(ep, [v*100 for v in h["val_loss"]],      color="#74b9ff", lw=1.5, ls="--", label="검증 손실", alpha=0.7)
    best_val = results[ps]["val_acc"]*100
    best_ep  = np.argmax(h["val_accuracy"])+1
    ax.axvline(best_ep, color="white", ls=":", lw=1, alpha=0.5)
    lbl = f"{ps}×{ps}" if ps < feat_h else "GAP"
    b = results[ps]["per_class"].get("Bengin cases",    0)*100
    m_v = results[ps]["per_class"].get("Malignant cases", 0)*100
    n_v = results[ps]["per_class"].get("Normal cases",    0)*100
    ax.set_title(f"Pool {lbl}  |  테스트: {results[ps]['acc']*100:.1f}%\n"
                 f"양성:{b:.0f}% 악성:{m_v:.0f}% 정상:{n_v:.0f}% ({results[ps]['epochs_run']}ep)",
                 color="white", fontsize=9, fontweight="bold")
    ax.set_xlabel("Epoch", color=TICK, fontsize=8)
    ax.set_ylabel("값 (%)", color=TICK, fontsize=8)
    ax.set_ylim(0, 105)
    ax.legend(facecolor="#222", labelcolor="white", fontsize=7)
    for sp in ax.spines.values(): sp.set_color("#444")
    ax.tick_params(colors=TICK)
    ax.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)
for j in range(len(POOL_SIZES), len(axes9)): axes9[j].set_visible(False)
fig9.suptitle("풀링 크기별 학습 곡선 전체\n(정확도 + 손실, 클래스별 결과 포함)",
              color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph9_learning_curves.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프9 저장: 학습 곡선")

# ─────────────────────────────────────────────────────
# 그래프 10: 종합 요약 대시보드
# ─────────────────────────────────────────────────────
from matplotlib.gridspec import GridSpec
fig10 = plt.figure(figsize=(22, 14))
fig10.patch.set_facecolor(BG)
gs10 = GridSpec(3, 3, figure=fig10, hspace=0.5, wspace=0.35)

# [0,0:2] 전체 정확도 막대
a1 = fig10.add_subplot(gs10[0, :2]); a1.set_facecolor(CELL)
b10 = a1.bar(x_pos, accs, color=cmap_rg, edgecolor="#333", zorder=3)
for bar, acc in zip(b10, accs):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    a1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=10, fontweight="bold")
a1.set_xticks(x_pos); a1.set_xticklabels(pool_labels)
a1.set_ylim(0, max(accs)+18)
a1.set_title("풀링 크기별 테스트 정확도", color="white", fontsize=12, fontweight="bold")
a1.set_ylabel("정확도 (%)", color=TICK)
for sp in a1.spines.values(): sp.set_color("#444")
a1.tick_params(colors=TICK); a1.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a1.set_axisbelow(True)

# [0,2] 요약 텍스트
a2 = fig10.add_subplot(gs10[0, 2]); a2.set_facecolor(CELL); a2.axis("off")
lbl_b2 = f"{best_ps}×{best_ps}" if best_ps<feat_h else "GAP"
lbl_w2 = f"{worst_ps}×{worst_ps}" if worst_ps<feat_h else "GAP"
b_best = results[best_ps]["per_class"].get("Bengin cases",    0)*100
m_best = results[best_ps]["per_class"].get("Malignant cases", 0)*100
n_best = results[best_ps]["per_class"].get("Normal cases",    0)*100
summary_txt = (
    f"📊 실험 요약\n\n"
    f"풀링: {POOL_SIZES}\n"
    f"오버샘플링: {TARGET_COUNT}장/클래스\n\n"
    f"🏆 최고: Pool {lbl_b2}\n"
    f"  전체: {results[best_ps]['acc']*100:.1f}%\n"
    f"  양성: {b_best:.1f}%\n"
    f"  악성: {m_best:.1f}%\n"
    f"  정상: {n_best:.1f}%\n\n"
    f"📉 최저: Pool {lbl_w2}\n"
    f"  전체: {results[worst_ps]['acc']*100:.1f}%\n\n"
    f"🎯 결론\n"
    f"  풀링↑ → 정확도↑\n"
    f"  풀링↓ → 보안↑"
)
a2.text(0.05, 0.97, summary_txt, transform=a2.transAxes,
        color="white", fontsize=9.5, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#21262d", edgecolor="#555"))
a2.set_title("실험 결론", color="white", fontsize=12, fontweight="bold")

# [1,0] 클래스별 히트맵
a3 = fig10.add_subplot(gs10[1, :2]); a3.set_facecolor(CELL)
hm2 = np.array([b_accs, m_accs, n_accs, accs])
im3 = a3.imshow(hm2, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
a3.set_xticks(range(len(pool_labels))); a3.set_xticklabels(pool_labels, color="white", fontsize=10)
a3.set_yticks(range(4))
a3.set_yticklabels(["양성", "악성", "정상", "전체"], color="white", fontsize=10)
for i in range(4):
    for j in range(len(POOL_SIZES)):
        v = hm2[i, j]
        a3.text(j, i, f"{v:.0f}%", ha="center", va="center",
                color="black" if v>60 else "white", fontsize=10, fontweight="bold")
a3.set_title("클래스별 + 전체 정확도 히트맵", color="white", fontsize=12, fontweight="bold")
plt.colorbar(im3, ax=a3).ax.yaxis.set_tick_params(color="white")

# [1,2] 보안 vs 성능
a4 = fig10.add_subplot(gs10[1, 2]); a4.set_facecolor(CELL)
sc4 = a4.scatter(security, accs, c=accs, cmap="RdYlGn", s=150,
                 edgecolors="white", lw=1, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(pool_labels):
    a4.annotate(lbl, (security[i], accs[i]), textcoords="offset points",
                xytext=(6, 4), color="white", fontsize=8)
a4.set_xlabel("보안 강도 →", color=TICK); a4.set_ylabel("AI 정확도 (%)", color=TICK)
a4.set_title("보안 vs 성능", color="white", fontsize=12, fontweight="bold")
for sp in a4.spines.values(): sp.set_color("#444")
a4.tick_params(colors=TICK); a4.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a4.set_axisbelow(True)

# [2,0] 학습 곡선 (최고 풀링)
a5 = fig10.add_subplot(gs10[2, 0]); a5.set_facecolor(CELL)
h_b  = results[best_ps]["history"]
ep_b = range(1, len(h_b["accuracy"])+1)
a5.plot(ep_b, [v*100 for v in h_b["accuracy"]],     color="#6bcb77", lw=2, label="학습 정확도")
a5.plot(ep_b, [v*100 for v in h_b["val_accuracy"]], color="#ffd93d", lw=2, label="검증 정확도")
a5.set_title(f"최고 Pool {lbl_b2} 학습 곡선", color="white", fontsize=11, fontweight="bold")
a5.set_xlabel("Epoch", color=TICK); a5.set_ylabel("정확도 (%)", color=TICK); a5.set_ylim(0, 105)
a5.legend(facecolor="#222", labelcolor="white", fontsize=9)
for sp in a5.spines.values(): sp.set_color("#444")
a5.tick_params(colors=TICK); a5.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a5.set_axisbelow(True)

# [2,1] 전송 크기 막대
a6 = fig10.add_subplot(gs10[2, 1]); a6.set_facecolor(CELL)
a6.bar(x_pos, compressed, color=cmap_rg, edgecolor="#222", zorder=3)
a6.set_yscale("log")
a6.set_xticks(x_pos); a6.set_xticklabels(pool_labels)
a6.set_title("전송 데이터 크기 (log)", color="white", fontsize=11, fontweight="bold")
a6.set_ylabel("전송 크기 (log)", color=TICK)
for sp in a6.spines.values(): sp.set_color("#444")
a6.tick_params(colors=TICK); a6.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a6.set_axisbelow(True)

# [2,2] epoch 수
a7 = fig10.add_subplot(gs10[2, 2]); a7.set_facecolor(CELL)
a7b = a7.twinx()
a7.bar(x_pos, epochs_run, color=cmap_rg, edgecolor="#222", zorder=3, alpha=0.85)
a7b.plot(x_pos, accs, "o-", color="#f9ca24", lw=2, ms=8, mfc="white", zorder=4)
for i, ep in enumerate(epochs_run):
    a7.text(i, ep+0.3, f"{ep}ep", ha="center", color="white", fontsize=8)
a7.set_xticks(x_pos); a7.set_xticklabels(pool_labels)
a7.set_ylim(0, EPOCHS_S2+10)
a7.set_title("학습 epoch 수 & 정확도", color="white", fontsize=11, fontweight="bold")
a7.set_ylabel("epoch 수", color="#6bcb77", fontsize=10)
a7b.set_ylabel("정확도 (%)", color="#f9ca24", fontsize=10)
a7.tick_params(axis="y", colors="#6bcb77"); a7b.tick_params(axis="y", colors="#f9ca24")
a7.tick_params(axis="x", colors=TICK)
for sp in a7.spines.values(): sp.set_color("#444")
a7.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a7.set_axisbelow(True)

fig10.suptitle(f"Edge-Server Split CNN 종합 대시보드\n"
               f"오버샘플링 {TARGET_COUNT}장/클래스 | MobileNetV2 | Adamax",
               color="white", fontsize=14, fontweight="bold", y=1.01)
plt.savefig(f"{SAVE_DIR}/graph10_dashboard.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프10 저장: 종합 대시보드")

print(f"""
╔══════════════════════════════════════════╗
║  ✅ 그래프 10개 저장 완료! → {SAVE_DIR}/
╠══════════════════════════════════════════╣
║  graph1  - 전체 정확도 막대 (테스트+검증)
║  graph2  - 클래스별 + 전체 정확도
║  graph3  - 정확도 추세선
║  graph4  - 보안-성능 트레이드오프
║  graph5  - 전송 크기 vs 정확도
║  graph6  - 클래스별 히트맵
║  graph7  - Confusion Matrix
║  graph8  - 레이더 차트
║  graph9  - 학습 곡선 전체
║  graph10 - 종합 대시보드
╚══════════════════════════════════════════╝

🏆 최고: Pool {best_ps}×{best_ps} → {results[best_ps]["acc"]*100:.2f}%
📉 최저: Pool {worst_ps}×{worst_ps} → {results[worst_ps]["acc"]*100:.2f}%
""")


In [ ]:
 # =============================================================
# 엣지-서버 분리 풀링 실험 코드
# 원본 코드 기반 + 풀링 실험 추가
# =============================================================

# ── 시스템 라이브러리 ──────────────────────────────────────
import os          # 파일/폴더 경로 처리
import shutil      # 파일 복사/삭제
import itertools   # Confusion Matrix 그릴 때 (원본 코드 그대로)
import random      # 오버샘플링 랜덤 선택
import warnings    # 경고 메시지 무시
warnings.filterwarnings("ignore")

# ── 데이터 처리 라이브러리 ────────────────────────────────
import numpy as np                              # 행렬/배열 계산
import pandas as pd                             # 데이터프레임 (원본 코드 그대로)
import matplotlib.pyplot as plt                 # 그래프 그리기
import matplotlib                               # 폰트 설정용
import matplotlib.font_manager as fm            # 한글 폰트 적용
import glob                                     # 파일 경로 패턴 검색
import subprocess                               # 폰트 설치 명령 실행

# ── 머신러닝 라이브러리 ───────────────────────────────────
from PIL import Image, ImageEnhance             # 이미지 열기/변환/밝기 조절
from sklearn.model_selection import train_test_split  # 데이터 분할 (원본 코드 그대로)
from sklearn.metrics import confusion_matrix, classification_report  # 성능 평가 (원본 그대로)

# ── 딥러닝 라이브러리 (TensorFlow/Keras) ─────────────────
import tensorflow as tf
from tensorflow.keras.models import Model                    # 레이어 연결 모델 (중간 출력 추출용)
from tensorflow.keras.optimizers import Adamax               # 원본 코드와 동일한 optimizer
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # 원본 코드 그대로
from tensorflow.keras.layers import (
    Dense,                  # 완전연결 레이어 (분류기)
    GlobalAveragePooling2D, # Conv 출력 전체 평균 → 1개 숫자로 압축
    MaxPooling2D,           # 특징맵 일부 압축 (풀링 실험 핵심!)
    Flatten,                # 2D → 1D로 펼치기
    Dropout                 # 과적합 방지 (원본 코드에도 있음)
)
from tensorflow.keras.applications import MobileNetV2        # 사전학습 CNN 모델
from tensorflow.keras.callbacks import EarlyStopping         # 과적합 방지 조기종료

print("modules loaded")

# =============================================================
# ★★ 한글 폰트 설정 (그래프 한글 깨짐 방지) ★★
# =============================================================
subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], capture_output=True)
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# matplotlib 캐시 삭제 (이전 폰트 설정 초기화)
cache_dir = matplotlib.get_cachedir()
for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
    try: os.remove(f)
    except: pass

# 나눔고딕 폰트 등록 및 적용
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams["font.family"]      = "sans-serif"
matplotlib.rcParams["font.sans-serif"]  = [font_name]
matplotlib.rcParams["axes.unicode_minus"] = False  # 마이너스(-) 기호 깨짐 방지
print(f"✅ 한글 폰트 적용: {font_name}")

# =============================================================
# 실험 설정값
# =============================================================
BASE_DIR     = "/workspace/Lung Cancer/Train"  # 원본 폐암 데이터 경로
TARGET_COUNT = 1000    # 오버샘플링 목표 (클래스당 1000장)
IMG_SIZE     = (224, 224)  # 입력 이미지 크기 (MobileNetV2 기본값)
BATCH_SIZE   = 64      # 한 번에 학습할 이미지 수 (원본 코드와 동일)
EPOCHS       = 50      # 1단계 헤드 학습 최대 에폭
EPOCHS_FT    = 20      # 2단계 fine-tuning 최대 에폭
EPOCHS_S2    = 50      # 풀링 분류기 학습 최대 에폭

# 실험할 풀링 크기 목록 (1×1 ~ 7×7)
# MobileNetV2의 특징맵이 7×7 이므로 최대 7까지만 가능
POOL_SIZES   = [1, 2, 3, 4, 5, 6, 7]

SAVE_DIR     = "/workspace/viz"   # 그래프 저장 폴더
os.makedirs(SAVE_DIR, exist_ok=True)

# 그래프 다크 테마 색상
BG   = "#0d1117"   # 배경색 (거의 검정)
CELL = "#161b22"   # 그래프 영역 색
GRID = "#333333"   # 격자선 색
TICK = "#aaaaaa"   # 눈금/라벨 색

def style_ax(ax):
    """다크 테마 스타일 적용 함수"""
    ax.set_facecolor(CELL)
    for sp in ax.spines.values():
        sp.set_color("#444444")   # 테두리 색
    ax.tick_params(colors=TICK, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)  # 격자선이 막대 뒤에 오도록

# =============================================================
# 오버샘플링 함수
# ★ 원본 데이터를 건드리지 않고 별도 폴더에 복사 후 증강
# =============================================================
def oversample_dir(base_dir, target_count, seed=42):
    """
    [역할] 클래스 불균형 해소
    Benign 120장 / Malignant 561장 / Normal 416장 → 모두 1000장으로 균일화
    원본 폴더 보호: base_dir_oversampled/ 폴더에 따로 저장
    """
    out_dir = base_dir.rstrip("/") + "_oversampled"  # 출력 폴더 이름
    # 클래스 폴더 목록 (알파벳 순 정렬)
    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, d))])

    # 이미 오버샘플링 완료된 경우 재사용
    if os.path.exists(out_dir):
        all_done = all(
            os.path.exists(os.path.join(out_dir, c)) and
            len([f for f in os.listdir(os.path.join(out_dir, c))
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]) >= target_count
            for c in classes
        )
        if all_done:
            total = sum(len([f for f in os.listdir(os.path.join(out_dir, c))
                             if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                        for c in classes)
            print(f"  오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    random.seed(seed)  # 재현 가능한 랜덤

    # Step 1: 원본 파일을 출력 폴더로 복사 (aug_ 접두사 파일 제외)
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]  # 이미 증강된 파일 제외
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir, cls, f), dst)

    # Step 2: 부족한 클래스를 증강(회전/뒤집기/밝기조절)으로 채우기
    for cls in classes:
        cls_out   = os.path.join(out_dir, cls)
        # 증강 소스: 항상 순수 원본 파일만 선택
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]
        # 현재 출력 폴더에 있는 파일 수
        cur_files = [f for f in os.listdir(cls_out)
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
        needed = target_count - len(cur_files)  # 추가로 필요한 장 수
        if needed <= 0:
            print(f"  {cls}: {len(cur_files)}장 (충분)"); continue
        print(f"  {cls}: {len(cur_files)}장 → {target_count}장 ({needed}장 증강)")
        generated = 0
        while generated < needed:
            src_f = random.choice(src_files)  # 원본 중 랜덤 선택
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(random.uniform(-30, 30), expand=False)  # 최대 ±30도 회전
                if random.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)   # 50% 확률 좌우반전
                if random.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)   # 50% 확률 상하반전
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))   # 밝기 0.7~1.3배 조절
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"), "JPEG", quality=95)
                generated += 1
            except: continue

    total = sum(len([f for f in os.listdir(os.path.join(out_dir, c))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                for c in classes)
    print(f"  ✅ 오버샘플링 완료: 총 {total}장 → {out_dir}")
    return out_dir

# =============================================================
# 오버샘플링 실행
# =============================================================
print("\n오버샘플링 시작...")
train_dir = oversample_dir(BASE_DIR, TARGET_COUNT)  # 오버샘플링 결과 폴더 경로

# =============================================================
# 원본 코드 방식: 파일 경로 + 라벨 → DataFrame 생성
# =============================================================
filepaths = []  # 이미지 파일 경로 목록
labels    = []  # 각 이미지의 클래스 이름 목록

# 오버샘플링된 폴더의 클래스 목록
classes_list = sorted([d for d in os.listdir(train_dir)
                        if os.path.isdir(os.path.join(train_dir, d))])

for cls in classes_list:
    cls_dir = os.path.join(train_dir, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
            filepaths.append(os.path.join(cls_dir, f))  # 전체 경로
            labels.append(cls)                           # 클래스 이름

# 원본 코드와 동일: Series → DataFrame
df = pd.concat([pd.Series(filepaths, name="filepaths"),
                pd.Series(labels,    name="labels")], axis=1)

print(f"\n전체 데이터: {len(df)}장")
print(df["labels"].value_counts())  # 클래스별 장 수 확인

# =============================================================
# 원본 코드와 동일: 학습/검증/테스트 분리 (80% / 10% / 10%)
# =============================================================
# 1차 분리: 80% 학습, 20% 임시
train_df, dummy_df = train_test_split(
    df, train_size=0.8, shuffle=True,
    random_state=123,          # 재현 가능한 랜덤 (원본 코드와 동일)
    stratify=df["labels"]      # 클래스 비율 유지하며 분리
)
# 2차 분리: 임시 20%를 검증 10%, 테스트 10%로 나눔
valid_df, test_df = train_test_split(
    dummy_df, train_size=0.5, shuffle=True,
    random_state=123,
    stratify=dummy_df["labels"]
)

print(f"\n학습: {len(train_df)}장 | 검증: {len(valid_df)}장 | 테스트: {len(test_df)}장")

# =============================================================
# 원본 코드 방식: ImageDataGenerator
# 학습용: 증강(회전/확대/뒤집기) 적용
# 검증/테스트용: 증강 없이 정규화만
# =============================================================
tr_gen = ImageDataGenerator(
    rescale=1./255,         # 픽셀값 0~255 → 0~1 정규화
    rotation_range=15,      # 최대 ±15도 회전
    zoom_range=0.1,         # 최대 10% 확대/축소
    horizontal_flip=True    # 좌우 뒤집기
)
ts_gen = ImageDataGenerator(rescale=1./255)  # 테스트/검증: 정규화만

# flow_from_dataframe: DataFrame의 filepath에서 이미지를 읽어 배치 생성
train_gen = tr_gen.flow_from_dataframe(
    train_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE,           # 224×224로 리사이즈
    class_mode="categorical",       # 다중 분류 (원본 코드와 동일)
    color_mode="rgb",               # 컬러 이미지
    shuffle=True,                   # 학습 시 순서 섞기
    batch_size=BATCH_SIZE           # 64장씩 (원본 코드와 동일)
)
valid_gen = ts_gen.flow_from_dataframe(
    valid_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE
)
test_gen = ts_gen.flow_from_dataframe(
    test_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE  # 테스트는 반드시 shuffle=False
)

g_dict      = train_gen.class_indices   # {'Bengin cases': 0, 'Malignant cases': 1, 'Normal cases': 2}
class_names = list(g_dict.keys())       # ['Bengin cases', 'Malignant cases', 'Normal cases']
print(f"\n클래스: {g_dict}")

# =============================================================
# STEP 1: MobileNetV2 사전학습
#
# ★ 왜 원본 VGG(Sequential) 대신 MobileNetV2를 쓰나?
#   → 풀링 실험은 Conv 출력(특징맵)을 중간에서 꺼내야 함
#   → Sequential 모델은 중간 레이어 출력 추출 불가
#   → Functional API(Model)를 써야 중간 추출 가능
#   → MobileNetV2는 이미 ImageNet으로 학습된 강력한 Conv 보유
# =============================================================
print("\n" + "="*55)
print("STEP 1: MobileNetV2 사전학습")
print("="*55)

# MobileNetV2의 Conv 부분만 가져오기 (분류기 헤드 제외)
# include_top=False: 마지막 Dense 레이어 제외
# weights="imagenet": ImageNet 1000만 장으로 사전학습된 가중치 사용
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)  # 224×224×3(RGB)
)
base_model.trainable = False  # Conv 가중치 고정 (헤드만 먼저 학습)

# 분류기 헤드 추가 (원본 코드의 Dense 레이어와 동일한 역할)
x   = GlobalAveragePooling2D()(base_model.output)  # 7×7×1280 → 1280
x   = Dropout(0.3)(x)                              # 30% 뉴런 랜덤 제거 (과적합 방지)
out = Dense(len(class_names), activation="softmax")(x)  # 3개 클래스 확률

# Functional API로 전체 모델 생성
pretrain_model = Model(inputs=base_model.input, outputs=out)

# 원본 코드와 동일: Adamax optimizer, categorical_crossentropy
pretrain_model.compile(
    Adamax(learning_rate=0.001),       # 원본 코드와 동일한 optimizer
    loss="categorical_crossentropy",   # 원본 코드와 동일한 손실함수
    metrics=["accuracy"]
)

# 1단계: Conv 고정, 헤드(Dense)만 학습
print(f"[1단계] 헤드 학습 (최대 {EPOCHS}ep)...")
pretrain_model.fit(
    train_gen,
    epochs=EPOCHS,
    verbose=1,
    validation_data=valid_gen
)

# 2단계: Conv 마지막 30개 레이어 열어서 미세조정(fine-tuning)
# 전체를 열면 과적합 위험 → 마지막 30개만
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False  # 앞부분 레이어는 계속 고정

# fine-tuning은 학습률 낮춰서 (가중치가 너무 많이 변하면 안 됨)
pretrain_model.compile(
    Adamax(learning_rate=0.0001),      # 10배 낮춤
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"\n[2단계] Fine-tuning (최대 {EPOCHS_FT}ep)...")
history_pretrain = pretrain_model.fit(
    train_gen,
    epochs=EPOCHS_FT,
    verbose=1,
    validation_data=valid_gen
)

print(f"\n✅ 사전학습 완료!")
print(f"   최종 학습 정확도:  {history_pretrain.history['accuracy'][-1]*100:.2f}%")
print(f"   최종 검증 정확도:  {history_pretrain.history['val_accuracy'][-1]*100:.2f}%")

# =============================================================
# conv_extractor 생성 (핵심!)
#
# ★ 왜 base_model이 아니라 pretrain_model에서 추출?
#   → base_model.trainable = False 하면 가중치가 분리됨
#   → fine-tuning 결과가 반영 안 될 수 있음
#   → pretrain_model 내부 레이어를 직접 순회해서
#      4D 출력(H×W×C)을 내는 마지막 레이어를 찾아 추출
# =============================================================
conv_output = None
for layer in pretrain_model.layers:
    if hasattr(layer, "output_shape"):
        shape = layer.output_shape
        # 4D = (배치, 높이, 너비, 채널) → Conv 레이어 출력
        if isinstance(shape, tuple) and len(shape) == 4:
            conv_output = layer.output  # 마지막 Conv 레이어 출력

# conv_extractor: 입력 이미지 → Conv 특징맵만 출력
conv_extractor = Model(
    inputs=pretrain_model.input,
    outputs=conv_output,
    name="conv_extractor"
)
conv_extractor.trainable = False  # Conv는 이제 완전히 고정

feat_h, feat_w, feat_c = conv_extractor.output_shape[1:]  # 예: 7, 7, 1280
print(f"\n특징맵 크기: {feat_h}×{feat_w}×{feat_c}")
print(f"(이 {feat_h}×{feat_w}×{feat_c}개 숫자를 풀링으로 압축해서 서버 전송)")

# =============================================================
# STEP 2: 풀링 크기별 실험
# =============================================================
print("\n" + "="*55)
print("STEP 2: 풀링 크기별 실험")
print("="*55)

results = {}  # 각 풀링 크기의 결과를 저장할 딕셔너리

for pool_size in POOL_SIZES:
    print(f"\n--- Pool {pool_size}×{pool_size} ---")

    # ── 풀링 레이어 선택 ──────────────────────────────
    if pool_size >= feat_h:
        # pool_size가 특징맵 크기(7)와 같거나 크면 → 전체 평균 (GlobalAvgPool)
        # 7×7×1280 → 1280개 숫자 (최대 압축)
        x          = GlobalAveragePooling2D()(conv_extractor.output)
        compressed = feat_c              # 전송할 숫자 수: 1280개
        out_h, out_w = 1, 1              # 출력 크기: 1×1
        label = "GlobalAvgPool"
    else:
        # pool_size만큼 격자로 나눠서 각 격자의 최댓값 선택 (MaxPooling)
        # 예) pool_size=2: 7×7 → 3×3 (각 2×2 구역의 최댓값)
        x     = MaxPooling2D(pool_size=(pool_size, pool_size))(conv_extractor.output)
        out_h = feat_h // pool_size      # 출력 높이
        out_w = feat_w // pool_size      # 출력 너비
        x     = Flatten()(x)             # 3D → 1D (Dense 레이어 입력용)
        compressed = out_h * out_w * feat_c  # 전송할 숫자 수
        label = f"MaxPool {pool_size}×{pool_size}"

    print(f"  특징맵 {feat_h}×{feat_w}×{feat_c} → {out_h}×{out_w}×{feat_c}")
    print(f"  전송 크기: {compressed:,}개 숫자 (원본 이미지의 {compressed/(IMG_SIZE[0]*IMG_SIZE[1]*3)*100:.1f}%)")

    # ── 서버 측 분류기 (원본 코드의 Dense와 동일) ─────
    x   = Dropout(0.3)(x)                                    # 과적합 방지
    out = Dense(len(class_names), activation="softmax")(x)   # 최종 분류 (3클래스)

    # ── 파이프라인 모델: 이미지 → Conv → 풀링 → 분류 ─
    pipeline = Model(
        inputs=conv_extractor.input,
        outputs=out,
        name=f"pipeline_pool{pool_size}"
    )
    # 원본 코드와 동일: Adamax, categorical_crossentropy
    pipeline.compile(
        Adamax(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    # 조기종료: 검증 정확도가 10 에폭 동안 안 오르면 중단
    es = EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        restore_best_weights=True,  # 가장 좋았던 가중치로 복원
        verbose=1
    )

    hist = pipeline.fit(
        train_gen,
        epochs=EPOCHS_S2,
        verbose=1,
        validation_data=valid_gen,
        callbacks=[es]
    )

    # ── 테스트 데이터로 최종 평가 ──────────────────────
    test_gen.reset()  # 제너레이터 처음부터 다시 시작
    preds  = pipeline.predict(
        test_gen,
        steps=len(test_df)//BATCH_SIZE + 1,  # 모든 테스트 이미지 예측
        verbose=0
    )
    y_pred = np.argmax(preds[:len(test_df)], axis=1)   # 가장 높은 확률의 클래스 인덱스
    y_true = test_gen.classes[:len(y_pred)]             # 실제 정답 인덱스
    acc    = float(np.mean(y_true == y_pred))           # 전체 정확도
    cm     = confusion_matrix(y_true, y_pred)           # Confusion Matrix

    # 클래스별 정확도 계산 (대각선 / 행 합계)
    per_class = {}
    for i, cn in enumerate(class_names):
        per_class[cn] = float(cm[i,i] / cm[i].sum()) if cm[i].sum() > 0 else 0.0

    # 결과 저장
    results[pool_size] = {
        "acc":        acc,
        "val_acc":    max(hist.history["val_accuracy"]),  # 최고 검증 정확도
        "compressed": compressed,
        "out_size":   f"{out_h}×{out_w}",
        "label":      label,
        "history":    hist.history,
        "cm":         cm.tolist(),
        "per_class":  per_class,
        "epochs_run": len(hist.history["accuracy"])       # 실제 학습한 에폭 수
    }
    print(f"  ✅ Pool {pool_size}×{pool_size}: {acc*100:.2f}%")
    del pipeline  # 메모리 해제

# =============================================================
# 결과 표 출력
# =============================================================
best_ps  = max(results, key=lambda k: results[k]["acc"])   # 가장 좋은 풀링 크기
worst_ps = min(results, key=lambda k: results[k]["acc"])   # 가장 나쁜 풀링 크기

print("\n\n" + "="*95)
print("📊 풀링 크기별 실험 결과 (상세)")
print("="*95)
print(f"  {'풀링':>5}  {'출력크기':>8}  {'전송크기':>10}  {'압축률':>7}  {'전체정확도':>10}  "
      f"{'양성Benign':>10}  {'악성Malig':>10}  {'정상Normal':>10}  {'평가':>6}")
print("-"*95)
for ps in POOL_SIZES:
    r     = results[ps]
    lbl   = f"{ps}×{ps}" if ps < feat_h else "GAP"
    ratio = r["compressed"] / (IMG_SIZE[0]*IMG_SIZE[1]*3) * 100  # 원본 대비 압축률
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    grade = "🟢 우수" if r["acc"]>=0.85 else ("🟡 양호" if r["acc"]>=0.70 else ("🟠 미흡" if r["acc"]>=0.55 else "🔴 불량"))
    mark  = " 🏆" if ps==best_ps else (" 📉" if ps==worst_ps else "")
    print(f"  {lbl:>5}  {r['out_size']:>8}  {r['compressed']:>9,}개  {ratio:>6.1f}%  "
          f"{r['acc']*100:>9.2f}%  {b:>9.1f}%  {m:>9.1f}%  {n:>9.1f}%  {grade}{mark}")
print("="*95)

# 클래스별 상세 분석
print("\n📋 클래스별 상세 분석")
print("-"*95)
for ps in POOL_SIZES:
    r   = results[ps]
    lbl = f"{ps}×{ps}" if ps < feat_h else "GAP"
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    weak = []
    if b < 70: weak.append(f"양성↓({b:.0f}%)")
    if m < 70: weak.append(f"악성↓({m:.0f}%)")
    if n < 70: weak.append(f"정상↓({n:.0f}%)")
    note = "⚠️ " + ", ".join(weak) if weak else "✅ 모든 클래스 70%↑"
    print(f"  Pool {lbl:>4}: 전체 {r['acc']*100:.1f}% | "
          f"양성(Benign) {b:.1f}% | 악성(Malignant) {m:.1f}% | 정상(Normal) {n:.1f}% | {note}")
print("="*95)

# =============================================================
# 원본 코드 스타일: 사전학습 곡선 그래프
# =============================================================
tr_acc  = history_pretrain.history["accuracy"]
tr_loss = history_pretrain.history["loss"]
val_acc = history_pretrain.history["val_accuracy"]
val_loss= history_pretrain.history["val_loss"]
Epochs  = [i+1 for i in range(len(tr_acc))]

plt.figure(figsize=(20, 8))
plt.style.use("fivethirtyeight")  # 원본 코드와 동일한 스타일

plt.subplot(1, 2, 1)
plt.plot(Epochs, tr_loss,  "r", label="학습 손실")
plt.plot(Epochs, val_loss, "g", label="검증 손실")
plt.scatter(np.argmin(val_loss)+1, min(val_loss), s=150, c="blue",
            label=f"최저 검증 손실: epoch {np.argmin(val_loss)+1}")
plt.title("사전학습 손실 곡선")
plt.xlabel("에폭"); plt.ylabel("손실"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(Epochs, tr_acc,  "r", label="학습 정확도")
plt.plot(Epochs, val_acc, "g", label="검증 정확도")
plt.scatter(np.argmax(val_acc)+1, max(val_acc), s=150, c="blue",
            label=f"최고 검증 정확도: epoch {np.argmax(val_acc)+1}")
plt.title("사전학습 정확도 곡선")
plt.xlabel("에폭"); plt.ylabel("정확도"); plt.legend()

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/pretrain_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 사전학습 그래프 저장")

# 사전학습 모델 Confusion Matrix (원본 코드 스타일)
test_gen.reset()
preds_final  = pretrain_model.predict(test_gen, steps=len(test_df)//BATCH_SIZE+1, verbose=0)
y_pred_final = np.argmax(preds_final[:len(test_df)], axis=1)
cm_final     = confusion_matrix(test_gen.classes[:len(y_pred_final)], y_pred_final)

plt.figure(figsize=(10, 10))
plt.imshow(cm_final, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix - 사전학습 모델")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)
thresh = cm_final.max() / 2.
for i, j in itertools.product(range(cm_final.shape[0]), range(cm_final.shape[1])):
    plt.text(j, i, cm_final[i,j], horizontalalignment="center",
             color="white" if cm_final[i,j] > thresh else "black")
plt.tight_layout()
plt.ylabel("실제 정답"); plt.xlabel("AI 예측")
plt.savefig(f"{SAVE_DIR}/pretrain_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
# 원본 코드와 동일: classification_report 출력
print(classification_report(
    test_gen.classes[:len(y_pred_final)],
    y_pred_final, target_names=class_names
))

# =============================================================
# 그래프용 데이터 준비
# =============================================================
pool_labels = [f"{ps}×{ps}" if ps < feat_h else "GAP" for ps in POOL_SIZES]
accs        = [results[ps]["acc"]*100     for ps in POOL_SIZES]   # 테스트 정확도
val_accs    = [results[ps]["val_acc"]*100 for ps in POOL_SIZES]   # 최고 검증 정확도
compressed  = [results[ps]["compressed"]  for ps in POOL_SIZES]   # 전송 데이터 크기
epochs_run  = [results[ps]["epochs_run"]  for ps in POOL_SIZES]   # 실제 학습 에폭 수
b_accs = [results[ps]["per_class"].get("Bengin cases",    0)*100 for ps in POOL_SIZES]  # 양성 정확도
m_accs = [results[ps]["per_class"].get("Malignant cases", 0)*100 for ps in POOL_SIZES]  # 악성 정확도
n_accs = [results[ps]["per_class"].get("Normal cases",    0)*100 for ps in POOL_SIZES]  # 정상 정확도
# 보안 강도: 전송 데이터가 적을수록 보안 강함 (100% = 원본 이미지 크기)
security    = [100-(c/(IMG_SIZE[0]*IMG_SIZE[1]*3)*100) for c in compressed]
x_pos       = np.arange(len(POOL_SIZES))   # x축 위치 [0,1,2,3,4,5,6]
cmap_rg     = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(POOL_SIZES)))  # 빨강→노랑→초록 색상

# =============================================================
# 그래프 1: 테스트 & 검증 정확도 막대 비교
# =============================================================
fig1, ax1 = plt.subplots(figsize=(14, 7))
fig1.patch.set_facecolor(BG)
w = 0.35  # 막대 너비
# 테스트 정확도 막대 (RdYlGn 색상)
bars1 = ax1.bar(x_pos - w/2, accs,     width=w, color=cmap_rg, edgecolor="#222", zorder=3, label="테스트 정확도")
# 검증 정확도 막대 (파란 계열)
bars2 = ax1.bar(x_pos + w/2, val_accs, width=w,
                color=plt.cm.Blues(np.linspace(0.4, 0.9, len(POOL_SIZES))),
                edgecolor="#222", zorder=3, label="최고 검증 정확도")
ax1.axhline(max(accs), color="#ff6b6b", ls="--", lw=1.5, alpha=0.8, label=f"최고 테스트 {max(accs):.1f}%")
ax1.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
# 막대 위에 수치 표시
for bar, acc in zip(bars1, accs):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
for bar, acc in zip(bars2, val_accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{acc:.1f}%", ha="center", va="bottom", color="#74b9ff", fontsize=8)
ax1.set_xticks(x_pos); ax1.set_xticklabels(pool_labels, fontsize=10)
ax1.set_ylim(0, max(max(accs), max(val_accs))+20)
ax1.set_title("풀링 크기별 테스트 & 검증 정확도\n(작은 풀링 = 보안 강함 / 큰 풀링 = 정보 많음)",
              color="white", fontsize=13, fontweight="bold")
ax1.set_ylabel("정확도 (%)", color=TICK); ax1.set_xlabel("풀링 크기", color=TICK)
ax1.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax1)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph1_accuracy_bar.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프1 저장")

# =============================================================
# 그래프 2: 클래스별 + 전체 정확도 그룹 막대
# =============================================================
fig2, ax2 = plt.subplots(figsize=(16, 7))
fig2.patch.set_facecolor(BG)
w = 0.2  # 막대 4개가 나란히 있으므로 좁게
ax2.bar(x_pos-w*1.5, b_accs, width=w, label="양성 (Benign)",    color="#ff6b6b", edgecolor="#222", zorder=3)
ax2.bar(x_pos-w*0.5, m_accs, width=w, label="악성 (Malignant)", color="#ffd93d", edgecolor="#222", zorder=3)
ax2.bar(x_pos+w*0.5, n_accs, width=w, label="정상 (Normal)",    color="#6bcb77", edgecolor="#222", zorder=3)
ax2.bar(x_pos+w*1.5, accs,   width=w, label="전체 (Overall)",   color="#74b9ff", edgecolor="#222", zorder=3, alpha=0.85)
ax2.axhline(80, color="white", ls="--", lw=1, alpha=0.5, label="80% 기준선")
# 막대 위 수치 표시
for i, ps in enumerate(POOL_SIZES):
    b = results[ps]["per_class"].get("Bengin cases",    0)*100
    m = results[ps]["per_class"].get("Malignant cases", 0)*100
    n = results[ps]["per_class"].get("Normal cases",    0)*100
    a = results[ps]["acc"]*100
    for xoff, val, col in [(-w*1.5,b,"#ff6b6b"),(-w*0.5,m,"#ffd93d"),(w*0.5,n,"#6bcb77"),(w*1.5,a,"#74b9ff")]:
        ax2.text(i+xoff, val+0.5, f"{val:.0f}%", ha="center", va="bottom", color=col, fontsize=7, fontweight="bold")
ax2.set_xticks(x_pos); ax2.set_xticklabels(pool_labels, fontsize=10); ax2.set_ylim(0, 120)
ax2.set_title("풀링 크기별 클래스별 + 전체 정확도\n(클래스별로 풀링에 따른 성능 변화 비교)",
              color="white", fontsize=13, fontweight="bold")
ax2.set_ylabel("정확도 (%)", color=TICK)
ax2.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax2)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph2_class_accuracy.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프2 저장")

# =============================================================
# 그래프 3: 정확도 추세선 (전체 + 클래스별)
# =============================================================
fig3, ax3 = plt.subplots(figsize=(14, 7))
fig3.patch.set_facecolor(BG)
# 전체 정확도: 굵은 실선
ax3.plot(pool_labels, accs,   "o-",  color="#74b9ff", lw=3,   ms=10, mfc="white", mec="#74b9ff", label="전체 정확도", zorder=4)
# 클래스별 정확도: 얇은 점선
ax3.plot(pool_labels, b_accs, "s--", color="#ff6b6b", lw=1.8, ms=7,  mfc="white", label="양성 (Benign)")
ax3.plot(pool_labels, m_accs, "^--", color="#ffd93d", lw=1.8, ms=7,  mfc="white", label="악성 (Malignant)")
ax3.plot(pool_labels, n_accs, "D--", color="#6bcb77", lw=1.8, ms=7,  mfc="white", label="정상 (Normal)")
ax3.fill_between(range(len(pool_labels)), accs, alpha=0.07, color="#74b9ff")  # 영역 채우기
ax3.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
# 전체 정확도 수치 표시
for i, a in enumerate(accs):
    ax3.annotate(f"{a:.1f}%", (i, a), textcoords="offset points",
                 xytext=(0,10), ha="center", color="#74b9ff", fontsize=9, fontweight="bold")
ax3.set_ylim(0, max(accs)+20)
ax3.set_title("풀링 크기별 정확도 추세\n(전체 + 클래스별 변화 비교)", color="white", fontsize=13, fontweight="bold")
ax3.set_ylabel("정확도 (%)", color=TICK); ax3.set_xlabel("풀링 크기 (왼쪽=보안강함)", color=TICK)
ax3.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph3_trend.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프3 저장")

# =============================================================
# 그래프 4: 보안-성능 트레이드오프 산점도
# =============================================================
fig4, ax4 = plt.subplots(figsize=(10, 8))
fig4.patch.set_facecolor(BG)
# 점의 색상 = 정확도 (초록=높음, 빨강=낮음)
sc = ax4.scatter(security, accs, c=accs, cmap="RdYlGn", s=300,
                 edgecolors="white", lw=1.5, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(pool_labels):
    ax4.annotate(lbl, (security[i], accs[i]), textcoords="offset points",
                 xytext=(10, 5), color="white", fontsize=10, fontweight="bold")
ax4.axvspan(0,  20, alpha=0.05, color="#6bcb77", label="고성능 구역")   # 보안 낮고 성능 높은 구역
ax4.axvspan(80, 100, alpha=0.05, color="#ff6b6b", label="고보안 구역")  # 보안 높고 성능 낮은 구역
cbar = plt.colorbar(sc, ax=ax4)
cbar.set_label("AI 정확도 (%)", color="white", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax4.set_xlabel("보안 강도 (%) →\n(높을수록 더 많이 압축, 원본 정보 적음)", color=TICK, fontsize=11)
ax4.set_ylabel("AI 정확도 (%)", color=TICK, fontsize=11)
ax4.set_title("보안성 vs AI 성능 트레이드오프\n이상적 위치: 왼쪽 위 (보안 낮음 + 정확도 높음)", color="white", fontsize=13, fontweight="bold")
ax4.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax4)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph4_tradeoff.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프4 저장")

# =============================================================
# 그래프 5: 전송 데이터 크기 vs 정확도 이중축
# =============================================================
fig5, ax5a = plt.subplots(figsize=(14, 6))
fig5.patch.set_facecolor(BG)
ax5b = ax5a.twinx()  # 오른쪽 y축 추가
# 왼쪽 y축: 정확도 막대
ax5a.bar(x_pos, accs, color=cmap_rg, edgecolor="#222", zorder=3, width=0.6, alpha=0.85)
# 오른쪽 y축: 전송 크기 꺾은선 (로그 스케일)
ax5b.plot(x_pos, compressed, "D-", color="#74b9ff", lw=2.5, ms=10, mfc="white", zorder=4, label="전송 데이터 크기")
ax5b.set_yscale("log")  # 값 차이가 크므로 로그 스케일
for i, (a, c) in enumerate(zip(accs, compressed)):
    ax5a.text(i, a+0.5, f"{a:.1f}%", ha="center", color="white", fontsize=9, fontweight="bold")
    ax5b.text(i, c*1.3, f"{c:,}개", ha="center", color="#74b9ff", fontsize=7)
ax5a.set_xticks(x_pos); ax5a.set_xticklabels(pool_labels); ax5a.set_ylim(0, max(accs)+18)
ax5a.set_title("전송 데이터 크기 vs AI 정확도\n(풀링이 클수록 전송 데이터 많음 → 정확도↑, 보안↓)",
               color="white", fontsize=13, fontweight="bold")
ax5a.set_ylabel("정확도 (%)", color="#6bcb77", fontsize=12)
ax5b.set_ylabel("전송 데이터 크기 (로그 스케일, 개)", color="#74b9ff", fontsize=12)
ax5a.tick_params(axis="y", colors="#6bcb77"); ax5b.tick_params(axis="y", colors="#74b9ff")
ax5a.tick_params(axis="x", colors=TICK)
for sp in ax5a.spines.values(): sp.set_color("#444")
ax5a.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); ax5a.set_axisbelow(True)
ax5b.legend(facecolor="#222", labelcolor="white")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph5_datasize_vs_acc.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프5 저장")

# =============================================================
# 그래프 6: 클래스별 + 전체 정확도 히트맵
# =============================================================
fig6, ax6 = plt.subplots(figsize=(14, 5))
fig6.patch.set_facecolor(BG)
# 행: 양성/악성/정상/전체, 열: 풀링 크기별
hm = np.array([b_accs, m_accs, n_accs, accs])
im = ax6.imshow(hm, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
ax6.set_xticks(range(len(pool_labels))); ax6.set_xticklabels(pool_labels, color="white", fontsize=11)
ax6.set_yticks(range(4))
ax6.set_yticklabels(["양성\n(Benign)", "악성\n(Malignant)", "정상\n(Normal)", "전체\n(Overall)"],
                     color="white", fontsize=11)
# 각 셀에 수치 표시
for i in range(4):
    for j in range(len(POOL_SIZES)):
        v = hm[i, j]
        ax6.text(j, i, f"{v:.1f}%", ha="center", va="center",
                 color="black" if v > 60 else "white", fontsize=10, fontweight="bold")
cbar6 = plt.colorbar(im, ax=ax6)
cbar6.set_label("정확도 (%)", color="white")
cbar6.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar6.ax.yaxis.get_ticklabels(), color="white")
ax6.set_title("클래스별 + 전체 정확도 히트맵\n초록=높음 / 빨강=낮음", color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph6_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프6 저장")

# =============================================================
# 그래프 7: Confusion Matrix (최고 + 최저 풀링 비교)
# =============================================================
fig7, axes7 = plt.subplots(1, 2, figsize=(16, 7))
fig7.patch.set_facecolor(BG)
for ax_cm, ps, ttl in zip(axes7, [best_ps, worst_ps], ["🏆 최고 성능", "📉 최저 성능"]):
    ax_cm.set_facecolor(CELL)
    cm_arr = np.array(results[ps]["cm"])   # Confusion Matrix 배열
    lbl    = f"{ps}×{ps}" if ps < feat_h else "GAP"
    im7    = ax_cm.imshow(cm_arr, interpolation="nearest", cmap="Blues")
    plt.colorbar(im7, ax=ax_cm)
    tick_marks = np.arange(len(class_names))
    # 클래스 이름 한글화
    short_names = [c.replace(" cases","").replace("Bengin","양성").replace("Malignant","악성").replace("Normal","정상")
                   for c in class_names]
    ax_cm.set_xticks(tick_marks); ax_cm.set_xticklabels(short_names, color="white", fontsize=10, rotation=15)
    ax_cm.set_yticks(tick_marks); ax_cm.set_yticklabels(short_names, color="white", fontsize=10)
    thresh = cm_arr.max() / 2.
    # 각 셀에 숫자 표시 (원본 코드와 동일한 방식)
    for i, j in itertools.product(range(cm_arr.shape[0]), range(cm_arr.shape[1])):
        ax_cm.text(j, i, str(cm_arr[i,j]), ha="center", va="center",
                   color="white" if cm_arr[i,j] < thresh else "black", fontsize=14, fontweight="bold")
    ax_cm.set_title(f"{ttl}\nPool {lbl} | 정확도: {results[ps]['acc']*100:.1f}%",
                    color="white", fontsize=12, fontweight="bold")
    ax_cm.set_xlabel("AI 예측 (Predicted)", color=TICK, fontsize=11)
    ax_cm.set_ylabel("실제 정답 (Actual)",  color=TICK, fontsize=11)
fig7.suptitle("Confusion Matrix: 최고 풀링 vs 최저 풀링 비교\n대각선=정답 / 나머지=오답",
              color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph7_confusion.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프7 저장")

# =============================================================
# 그래프 8: 레이더 차트 (주요 풀링 크기 비교)
# =============================================================
# 대표 풀링 4개 선택 (첫번째, 중간, 끝에서 둘째, 마지막)
key_ps = sorted(set([POOL_SIZES[0], POOL_SIZES[len(POOL_SIZES)//2], POOL_SIZES[-2], POOL_SIZES[-1]]))
categories = ["전체 정확도", "양성 정확도", "악성 정확도", "정상 정확도", "보안 강도"]
N = len(categories)
# 레이더 차트 각도 계산 (5개 축을 원형으로 균등 배치)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # 닫힌 다각형을 위해 첫 값 반복

fig8, ax8 = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig8.patch.set_facecolor(BG); ax8.set_facecolor(CELL)
colors_radar = ["#f9ca24", "#6bcb77", "#ff6b6b", "#7c83fd"]

for idx, ps in enumerate(key_ps):
    r   = results[ps]
    b   = r["per_class"].get("Bengin cases",    0)*100
    m   = r["per_class"].get("Malignant cases", 0)*100
    n   = r["per_class"].get("Normal cases",    0)*100
    sec = 100 - (r["compressed"] / (IMG_SIZE[0]*IMG_SIZE[1]*3) * 100)  # 보안 강도
    vals = [r["acc"]*100, b, m, n, sec]
    vals += vals[:1]  # 닫힌 다각형
    lbl = f"Pool {ps}×{ps}" if ps < feat_h else "Pool GAP"
    ax8.plot(angles, vals, "o-", lw=2, color=colors_radar[idx % 4], label=lbl)
    ax8.fill(angles, vals, alpha=0.05, color=colors_radar[idx % 4])

ax8.set_xticks(angles[:-1])
ax8.set_xticklabels(categories, color="white", fontsize=11)
ax8.set_ylim(0, 100); ax8.set_yticks([20, 40, 60, 80, 100])
ax8.set_yticklabels(["20%","40%","60%","80%","100%"], color=TICK, fontsize=8)
ax8.grid(color="#444", linestyle="--", alpha=0.6)
ax8.spines["polar"].set_color("#444")
ax8.set_title("주요 풀링 크기별 성능 레이더 차트\n(각 축: 정확도/보안 강도)",
              color="white", fontsize=13, fontweight="bold", pad=25)
ax8.legend(facecolor="#222", labelcolor="white", fontsize=11, loc="upper right", bbox_to_anchor=(1.4, 1.1))
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph8_radar.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프8 저장")

# =============================================================
# 그래프 9: 풀링별 학습 곡선 전체 (7개 서브플롯)
# =============================================================
fig9, axes9 = plt.subplots(2, 4, figsize=(24, 11))
fig9.patch.set_facecolor(BG)
axes9 = axes9.flatten()  # 2D 배열 → 1D로 펼치기

for i, ps in enumerate(POOL_SIZES):
    ax = axes9[i]; ax.set_facecolor(CELL)
    h  = results[ps]["history"]         # 해당 풀링의 학습 기록
    ep = range(1, len(h["accuracy"])+1) # 에폭 번호 1부터

    # 정확도 실선 (굵게)
    ax.plot(ep, [v*100 for v in h["accuracy"]],     color="#6bcb77", lw=2, label="학습 정확도")
    ax.plot(ep, [v*100 for v in h["val_accuracy"]], color="#ffd93d", lw=2, label="검증 정확도")
    # 손실 점선 (가늘게, 참고용)
    ax.plot(ep, [v*100 for v in h["loss"]],     color="#ff6b6b", lw=1.5, ls="--", label="학습 손실", alpha=0.7)
    ax.plot(ep, [v*100 for v in h["val_loss"]], color="#74b9ff", lw=1.5, ls="--", label="검증 손실", alpha=0.7)
    # 최고 검증 정확도 시점 수직선
    best_ep = np.argmax(h["val_accuracy"])+1
    ax.axvline(best_ep, color="white", ls=":", lw=1, alpha=0.5, label=f"최고 ep:{best_ep}")

    lbl = f"{ps}×{ps}" if ps < feat_h else "GAP"
    b   = results[ps]["per_class"].get("Bengin cases",    0)*100
    m_v = results[ps]["per_class"].get("Malignant cases", 0)*100
    n_v = results[ps]["per_class"].get("Normal cases",    0)*100
    ax.set_title(
        f"Pool {lbl}  |  테스트: {results[ps]['acc']*100:.1f}%\n"
        f"양성:{b:.0f}% 악성:{m_v:.0f}% 정상:{n_v:.0f}% ({results[ps]['epochs_run']}ep)",
        color="white", fontsize=9, fontweight="bold"
    )
    ax.set_xlabel("에폭", color=TICK, fontsize=8)
    ax.set_ylabel("값 (%)", color=TICK, fontsize=8)
    ax.set_ylim(0, 105)
    ax.legend(facecolor="#222", labelcolor="white", fontsize=7)
    for sp in ax.spines.values(): sp.set_color("#444")
    ax.tick_params(colors=TICK)
    ax.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

# 남은 서브플롯 숨기기
for j in range(len(POOL_SIZES), len(axes9)):
    axes9[j].set_visible(False)

fig9.suptitle("풀링 크기별 학습 곡선 전체\n(정확도 + 손실, 타이틀에 클래스별 결과 포함)",
              color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph9_learning_curves.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프9 저장")

# =============================================================
# 그래프 10: 종합 요약 대시보드 (6개 서브플롯)
# =============================================================
from matplotlib.gridspec import GridSpec
fig10 = plt.figure(figsize=(22, 14))
fig10.patch.set_facecolor(BG)
gs10 = GridSpec(3, 3, figure=fig10, hspace=0.5, wspace=0.35)

# ── [0행 0~1열] 전체 정확도 막대 ──────────────────────
a1 = fig10.add_subplot(gs10[0, :2]); a1.set_facecolor(CELL)
b10 = a1.bar(x_pos, accs, color=cmap_rg, edgecolor="#333", zorder=3)
for bar, acc in zip(b10, accs):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    a1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=10, fontweight="bold")
a1.set_xticks(x_pos); a1.set_xticklabels(pool_labels)
a1.set_ylim(0, max(accs)+18)
a1.set_title("풀링 크기별 테스트 정확도", color="white", fontsize=12, fontweight="bold")
a1.set_ylabel("정확도 (%)", color=TICK)
for sp in a1.spines.values(): sp.set_color("#444")
a1.tick_params(colors=TICK); a1.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a1.set_axisbelow(True)

# ── [0행 2열] 텍스트 요약 ──────────────────────────────
a2 = fig10.add_subplot(gs10[0, 2]); a2.set_facecolor(CELL); a2.axis("off")
lbl_b2 = f"{best_ps}×{best_ps}" if best_ps<feat_h else "GAP"
lbl_w2 = f"{worst_ps}×{worst_ps}" if worst_ps<feat_h else "GAP"
b_best = results[best_ps]["per_class"].get("Bengin cases",    0)*100
m_best = results[best_ps]["per_class"].get("Malignant cases", 0)*100
n_best = results[best_ps]["per_class"].get("Normal cases",    0)*100
summary_txt = (
    f"📊 실험 요약\n\n"
    f"풀링: {POOL_SIZES}\n"
    f"오버샘플링: {TARGET_COUNT}장/클래스\n\n"
    f"🏆 최고: Pool {lbl_b2}\n"
    f"  전체: {results[best_ps]['acc']*100:.1f}%\n"
    f"  양성: {b_best:.1f}%\n"
    f"  악성: {m_best:.1f}%\n"
    f"  정상: {n_best:.1f}%\n\n"
    f"📉 최저: Pool {lbl_w2}\n"
    f"  전체: {results[worst_ps]['acc']*100:.1f}%\n\n"
    f"🎯 결론\n"
    f"  풀링↑ → 정확도↑\n"
    f"  풀링↓ → 보안↑"
)
a2.text(0.05, 0.97, summary_txt, transform=a2.transAxes,
        color="white", fontsize=9.5, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#21262d", edgecolor="#555"))
a2.set_title("실험 결론", color="white", fontsize=12, fontweight="bold")

# ── [1행 0~1열] 클래스별 히트맵 ───────────────────────
a3 = fig10.add_subplot(gs10[1, :2]); a3.set_facecolor(CELL)
hm2 = np.array([b_accs, m_accs, n_accs, accs])
im3 = a3.imshow(hm2, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
a3.set_xticks(range(len(pool_labels))); a3.set_xticklabels(pool_labels, color="white", fontsize=10)
a3.set_yticks(range(4))
a3.set_yticklabels(["양성", "악성", "정상", "전체"], color="white", fontsize=10)
for i in range(4):
    for j in range(len(POOL_SIZES)):
        v = hm2[i, j]
        a3.text(j, i, f"{v:.0f}%", ha="center", va="center",
                color="black" if v>60 else "white", fontsize=10, fontweight="bold")
a3.set_title("클래스별 + 전체 정확도 히트맵", color="white", fontsize=12, fontweight="bold")
plt.colorbar(im3, ax=a3).ax.yaxis.set_tick_params(color="white")

# ── [1행 2열] 보안 vs 성능 산점도 ─────────────────────
a4 = fig10.add_subplot(gs10[1, 2]); a4.set_facecolor(CELL)
sc4 = a4.scatter(security, accs, c=accs, cmap="RdYlGn", s=150,
                 edgecolors="white", lw=1, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(pool_labels):
    a4.annotate(lbl, (security[i], accs[i]), textcoords="offset points",
                xytext=(6, 4), color="white", fontsize=8)
a4.set_xlabel("보안 강도 →", color=TICK); a4.set_ylabel("AI 정확도 (%)", color=TICK)
a4.set_title("보안 vs 성능", color="white", fontsize=12, fontweight="bold")
for sp in a4.spines.values(): sp.set_color("#444")
a4.tick_params(colors=TICK); a4.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a4.set_axisbelow(True)

# ── [2행 0열] 최고 풀링 학습 곡선 ─────────────────────
a5 = fig10.add_subplot(gs10[2, 0]); a5.set_facecolor(CELL)
h_b  = results[best_ps]["history"]
ep_b = range(1, len(h_b["accuracy"])+1)
a5.plot(ep_b, [v*100 for v in h_b["accuracy"]],     color="#6bcb77", lw=2, label="학습 정확도")
a5.plot(ep_b, [v*100 for v in h_b["val_accuracy"]], color="#ffd93d", lw=2, label="검증 정확도")
a5.set_title(f"최고 Pool {lbl_b2} 학습 곡선", color="white", fontsize=11, fontweight="bold")
a5.set_xlabel("에폭", color=TICK); a5.set_ylabel("정확도 (%)", color=TICK); a5.set_ylim(0, 105)
a5.legend(facecolor="#222", labelcolor="white", fontsize=9)
for sp in a5.spines.values(): sp.set_color("#444")
a5.tick_params(colors=TICK); a5.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a5.set_axisbelow(True)

# ── [2행 1열] 전송 데이터 크기 막대 (로그) ────────────
a6 = fig10.add_subplot(gs10[2, 1]); a6.set_facecolor(CELL)
a6.bar(x_pos, compressed, color=cmap_rg, edgecolor="#222", zorder=3)
a6.set_yscale("log")  # 크기 차이가 크므로 로그 스케일
a6.set_xticks(x_pos); a6.set_xticklabels(pool_labels)
a6.set_title("전송 데이터 크기 (로그 스케일)", color="white", fontsize=11, fontweight="bold")
a6.set_ylabel("전송 크기 (로그, 개)", color=TICK)
for sp in a6.spines.values(): sp.set_color("#444")
a6.tick_params(colors=TICK); a6.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a6.set_axisbelow(True)

# ── [2행 2열] 학습 에폭 수 & 정확도 이중축 ───────────
a7 = fig10.add_subplot(gs10[2, 2]); a7.set_facecolor(CELL)
a7b = a7.twinx()  # 오른쪽 y축
a7.bar(x_pos, epochs_run, color=cmap_rg, edgecolor="#222", zorder=3, alpha=0.85)
a7b.plot(x_pos, accs, "o-", color="#f9ca24", lw=2, ms=8, mfc="white", zorder=4)
for i, ep in enumerate(epochs_run):
    a7.text(i, ep+0.3, f"{ep}ep", ha="center", color="white", fontsize=8)
a7.set_xticks(x_pos); a7.set_xticklabels(pool_labels)
a7.set_ylim(0, EPOCHS_S2+10)
a7.set_title("학습 에폭 수 & 정확도", color="white", fontsize=11, fontweight="bold")
a7.set_ylabel("에폭 수", color="#6bcb77", fontsize=10)
a7b.set_ylabel("정확도 (%)", color="#f9ca24", fontsize=10)
a7.tick_params(axis="y", colors="#6bcb77"); a7b.tick_params(axis="y", colors="#f9ca24")
a7.tick_params(axis="x", colors=TICK)
for sp in a7.spines.values(): sp.set_color("#444")
a7.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a7.set_axisbelow(True)

fig10.suptitle(
    f"Edge-Server Split CNN 종합 대시보드\n"
    f"오버샘플링 {TARGET_COUNT}장/클래스 | MobileNetV2 | Adamax",
    color="white", fontsize=14, fontweight="bold", y=1.01
)
plt.savefig(f"{SAVE_DIR}/graph10_dashboard.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프10 저장")

# =============================================================
# 최종 출력
# =============================================================
print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ 그래프 10개 저장 완료! → {SAVE_DIR}/
╠══════════════════════════════════════════════════════╣
║  pretrain_history  - 사전학습 곡선 (원본 스타일)
║  pretrain_confusion - 사전학습 Confusion Matrix
║  graph1  - 테스트 & 검증 정확도 막대
║  graph2  - 클래스별 + 전체 정확도
║  graph3  - 정확도 추세선
║  graph4  - 보안-성능 트레이드오프
║  graph5  - 전송 크기 vs 정확도
║  graph6  - 클래스별 히트맵
║  graph7  - Confusion Matrix (최고 vs 최저)
║  graph8  - 레이더 차트
║  graph9  - 학습 곡선 전체 (7개)
║  graph10 - 종합 대시보드
╚══════════════════════════════════════════════════════╝

🏆 최고: Pool {best_ps}×{best_ps} → {results[best_ps]["acc"]*100:.2f}%
📉 최저: Pool {worst_ps}×{worst_ps} → {results[worst_ps]["acc"]*100:.2f}%
""")


In [ ]:
# =============================================================
# 엣지-서버 분리 풀링 실험 코드 (RTX 4090 최적화 버전)
# IMG_SIZE=512, Mixed Precision, 더 다양한 풀링 실험
# =============================================================

import os
import shutil
import itertools
import random
import warnings
import gc           # 메모리 해제
import subprocess
import glob
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from PIL import Image, ImageEnhance
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, MaxPooling2D, Flatten, Dropout
)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import mixed_precision

print("modules loaded")

# =============================================================
# RTX 4090 GPU 최적화 설정
# =============================================================
# GPU 메모리 동적 할당 (필요할 때만 사용, OOM 방지)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ GPU 설정 완료: {gpus}")

# Mixed Precision: float16으로 계산 → 속도 2배↑, 메모리 절반↓
# RTX 4090은 FP16 연산에 최적화되어 있음
mixed_precision.set_global_policy("mixed_float16")
print("✅ Mixed Precision FP16 활성화 (속도↑, 메모리↓)")
print(f"GPU 목록: {tf.config.list_physical_devices('GPU')}")

# =============================================================
# 한글 폰트 설정
# =============================================================
subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], capture_output=True)
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

cache_dir = matplotlib.get_cachedir()
for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
    try: os.remove(f)
    except: pass

fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams["font.family"]       = "sans-serif"
matplotlib.rcParams["font.sans-serif"]   = [font_name]
matplotlib.rcParams["axes.unicode_minus"] = False
print(f"✅ 한글 폰트 적용: {font_name}")

# =============================================================
# 실험 설정
# =============================================================
BASE_DIR     = "/workspace/Lung Cancer/Train"
TARGET_COUNT = 1000

# ★ 512×512로 업그레이드! (원본 데이터 크기와 동일)
# MobileNetV2에 512 넣으면 특징맵이 16×16이 됨
# → 풀링 실험이 1×1 ~ 16×16으로 더 다양해짐!
IMG_SIZE     = (512, 512)

# BATCH_SIZE: 512×512는 메모리 많이 씀
# RTX 4090 24GB에서 안전한 값 = 16
# OOM 나면 8로 줄이기
BATCH_SIZE   = 16

EPOCHS       = 30      # Mixed Precision으로 빠르므로 충분
EPOCHS_FT    = 15
EPOCHS_S2    = 30

# 512×512 → 특징맵 16×16
# 풀링 실험: 1, 2, 4, 8, 16 (대표값만 선택해서 시간 절약)
# 1×1=최대압축, 16×16=압축없음
POOL_SIZES   = [1, 2, 4, 8, 16]

SAVE_DIR     = "/workspace/viz"
os.makedirs(SAVE_DIR, exist_ok=True)

# 그래프 다크 테마
BG   = "#0d1117"
CELL = "#161b22"
GRID = "#333333"
TICK = "#aaaaaa"

def style_ax(ax):
    ax.set_facecolor(CELL)
    for sp in ax.spines.values():
        sp.set_color("#444444")
    ax.tick_params(colors=TICK, labelsize=9)
    ax.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.6)
    ax.set_axisbelow(True)

print(f"""
설정 확인:
  이미지: {IMG_SIZE[0]}×{IMG_SIZE[1]} (원본 크기!)
  배치:   {BATCH_SIZE}장
  풀링:   {POOL_SIZES}
  특징맵: 16×16×1280 (512 입력 시)
  Mixed Precision: FP16 ✅
""")

# =============================================================
# 오버샘플링
# =============================================================
def oversample_dir(base_dir, target_count, seed=42):
    out_dir = base_dir.rstrip("/") + "_oversampled"
    classes = sorted([d for d in os.listdir(base_dir)
                      if os.path.isdir(os.path.join(base_dir, d))])

    if os.path.exists(out_dir):
        all_done = all(
            os.path.exists(os.path.join(out_dir, c)) and
            len([f for f in os.listdir(os.path.join(out_dir, c))
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]) >= target_count
            for c in classes
        )
        if all_done:
            total = sum(len([f for f in os.listdir(os.path.join(out_dir, c))
                             if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                        for c in classes)
            print(f"  오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    random.seed(seed)
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir, cls, f), dst)

    for cls in classes:
        cls_out   = os.path.join(out_dir, cls)
        src_files = [f for f in os.listdir(os.path.join(base_dir, cls))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))
                     and not f.startswith("aug_")]
        cur_files = [f for f in os.listdir(cls_out)
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))]
        needed = target_count - len(cur_files)
        if needed <= 0:
            print(f"  {cls}: {len(cur_files)}장 (충분)"); continue
        print(f"  {cls}: {len(cur_files)}장 → {target_count}장 ({needed}장 증강)")
        generated = 0
        while generated < needed:
            src_f = random.choice(src_files)
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(random.uniform(-30, 30), expand=False)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if random.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.7, 1.3))
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"), "JPEG", quality=95)
                generated += 1
            except: continue

    total = sum(len([f for f in os.listdir(os.path.join(out_dir, c))
                     if f.lower().endswith((".jpg",".jpeg",".png",".bmp"))])
                for c in classes)
    print(f"  ✅ 오버샘플링 완료: 총 {total}장")
    return out_dir

# =============================================================
# 오버샘플링 실행
# =============================================================
print("\n오버샘플링 시작...")
train_dir = oversample_dir(BASE_DIR, TARGET_COUNT)

# =============================================================
# DataFrame 생성
# =============================================================
filepaths = []
labels    = []

classes_list = sorted([d for d in os.listdir(train_dir)
                        if os.path.isdir(os.path.join(train_dir, d))])
for cls in classes_list:
    cls_dir = os.path.join(train_dir, cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith((".jpg",".jpeg",".png",".bmp")):
            filepaths.append(os.path.join(cls_dir, f))
            labels.append(cls)

df = pd.concat([pd.Series(filepaths, name="filepaths"),
                pd.Series(labels,    name="labels")], axis=1)

print(f"\n전체 데이터: {len(df)}장")
print(df["labels"].value_counts())

# =============================================================
# 데이터 분리 (80/10/10)
# =============================================================
train_df, dummy_df = train_test_split(
    df, train_size=0.8, shuffle=True,
    random_state=123, stratify=df["labels"]
)
valid_df, test_df = train_test_split(
    dummy_df, train_size=0.5, shuffle=True,
    random_state=123, stratify=dummy_df["labels"]
)
print(f"\n학습: {len(train_df)}장 | 검증: {len(valid_df)}장 | 테스트: {len(test_df)}장")

# =============================================================
# ImageDataGenerator
# 512×512는 메모리가 크므로 prefetch 설정 주의
# =============================================================
tr_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True
)
ts_gen = ImageDataGenerator(rescale=1./255)

train_gen = tr_gen.flow_from_dataframe(
    train_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE,           # 512×512!
    class_mode="categorical",
    color_mode="rgb",
    shuffle=True,
    batch_size=BATCH_SIZE           # 16 (512 크기에서 안전)
)
valid_gen = ts_gen.flow_from_dataframe(
    valid_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE
)
test_gen = ts_gen.flow_from_dataframe(
    test_df, x_col="filepaths", y_col="labels",
    target_size=IMG_SIZE, class_mode="categorical",
    color_mode="rgb", shuffle=False, batch_size=BATCH_SIZE
)

g_dict      = train_gen.class_indices
class_names = list(g_dict.keys())
print(f"\n클래스: {g_dict}")

# =============================================================
# STEP 1: MobileNetV2 사전학습 (512×512 입력)
#
# 512×512 입력 시:
# → 특징맵이 16×16×1280 (224 때는 7×7)
# → 더 세밀한 공간 정보 보존
# → 풀링 실험이 1×1 ~ 16×16으로 더 다양!
# =============================================================
print("\n" + "="*55)
print("STEP 1: MobileNetV2 사전학습 (512×512)")
print("="*55)

# MobileNetV2는 원래 224용이지만 512도 입력 가능
# include_preprocessing=False: 정규화를 ImageDataGenerator에서 이미 했음
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)
base_model.trainable = False

x   = GlobalAveragePooling2D()(base_model.output)
x   = Dropout(0.3)(x)
# dtype="float32": Mixed Precision에서 마지막 레이어는 float32 필요
out = Dense(len(class_names), activation="softmax", dtype="float32")(x)

pretrain_model = Model(inputs=base_model.input, outputs=out)

pretrain_model.compile(
    Adamax(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"[1단계] 헤드 학습 (최대 {EPOCHS}ep)...")
pretrain_model.fit(
    train_gen, epochs=EPOCHS, verbose=1,
    validation_data=valid_gen
)

# Conv 마지막 30개 레이어 열기
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

pretrain_model.compile(
    Adamax(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print(f"\n[2단계] Fine-tuning (최대 {EPOCHS_FT}ep)...")
history_pretrain = pretrain_model.fit(
    train_gen, epochs=EPOCHS_FT, verbose=1,
    validation_data=valid_gen
)

print(f"\n✅ 사전학습 완료!")
print(f"   최종 학습 정확도: {history_pretrain.history['accuracy'][-1]*100:.2f}%")
print(f"   최종 검증 정확도: {history_pretrain.history['val_accuracy'][-1]*100:.2f}%")

# Conv 가중치 완전 고정
base_model.trainable = False

# =============================================================
# conv_extractor 생성
# pretrain_model 내부에서 마지막 4D 출력 레이어 찾기
# =============================================================
conv_output = None
for layer in pretrain_model.layers:
    if hasattr(layer, "output_shape"):
        shape = layer.output_shape
        if isinstance(shape, tuple) and len(shape) == 4:
            conv_output = layer.output

conv_extractor = Model(
    inputs=pretrain_model.input,
    outputs=conv_output,
    name="conv_extractor"
)
conv_extractor.trainable = False

feat_h, feat_w, feat_c = conv_extractor.output_shape[1:]
print(f"\n특징맵 크기: {feat_h}×{feat_w}×{feat_c}")
print(f"224 때:     7×7×1280  = 62,720개")
print(f"512 지금:  {feat_h}×{feat_w}×{feat_c} = {feat_h*feat_w*feat_c:,}개 (더 세밀!)")

# GPU 메모리 정리
gc.collect()
tf.keras.backend.clear_session()

# =============================================================
# STEP 2: 풀링 크기별 실험
# =============================================================
print("\n" + "="*55)
print(f"STEP 2: 풀링 크기별 실험 ({POOL_SIZES})")
print("="*55)

# conv_extractor를 다시 로드 (clear_session 후 재생성)
conv_extractor = Model(
    inputs=pretrain_model.input,
    outputs=conv_output,
    name="conv_extractor"
)
conv_extractor.trainable = False

results = {}

for pool_size in POOL_SIZES:
    print(f"\n--- Pool {pool_size}×{pool_size} ---")

    if pool_size >= feat_h:
        # 특징맵 전체를 하나로 압축 (GlobalAvgPool)
        x          = GlobalAveragePooling2D()(conv_extractor.output)
        compressed = feat_c
        out_h, out_w = 1, 1
        label = "GlobalAvgPool"
    else:
        # pool_size×pool_size 구역마다 최댓값 선택
        x     = MaxPooling2D(pool_size=(pool_size, pool_size))(conv_extractor.output)
        out_h = feat_h // pool_size
        out_w = feat_w // pool_size
        x     = Flatten()(x)
        compressed = out_h * out_w * feat_c
        label = f"MaxPool {pool_size}×{pool_size}"

    orig_pixels = IMG_SIZE[0] * IMG_SIZE[1] * 3
    print(f"  특징맵 {feat_h}×{feat_w}×{feat_c} → {out_h}×{out_w}×{feat_c}")
    print(f"  전송: {compressed:,}개 (원본 {orig_pixels:,}개의 {compressed/orig_pixels*100:.1f}%)")

    # dtype="float32": Mixed Precision 환경에서 출력 레이어는 float32
    x   = Dropout(0.3)(x)
    out = Dense(len(class_names), activation="softmax", dtype="float32")(x)

    pipeline = Model(
        inputs=conv_extractor.input,
        outputs=out,
        name=f"pipeline_pool{pool_size}"
    )
    pipeline.compile(
        Adamax(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    es = EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        restore_best_weights=True,
        verbose=1
    )

    hist = pipeline.fit(
        train_gen, epochs=EPOCHS_S2, verbose=1,
        validation_data=valid_gen, callbacks=[es]
    )

    # 테스트 평가
    test_gen.reset()
    preds  = pipeline.predict(
        test_gen,
        steps=len(test_df)//BATCH_SIZE + 1,
        verbose=0
    )
    y_pred = np.argmax(preds[:len(test_df)], axis=1)
    y_true = test_gen.classes[:len(y_pred)]
    acc    = float(np.mean(y_true == y_pred))
    cm     = confusion_matrix(y_true, y_pred)

    per_class = {}
    for i, cn in enumerate(class_names):
        per_class[cn] = float(cm[i,i] / cm[i].sum()) if cm[i].sum() > 0 else 0.0

    results[pool_size] = {
        "acc":        acc,
        "val_acc":    max(hist.history["val_accuracy"]),
        "compressed": compressed,
        "out_size":   f"{out_h}×{out_w}",
        "label":      label,
        "history":    hist.history,
        "cm":         cm.tolist(),
        "per_class":  per_class,
        "epochs_run": len(hist.history["accuracy"])
    }
    print(f"  ✅ Pool {pool_size}×{pool_size}: {acc*100:.2f}%")

    # 풀링마다 메모리 정리 (512×512는 메모리 많이 씀)
    del pipeline
    gc.collect()

# =============================================================
# 결과 표
# =============================================================
best_ps  = max(results, key=lambda k: results[k]["acc"])
worst_ps = min(results, key=lambda k: results[k]["acc"])
orig_pixels = IMG_SIZE[0] * IMG_SIZE[1] * 3

print("\n\n" + "="*100)
print("📊 풀링 크기별 실험 결과 (상세) - 512×512 버전")
print("="*100)
print(f"  {'풀링':>5}  {'출력크기':>8}  {'전송크기':>10}  {'압축률':>7}  {'전체정확도':>10}  "
      f"{'양성Benign':>10}  {'악성Malig':>10}  {'정상Normal':>10}  {'평가':>6}")
print("-"*100)
for ps in POOL_SIZES:
    r     = results[ps]
    lbl   = f"{ps}×{ps}" if ps < feat_h else "GAP"
    ratio = r["compressed"] / orig_pixels * 100
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    grade = "🟢 우수" if r["acc"]>=0.85 else ("🟡 양호" if r["acc"]>=0.70 else ("🟠 미흡" if r["acc"]>=0.55 else "🔴 불량"))
    mark  = " 🏆" if ps==best_ps else (" 📉" if ps==worst_ps else "")
    print(f"  {lbl:>5}  {r['out_size']:>8}  {r['compressed']:>9,}개  {ratio:>6.1f}%  "
          f"{r['acc']*100:>9.2f}%  {b:>9.1f}%  {m:>9.1f}%  {n:>9.1f}%  {grade}{mark}")
print("="*100)

print("\n📋 클래스별 상세 분석")
print("-"*100)
for ps in POOL_SIZES:
    r   = results[ps]
    lbl = f"{ps}×{ps}" if ps < feat_h else "GAP"
    b = r["per_class"].get("Bengin cases",    0)*100
    m = r["per_class"].get("Malignant cases", 0)*100
    n = r["per_class"].get("Normal cases",    0)*100
    weak = []
    if b < 70: weak.append(f"양성↓({b:.0f}%)")
    if m < 70: weak.append(f"악성↓({m:.0f}%)")
    if n < 70: weak.append(f"정상↓({n:.0f}%)")
    note = "⚠️ " + ", ".join(weak) if weak else "✅ 모든 클래스 70%↑"
    print(f"  Pool {lbl:>4}: 전체 {r['acc']*100:.1f}% | "
          f"양성 {b:.1f}% | 악성 {m:.1f}% | 정상 {n:.1f}% | {note}")
print("="*100)

print(f"""
💡 224 vs 512 비교:

  224×224 때:
  → 특징맵: 7×7×1280 = 62,720개
  → 풀링: 1×1 ~ 7×7 (7가지)

  512×512 지금:
  → 특징맵: {feat_h}×{feat_w}×{feat_c} = {feat_h*feat_w*feat_c:,}개
  → 풀링: {POOL_SIZES} ({len(POOL_SIZES)}가지)
  → 더 세밀한 공간 정보 → 정확도 향상 기대!
  → 원본 이미지 크기와 동일 → 정보 손실 최소화
""")

# =============================================================
# 사전학습 그래프 (원본 코드 스타일)
# =============================================================
tr_acc  = history_pretrain.history["accuracy"]
tr_loss = history_pretrain.history["loss"]
val_acc = history_pretrain.history["val_accuracy"]
val_loss= history_pretrain.history["val_loss"]
Epochs  = [i+1 for i in range(len(tr_acc))]

plt.figure(figsize=(20, 8))
plt.style.use("fivethirtyeight")

plt.subplot(1, 2, 1)
plt.plot(Epochs, tr_loss,  "r", label="학습 손실")
plt.plot(Epochs, val_loss, "g", label="검증 손실")
plt.scatter(np.argmin(val_loss)+1, min(val_loss), s=150, c="blue",
            label=f"최저: epoch {np.argmin(val_loss)+1}")
plt.title("사전학습 손실 곡선 (512×512)")
plt.xlabel("에폭"); plt.ylabel("손실"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(Epochs, tr_acc,  "r", label="학습 정확도")
plt.plot(Epochs, val_acc, "g", label="검증 정확도")
plt.scatter(np.argmax(val_acc)+1, max(val_acc), s=150, c="blue",
            label=f"최고: epoch {np.argmax(val_acc)+1}")
plt.title("사전학습 정확도 곡선 (512×512)")
plt.xlabel("에폭"); plt.ylabel("정확도"); plt.legend()

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/pretrain_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 사전학습 그래프 저장")

# 사전학습 Confusion Matrix
test_gen.reset()
preds_final  = pretrain_model.predict(test_gen, steps=len(test_df)//BATCH_SIZE+1, verbose=0)
y_pred_final = np.argmax(preds_final[:len(test_df)], axis=1)
cm_final     = confusion_matrix(test_gen.classes[:len(y_pred_final)], y_pred_final)

plt.figure(figsize=(10, 10))
plt.imshow(cm_final, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Confusion Matrix - 사전학습 모델 (512×512)")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)
thresh = cm_final.max() / 2.
for i, j in itertools.product(range(cm_final.shape[0]), range(cm_final.shape[1])):
    plt.text(j, i, cm_final[i,j], horizontalalignment="center",
             color="white" if cm_final[i,j] > thresh else "black")
plt.tight_layout()
plt.ylabel("실제 정답"); plt.xlabel("AI 예측")
plt.savefig(f"{SAVE_DIR}/pretrain_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(classification_report(test_gen.classes[:len(y_pred_final)],
                              y_pred_final, target_names=class_names))

# =============================================================
# 그래프용 데이터 준비
# =============================================================
pool_labels = [f"{ps}×{ps}" if ps < feat_h else "GAP" for ps in POOL_SIZES]
accs        = [results[ps]["acc"]*100     for ps in POOL_SIZES]
val_accs    = [results[ps]["val_acc"]*100 for ps in POOL_SIZES]
compressed  = [results[ps]["compressed"]  for ps in POOL_SIZES]
epochs_run  = [results[ps]["epochs_run"]  for ps in POOL_SIZES]
b_accs = [results[ps]["per_class"].get("Bengin cases",    0)*100 for ps in POOL_SIZES]
m_accs = [results[ps]["per_class"].get("Malignant cases", 0)*100 for ps in POOL_SIZES]
n_accs = [results[ps]["per_class"].get("Normal cases",    0)*100 for ps in POOL_SIZES]
security    = [100-(c/orig_pixels*100) for c in compressed]
x_pos       = np.arange(len(POOL_SIZES))
cmap_rg     = plt.cm.RdYlGn(np.linspace(0.15, 0.95, len(POOL_SIZES)))

# =============================================================
# 그래프 1: 테스트 & 검증 정확도
# =============================================================
fig1, ax1 = plt.subplots(figsize=(14, 7))
fig1.patch.set_facecolor(BG)
w = 0.35
bars1 = ax1.bar(x_pos-w/2, accs,     width=w, color=cmap_rg, edgecolor="#222", zorder=3, label="테스트 정확도")
bars2 = ax1.bar(x_pos+w/2, val_accs, width=w,
                color=plt.cm.Blues(np.linspace(0.4, 0.9, len(POOL_SIZES))),
                edgecolor="#222", zorder=3, label="최고 검증 정확도")
ax1.axhline(max(accs), color="#ff6b6b", ls="--", lw=1.5, alpha=0.8, label=f"최고 {max(accs):.1f}%")
ax1.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
for bar, acc in zip(bars1, accs):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=9, fontweight="bold")
for bar, acc in zip(bars2, val_accs):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f"{acc:.1f}%", ha="center", va="bottom", color="#74b9ff", fontsize=8)
ax1.set_xticks(x_pos); ax1.set_xticklabels(pool_labels, fontsize=10)
ax1.set_ylim(0, max(max(accs), max(val_accs))+20)
ax1.set_title(f"풀링 크기별 테스트 & 검증 정확도 (512×512)\n(작은 풀링 = 보안 강함 / 큰 풀링 = 정보 많음)",
              color="white", fontsize=13, fontweight="bold")
ax1.set_ylabel("정확도 (%)", color=TICK); ax1.set_xlabel("풀링 크기", color=TICK)
ax1.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax1)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph1_accuracy_bar.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프1 저장")

# =============================================================
# 그래프 2: 클래스별 + 전체 정확도
# =============================================================
fig2, ax2 = plt.subplots(figsize=(16, 7))
fig2.patch.set_facecolor(BG)
w = 0.2
ax2.bar(x_pos-w*1.5, b_accs, width=w, label="양성 (Benign)",    color="#ff6b6b", edgecolor="#222", zorder=3)
ax2.bar(x_pos-w*0.5, m_accs, width=w, label="악성 (Malignant)", color="#ffd93d", edgecolor="#222", zorder=3)
ax2.bar(x_pos+w*0.5, n_accs, width=w, label="정상 (Normal)",    color="#6bcb77", edgecolor="#222", zorder=3)
ax2.bar(x_pos+w*1.5, accs,   width=w, label="전체 (Overall)",   color="#74b9ff", edgecolor="#222", zorder=3, alpha=0.85)
ax2.axhline(80, color="white", ls="--", lw=1, alpha=0.5, label="80% 기준선")
for i, ps in enumerate(POOL_SIZES):
    b = results[ps]["per_class"].get("Bengin cases",    0)*100
    m = results[ps]["per_class"].get("Malignant cases", 0)*100
    n = results[ps]["per_class"].get("Normal cases",    0)*100
    a = results[ps]["acc"]*100
    for xoff, val, col in [(-w*1.5,b,"#ff6b6b"),(-w*0.5,m,"#ffd93d"),(w*0.5,n,"#6bcb77"),(w*1.5,a,"#74b9ff")]:
        ax2.text(i+xoff, val+0.5, f"{val:.0f}%", ha="center", va="bottom", color=col, fontsize=7, fontweight="bold")
ax2.set_xticks(x_pos); ax2.set_xticklabels(pool_labels, fontsize=10); ax2.set_ylim(0, 120)
ax2.set_title("클래스별 + 전체 정확도 (512×512)", color="white", fontsize=13, fontweight="bold")
ax2.set_ylabel("정확도 (%)", color=TICK)
ax2.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax2)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph2_class_accuracy.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프2 저장")

# =============================================================
# 그래프 3: 정확도 추세선
# =============================================================
fig3, ax3 = plt.subplots(figsize=(14, 7))
fig3.patch.set_facecolor(BG)
ax3.plot(pool_labels, accs,   "o-",  color="#74b9ff", lw=3,   ms=10, mfc="white", label="전체 정확도", zorder=4)
ax3.plot(pool_labels, b_accs, "s--", color="#ff6b6b", lw=1.8, ms=7,  mfc="white", label="양성 (Benign)")
ax3.plot(pool_labels, m_accs, "^--", color="#ffd93d", lw=1.8, ms=7,  mfc="white", label="악성 (Malignant)")
ax3.plot(pool_labels, n_accs, "D--", color="#6bcb77", lw=1.8, ms=7,  mfc="white", label="정상 (Normal)")
ax3.fill_between(range(len(pool_labels)), accs, alpha=0.07, color="#74b9ff")
ax3.axhline(80, color="white", ls=":", lw=1, alpha=0.4, label="80% 기준선")
for i, a in enumerate(accs):
    ax3.annotate(f"{a:.1f}%", (i, a), textcoords="offset points",
                 xytext=(0,10), ha="center", color="#74b9ff", fontsize=9, fontweight="bold")
ax3.set_ylim(0, max(accs)+20)
ax3.set_title("풀링 크기별 정확도 추세 (512×512)", color="white", fontsize=13, fontweight="bold")
ax3.set_ylabel("정확도 (%)", color=TICK); ax3.set_xlabel("풀링 크기", color=TICK)
ax3.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph3_trend.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프3 저장")

# =============================================================
# 그래프 4: 보안-성능 트레이드오프
# =============================================================
fig4, ax4 = plt.subplots(figsize=(10, 8))
fig4.patch.set_facecolor(BG)
sc = ax4.scatter(security, accs, c=accs, cmap="RdYlGn", s=300,
                 edgecolors="white", lw=1.5, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(pool_labels):
    ax4.annotate(lbl, (security[i], accs[i]), textcoords="offset points",
                 xytext=(10, 5), color="white", fontsize=10, fontweight="bold")
ax4.axvspan(0,  20, alpha=0.05, color="#6bcb77", label="고성능 구역")
ax4.axvspan(80, 100, alpha=0.05, color="#ff6b6b", label="고보안 구역")
cbar = plt.colorbar(sc, ax=ax4)
cbar.set_label("AI 정확도 (%)", color="white", fontsize=11)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
ax4.set_xlabel("보안 강도 (%) →", color=TICK, fontsize=11)
ax4.set_ylabel("AI 정확도 (%)", color=TICK, fontsize=11)
ax4.set_title("보안성 vs AI 성능 트레이드오프 (512×512)", color="white", fontsize=13, fontweight="bold")
ax4.legend(facecolor="#222", labelcolor="white", fontsize=10); style_ax(ax4)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph4_tradeoff.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프4 저장")

# =============================================================
# 그래프 5: 전송 크기 vs 정확도
# =============================================================
fig5, ax5a = plt.subplots(figsize=(14, 6))
fig5.patch.set_facecolor(BG)
ax5b = ax5a.twinx()
ax5a.bar(x_pos, accs, color=cmap_rg, edgecolor="#222", zorder=3, width=0.6, alpha=0.85)
ax5b.plot(x_pos, compressed, "D-", color="#74b9ff", lw=2.5, ms=10, mfc="white", zorder=4, label="전송 크기")
ax5b.set_yscale("log")
for i, (a, c) in enumerate(zip(accs, compressed)):
    ax5a.text(i, a+0.5, f"{a:.1f}%", ha="center", color="white", fontsize=9, fontweight="bold")
    ax5b.text(i, c*1.3, f"{c:,}개", ha="center", color="#74b9ff", fontsize=7)
ax5a.set_xticks(x_pos); ax5a.set_xticklabels(pool_labels); ax5a.set_ylim(0, max(accs)+18)
ax5a.set_title("전송 데이터 크기 vs AI 정확도 (512×512)", color="white", fontsize=13, fontweight="bold")
ax5a.set_ylabel("정확도 (%)", color="#6bcb77", fontsize=12)
ax5b.set_ylabel("전송 크기 (로그, 개)", color="#74b9ff", fontsize=12)
ax5a.tick_params(axis="y", colors="#6bcb77"); ax5b.tick_params(axis="y", colors="#74b9ff")
ax5a.tick_params(axis="x", colors=TICK)
for sp in ax5a.spines.values(): sp.set_color("#444")
ax5a.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); ax5a.set_axisbelow(True)
ax5b.legend(facecolor="#222", labelcolor="white")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph5_datasize_vs_acc.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프5 저장")

# =============================================================
# 그래프 6: 히트맵
# =============================================================
fig6, ax6 = plt.subplots(figsize=(14, 5))
fig6.patch.set_facecolor(BG)
hm = np.array([b_accs, m_accs, n_accs, accs])
im = ax6.imshow(hm, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
ax6.set_xticks(range(len(pool_labels))); ax6.set_xticklabels(pool_labels, color="white", fontsize=11)
ax6.set_yticks(range(4))
ax6.set_yticklabels(["양성\n(Benign)", "악성\n(Malignant)", "정상\n(Normal)", "전체\n(Overall)"], color="white", fontsize=11)
for i in range(4):
    for j in range(len(POOL_SIZES)):
        v = hm[i, j]
        ax6.text(j, i, f"{v:.1f}%", ha="center", va="center",
                 color="black" if v > 60 else "white", fontsize=10, fontweight="bold")
cbar6 = plt.colorbar(im, ax=ax6)
cbar6.set_label("정확도 (%)", color="white")
cbar6.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar6.ax.yaxis.get_ticklabels(), color="white")
ax6.set_title("클래스별 + 전체 정확도 히트맵 (512×512)", color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph6_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프6 저장")

# =============================================================
# 그래프 7: Confusion Matrix
# =============================================================
fig7, axes7 = plt.subplots(1, 2, figsize=(16, 7))
fig7.patch.set_facecolor(BG)
for ax_cm, ps, ttl in zip(axes7, [best_ps, worst_ps], ["🏆 최고 성능", "📉 최저 성능"]):
    ax_cm.set_facecolor(CELL)
    cm_arr = np.array(results[ps]["cm"])
    lbl    = f"{ps}×{ps}" if ps < feat_h else "GAP"
    im7    = ax_cm.imshow(cm_arr, interpolation="nearest", cmap="Blues")
    plt.colorbar(im7, ax=ax_cm)
    tick_marks = np.arange(len(class_names))
    short_names = [c.replace(" cases","").replace("Bengin","양성").replace("Malignant","악성").replace("Normal","정상")
                   for c in class_names]
    ax_cm.set_xticks(tick_marks); ax_cm.set_xticklabels(short_names, color="white", fontsize=10, rotation=15)
    ax_cm.set_yticks(tick_marks); ax_cm.set_yticklabels(short_names, color="white", fontsize=10)
    thresh = cm_arr.max() / 2.
    for i, j in itertools.product(range(cm_arr.shape[0]), range(cm_arr.shape[1])):
        ax_cm.text(j, i, str(cm_arr[i,j]), ha="center", va="center",
                   color="white" if cm_arr[i,j] < thresh else "black", fontsize=14, fontweight="bold")
    ax_cm.set_title(f"{ttl}\nPool {lbl} | {results[ps]['acc']*100:.1f}%",
                    color="white", fontsize=12, fontweight="bold")
    ax_cm.set_xlabel("AI 예측", color=TICK, fontsize=11)
    ax_cm.set_ylabel("실제 정답", color=TICK, fontsize=11)
fig7.suptitle("Confusion Matrix: 최고 vs 최저 풀링 (512×512)", color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph7_confusion.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프7 저장")

# =============================================================
# 그래프 8: 레이더 차트
# =============================================================
key_ps = sorted(set([POOL_SIZES[0], POOL_SIZES[len(POOL_SIZES)//2], POOL_SIZES[-2], POOL_SIZES[-1]]))
categories = ["전체 정확도", "양성 정확도", "악성 정확도", "정상 정확도", "보안 강도"]
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
fig8, ax8 = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig8.patch.set_facecolor(BG); ax8.set_facecolor(CELL)
colors_radar = ["#f9ca24", "#6bcb77", "#ff6b6b", "#7c83fd"]
for idx, ps in enumerate(key_ps):
    r   = results[ps]
    b   = r["per_class"].get("Bengin cases",    0)*100
    m   = r["per_class"].get("Malignant cases", 0)*100
    n   = r["per_class"].get("Normal cases",    0)*100
    sec = 100 - (r["compressed"] / orig_pixels * 100)
    vals = [r["acc"]*100, b, m, n, sec]; vals += vals[:1]
    lbl = f"Pool {ps}×{ps}" if ps < feat_h else "Pool GAP"
    ax8.plot(angles, vals, "o-", lw=2, color=colors_radar[idx%4], label=lbl)
    ax8.fill(angles, vals, alpha=0.05, color=colors_radar[idx%4])
ax8.set_xticks(angles[:-1]); ax8.set_xticklabels(categories, color="white", fontsize=11)
ax8.set_ylim(0, 100); ax8.set_yticks([20,40,60,80,100])
ax8.set_yticklabels(["20%","40%","60%","80%","100%"], color=TICK, fontsize=8)
ax8.grid(color="#444", linestyle="--", alpha=0.6); ax8.spines["polar"].set_color("#444")
ax8.set_title("레이더 차트 (512×512)", color="white", fontsize=13, fontweight="bold", pad=25)
ax8.legend(facecolor="#222", labelcolor="white", fontsize=11, loc="upper right", bbox_to_anchor=(1.4,1.1))
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph8_radar.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프8 저장")

# =============================================================
# 그래프 9: 학습 곡선
# =============================================================
n_cols = min(4, len(POOL_SIZES))
n_rows = (len(POOL_SIZES) + n_cols - 1) // n_cols
fig9, axes9 = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
fig9.patch.set_facecolor(BG)
axes9 = axes9.flatten() if hasattr(axes9, 'flatten') else [axes9]
for i, ps in enumerate(POOL_SIZES):
    ax = axes9[i]; ax.set_facecolor(CELL)
    h  = results[ps]["history"]
    ep = range(1, len(h["accuracy"])+1)
    ax.plot(ep, [v*100 for v in h["accuracy"]],     color="#6bcb77", lw=2, label="학습 정확도")
    ax.plot(ep, [v*100 for v in h["val_accuracy"]], color="#ffd93d", lw=2, label="검증 정확도")
    ax.plot(ep, [v*100 for v in h["loss"]],     color="#ff6b6b", lw=1.5, ls="--", label="학습 손실", alpha=0.7)
    ax.plot(ep, [v*100 for v in h["val_loss"]], color="#74b9ff", lw=1.5, ls="--", label="검증 손실", alpha=0.7)
    best_ep = np.argmax(h["val_accuracy"])+1
    ax.axvline(best_ep, color="white", ls=":", lw=1, alpha=0.5)
    lbl = f"{ps}×{ps}" if ps < feat_h else "GAP"
    b_v = results[ps]["per_class"].get("Bengin cases",    0)*100
    m_v = results[ps]["per_class"].get("Malignant cases", 0)*100
    n_v = results[ps]["per_class"].get("Normal cases",    0)*100
    ax.set_title(
        f"Pool {lbl} | 테스트: {results[ps]['acc']*100:.1f}%\n"
        f"양성:{b_v:.0f}% 악성:{m_v:.0f}% 정상:{n_v:.0f}% ({results[ps]['epochs_run']}ep)",
        color="white", fontsize=9, fontweight="bold"
    )
    ax.set_xlabel("에폭", color=TICK); ax.set_ylabel("값 (%)", color=TICK); ax.set_ylim(0, 105)
    ax.legend(facecolor="#222", labelcolor="white", fontsize=7)
    for sp in ax.spines.values(): sp.set_color("#444")
    ax.tick_params(colors=TICK); ax.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); ax.set_axisbelow(True)
for j in range(len(POOL_SIZES), len(axes9)): axes9[j].set_visible(False)
fig9.suptitle("풀링 크기별 학습 곡선 (512×512)", color="white", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/graph9_learning_curves.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프9 저장")

# =============================================================
# 그래프 10: 종합 대시보드
# =============================================================
from matplotlib.gridspec import GridSpec
fig10 = plt.figure(figsize=(22, 14)); fig10.patch.set_facecolor(BG)
gs10  = GridSpec(3, 3, figure=fig10, hspace=0.5, wspace=0.35)

a1 = fig10.add_subplot(gs10[0,:2]); a1.set_facecolor(CELL)
b10 = a1.bar(x_pos, accs, color=cmap_rg, edgecolor="#333", zorder=3)
for bar, acc in zip(b10, accs):
    grade = "🟢" if acc>=85 else ("🟡" if acc>=70 else ("🟠" if acc>=55 else "🔴"))
    a1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f"{grade}\n{acc:.1f}%", ha="center", va="bottom", color="white", fontsize=10, fontweight="bold")
a1.set_xticks(x_pos); a1.set_xticklabels(pool_labels); a1.set_ylim(0, max(accs)+18)
a1.set_title("풀링 크기별 테스트 정확도 (512×512)", color="white", fontsize=12, fontweight="bold")
a1.set_ylabel("정확도 (%)", color=TICK)
for sp in a1.spines.values(): sp.set_color("#444")
a1.tick_params(colors=TICK); a1.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a1.set_axisbelow(True)

a2 = fig10.add_subplot(gs10[0,2]); a2.set_facecolor(CELL); a2.axis("off")
lbl_b2 = f"{best_ps}×{best_ps}" if best_ps<feat_h else "GAP"
lbl_w2 = f"{worst_ps}×{worst_ps}" if worst_ps<feat_h else "GAP"
b_best = results[best_ps]["per_class"].get("Bengin cases",    0)*100
m_best = results[best_ps]["per_class"].get("Malignant cases", 0)*100
n_best = results[best_ps]["per_class"].get("Normal cases",    0)*100
summary_txt = (
    f"📊 실험 요약 (512×512)\n\n"
    f"풀링: {POOL_SIZES}\n"
    f"특징맵: {feat_h}×{feat_w}×{feat_c}\n"
    f"Mixed Precision: FP16\n\n"
    f"🏆 최고: Pool {lbl_b2}\n"
    f"  전체: {results[best_ps]['acc']*100:.1f}%\n"
    f"  양성: {b_best:.1f}%\n"
    f"  악성: {m_best:.1f}%\n"
    f"  정상: {n_best:.1f}%\n\n"
    f"📉 최저: Pool {lbl_w2}\n"
    f"  전체: {results[worst_ps]['acc']*100:.1f}%\n\n"
    f"🎯 결론\n"
    f"  풀링↑ → 정확도↑\n"
    f"  풀링↓ → 보안↑"
)
a2.text(0.05, 0.97, summary_txt, transform=a2.transAxes, color="white",
        fontsize=9.5, verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="#21262d", edgecolor="#555"))
a2.set_title("실험 결론", color="white", fontsize=12, fontweight="bold")

a3 = fig10.add_subplot(gs10[1,:2]); a3.set_facecolor(CELL)
hm2 = np.array([b_accs, m_accs, n_accs, accs])
im3 = a3.imshow(hm2, cmap="RdYlGn", aspect="auto", vmin=0, vmax=100)
a3.set_xticks(range(len(pool_labels))); a3.set_xticklabels(pool_labels, color="white", fontsize=10)
a3.set_yticks(range(4)); a3.set_yticklabels(["양성","악성","정상","전체"], color="white", fontsize=10)
for i in range(4):
    for j in range(len(POOL_SIZES)):
        v = hm2[i,j]
        a3.text(j, i, f"{v:.0f}%", ha="center", va="center",
                color="black" if v>60 else "white", fontsize=10, fontweight="bold")
a3.set_title("클래스별 히트맵", color="white", fontsize=12, fontweight="bold")
plt.colorbar(im3, ax=a3).ax.yaxis.set_tick_params(color="white")

a4 = fig10.add_subplot(gs10[1,2]); a4.set_facecolor(CELL)
sc4 = a4.scatter(security, accs, c=accs, cmap="RdYlGn", s=150,
                 edgecolors="white", lw=1, zorder=3, vmin=30, vmax=100)
for i, lbl in enumerate(pool_labels):
    a4.annotate(lbl, (security[i], accs[i]), textcoords="offset points", xytext=(6,4), color="white", fontsize=8)
a4.set_xlabel("보안 강도 →", color=TICK); a4.set_ylabel("AI 정확도 (%)", color=TICK)
a4.set_title("보안 vs 성능", color="white", fontsize=12, fontweight="bold")
for sp in a4.spines.values(): sp.set_color("#444")
a4.tick_params(colors=TICK); a4.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a4.set_axisbelow(True)

a5 = fig10.add_subplot(gs10[2,0]); a5.set_facecolor(CELL)
h_b = results[best_ps]["history"]; ep_b = range(1, len(h_b["accuracy"])+1)
a5.plot(ep_b, [v*100 for v in h_b["accuracy"]],     color="#6bcb77", lw=2, label="학습")
a5.plot(ep_b, [v*100 for v in h_b["val_accuracy"]], color="#ffd93d", lw=2, label="검증")
a5.set_title(f"최고 Pool {lbl_b2} 학습 곡선", color="white", fontsize=11, fontweight="bold")
a5.set_xlabel("에폭", color=TICK); a5.set_ylabel("정확도 (%)", color=TICK); a5.set_ylim(0, 105)
a5.legend(facecolor="#222", labelcolor="white", fontsize=9)
for sp in a5.spines.values(): sp.set_color("#444")
a5.tick_params(colors=TICK); a5.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a5.set_axisbelow(True)

a6 = fig10.add_subplot(gs10[2,1]); a6.set_facecolor(CELL)
a6.bar(x_pos, compressed, color=cmap_rg, edgecolor="#222", zorder=3)
a6.set_yscale("log"); a6.set_xticks(x_pos); a6.set_xticklabels(pool_labels)
a6.set_title("전송 크기 (로그)", color="white", fontsize=11, fontweight="bold")
a6.set_ylabel("전송 크기 (로그, 개)", color=TICK)
for sp in a6.spines.values(): sp.set_color("#444")
a6.tick_params(colors=TICK); a6.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a6.set_axisbelow(True)

a7 = fig10.add_subplot(gs10[2,2]); a7.set_facecolor(CELL); a7b = a7.twinx()
a7.bar(x_pos, epochs_run, color=cmap_rg, edgecolor="#222", zorder=3, alpha=0.85)
a7b.plot(x_pos, accs, "o-", color="#f9ca24", lw=2, ms=8, mfc="white", zorder=4)
for i, ep in enumerate(epochs_run): a7.text(i, ep+0.3, f"{ep}ep", ha="center", color="white", fontsize=8)
a7.set_xticks(x_pos); a7.set_xticklabels(pool_labels); a7.set_ylim(0, EPOCHS_S2+10)
a7.set_title("에폭 수 & 정확도", color="white", fontsize=11, fontweight="bold")
a7.set_ylabel("에폭 수", color="#6bcb77"); a7b.set_ylabel("정확도 (%)", color="#f9ca24")
a7.tick_params(axis="y", colors="#6bcb77"); a7b.tick_params(axis="y", colors="#f9ca24")
a7.tick_params(axis="x", colors=TICK)
for sp in a7.spines.values(): sp.set_color("#444")
a7.yaxis.grid(True, color=GRID, linestyle="--", alpha=0.5); a7.set_axisbelow(True)

fig10.suptitle(
    f"Edge-Server Split CNN 종합 대시보드 (512×512 | Mixed Precision FP16)\n"
    f"오버샘플링 {TARGET_COUNT}장/클래스 | MobileNetV2 | Adamax | RTX 4090",
    color="white", fontsize=13, fontweight="bold", y=1.01
)
plt.savefig(f"{SAVE_DIR}/graph10_dashboard.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show(); print("✅ 그래프10 저장")

print(f"""
╔══════════════════════════════════════════════════════╗
║  ✅ 완료! → {SAVE_DIR}/
╠══════════════════════════════════════════════════════╣
║  이미지 크기: 512×512 (원본 데이터와 동일!)
║  특징맵:     {feat_h}×{feat_w}×{feat_c} (224 때 7×7보다 세밀)
║  Mixed Precision: FP16 (속도↑, 메모리↓)
╚══════════════════════════════════════════════════════╝

🏆 최고: Pool {best_ps}×{best_ps} → {results[best_ps]["acc"]*100:.2f}%
📉 최저: Pool {worst_ps}×{worst_ps} → {results[worst_ps]["acc"]*100:.2f}%

💡 OOM(메모리 부족) 날 때 해결법:
   1. BATCH_SIZE = 16 → 8 로 줄이기
   2. POOL_SIZES에서 몇 개 제거하기
   3. gc.collect() 더 자주 호출
""")
